## Notebook 03 — DNN and LSTM Models + SHAP/LIME Analysis

**Models:** DNN (256→128→64 with BatchNorm/Dropout), LSTM (hidden=128, layers=2)  
**Datasets:** Edge-IIoT and CIC-IDS-2017  
**Steps:** Architecture definition → Training (10 seeds × 5 folds) → GradientExplainer SHAP → LIME → RQ1–RQ5 evaluation → FGSM adversarial analysis  

> Requires a CUDA-capable GPU (tested on NVIDIA RTX 3080). Falls back to CPU if CUDA is unavailable — training will be substantially slower.  
> Run the **Configuration** cell (below) before any other cell to set data paths.


In [ ]:
from pathlib import Path

# ── DATA PATHS — adjust to your local setup ──────────────────────────────────
# Point BASE_DIR to the folder that contains your checkpoint sub-directories.
# Expected structure:
#   BASE_DIR/
#     edgeiiot_checkpoints/   <- Edge-IIoT intermediate results
#     cicids2017_checkpoints/ <- CIC-IDS-2017 intermediate results
#     dnn_lstm_checkpoints/   <- DNN / LSTM intermediate results
#     ML-EdgeIIoT-dataset.csv <- raw Edge-IIoT CSV (for Notebook 01)
BASE_DIR       = Path("../data")
EDGEIIOT_DIR   = BASE_DIR / "edgeiiot_checkpoints"
CICIDS2017_DIR = BASE_DIR / "cicids2017_checkpoints"
DNN_LSTM_DIR   = BASE_DIR / "dnn_lstm_checkpoints"
FIGURES_DIR    = Path("../figures")
RESULTS_DIR    = BASE_DIR / "analiz_sonuclari"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import os, pickle, gc, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import shap
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz: {DEVICE} — {torch.cuda.get_device_name(0)}")

# Checkpoint dizinleri
EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)
os.makedirs(DNN_DIR, exist_ok=True)

def kaydet(nesne, yol):
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 {os.path.basename(yol)} ({os.path.getsize(yol)/1024**2:.1f} MB)")

def yukle(yol):
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol(yol):
    return os.path.exists(yol)

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

In [ ]:
class IDS_DNN(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            # Katman 1
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Katman 2
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Katman 3
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            # Çıkış
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

    def predict_proba(self, X):
        """sklearn uyumlu predict_proba — LIME/SHAP için"""
        self.eval()
        with torch.no_grad():
            if isinstance(X, np.ndarray):
                X = torch.FloatTensor(X).to(DEVICE)
            logits = self.forward(X)
            probs  = torch.softmax(logits, dim=1)
        return probs.cpu().numpy()


def model_egit(X_train, y_train, X_val, y_val,
               input_dim, num_classes,
               epochs=100, batch_size=256, patience=10, lr=1e-3):
    """Tek bir fold için DNN eğit, erken durdurma ile."""
    model = IDS_DNN(input_dim, num_classes).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss()

    # Tensor'e çevir
    Xt = torch.FloatTensor(X_train).to(DEVICE)
    yt = torch.LongTensor(y_train).to(DEVICE)
    Xv = torch.FloatTensor(X_val).to(DEVICE)
    yv = torch.LongTensor(y_val).to(DEVICE)

    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=batch_size, shuffle=True)

    en_iyi_val  = -1
    patience_say = 0
    en_iyi_state = None

    for epoch in range(epochs):
        # Eğitim
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        # Validasyon
        model.eval()
        with torch.no_grad():
            val_pred = model(Xv).argmax(dim=1).cpu().numpy()
        val_f1 = f1_score(yv.cpu().numpy(), val_pred, average='macro')
        scheduler.step(-val_f1)

        if val_f1 > en_iyi_val:
            en_iyi_val   = val_f1
            en_iyi_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_say = 0
        else:
            patience_say += 1
            if patience_say >= patience:
                break

    model.load_state_dict(en_iyi_state)
    return model, en_iyi_val

print("✅ DNN mimarisi hazır")

In [ ]:
# Veri yükle
veri_edge = yukle(os.path.join(EDGE_DIR, "veri.pkl"))
X_edge = veri_edge['X_multi_scaled']
y_edge = veri_edge['y_multi']
fn_edge = veri_edge['feature_names_multi']
cn_edge = list(veri_edge['sinif_isimleri'])

# DataFrame ise numpy'a çevir
if hasattr(X_edge, 'values'): X_edge = X_edge.values
if hasattr(y_edge, 'values'): y_edge = y_edge.values

INPUT_DIM_EDGE = X_edge.shape[1]
NUM_CLASS_EDGE = len(np.unique(y_edge))
print(f"Edge-IIoT: {X_edge.shape}, {NUM_CLASS_EDGE} sınıf, {INPUT_DIM_EDGE} özellik")

# ── Eğitim ───────────────────────────────────────────────────────────
SEEDS  = list(range(10))
N_FOLD = 5
CHECKPOINT_PATH = os.path.join(DNN_DIR, "dnn_edge_modeller.pkl")

if kontrol(CHECKPOINT_PATH):
    dnn_edge_sonuclar = yukle(CHECKPOINT_PATH)
    print(f"Checkpoint yüklendi: {len(dnn_edge_sonuclar)} kayıt")
else:
    dnn_edge_sonuclar = []

tamamlanan = {(r['seed'], r['fold']) for r in dnn_edge_sonuclar}

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_edge, y_edge)):
        if (seed, fold) in tamamlanan:
            print(f"  ♻  seed={seed} fold={fold} — atlandı")
            continue

        t0 = time.time()
        model, val_f1 = model_egit(
            X_edge[tr_idx], y_edge[tr_idx],
            X_edge[val_idx], y_edge[val_idx],
            INPUT_DIM_EDGE, NUM_CLASS_EDGE
        )
        sure = time.time() - t0

        model.cpu()
        dnn_edge_sonuclar.append({
            'seed'    : seed,
            'fold'    : fold,
            'f1_macro': round(val_f1, 4),
            'model_obj': model,
            'sure'    : round(sure, 1)
        })
        kaydet(dnn_edge_sonuclar, CHECKPOINT_PATH)
        print(f"  ✅ seed={seed} fold={fold}  F1={val_f1:.4f}  ({sure:.0f}s)")

# ── Özet ─────────────────────────────────────────────────────────────
f1_ler = [r['f1_macro'] for r in dnn_edge_sonuclar]
print(f"\n📊 DNN Edge-IIoT Özet:")
print(f"  Ortalama F1 : {np.mean(f1_ler):.4f}")
print(f"  Std          : {np.std(f1_ler):.4f}")
print(f"  Min/Max      : {np.min(f1_ler):.4f} / {np.max(f1_ler):.4f}")

In [ ]:
# ── DNN için SHAP (GradientExplainer) — Düzeltilmiş ─────────────────
import shap
import lime.lime_tabular

# Referans model: seed=0, fold=0
ref_model = None
for r in dnn_edge_sonuclar:
    if r['seed'] == 0 and r['fold'] == 0:
        ref_model = r['model_obj']
        break

ref_model = ref_model.to(DEVICE)
ref_model.eval()

# 500 örnek (diğer XAI'larla aynı seed)
np.random.seed(42)
orneklem_idx = np.random.choice(len(X_edge), size=500, replace=False)
X_orneklem   = X_edge[orneklem_idx]

# Background için 100 örnek
np.random.seed(0)
bg_idx       = np.random.choice(len(X_edge), size=100, replace=False)
X_background = torch.FloatTensor(X_edge[bg_idx]).to(DEVICE)
X_orneklem_t = torch.FloatTensor(X_orneklem).to(DEVICE)

# ── SHAP ─────────────────────────────────────────────────────────────
print("SHAP GradientExplainer başlıyor...")
t0        = time.time()
explainer = shap.GradientExplainer(ref_model, X_background)
shap_raw  = explainer.shap_values(X_orneklem_t)

# Shape tespiti ve düzeltme
shap_arr = np.array(shap_raw)
print(f"  shap_raw shape: {shap_arr.shape}")

if shap_arr.ndim == 3:
    if shap_arr.shape[0] == 500:        # (500, 74, 15)
        shap_mean = np.mean(np.abs(shap_arr), axis=2)   # → (500, 74)
    elif shap_arr.shape[1] == 500:      # (15, 500, 74)
        shap_mean = np.mean(np.abs(shap_arr), axis=0)   # → (500, 74)
    else:                               # (74, 15, 500) gibi nadir
        shap_mean = np.mean(np.abs(shap_arr), axis=1).T
elif shap_arr.ndim == 2:
    shap_mean = np.abs(shap_arr)        # zaten (500, 74)

shap_sure = (time.time() - t0) / 500 * 1000
print(f"✅ SHAP tamamlandı — shape: {shap_mean.shape}, {shap_sure:.2f} ms/örnek")

# ── LIME ─────────────────────────────────────────────────────────────
def dnn_predict_proba(X_np):
    ref_model.eval()
    with torch.no_grad():
        t = torch.FloatTensor(X_np).to(DEVICE)
        return torch.softmax(ref_model(t), dim=1).cpu().numpy()

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

lime_explainer_dnn = lime.lime_tabular.LimeTabularExplainer(
    X_edge,
    feature_names=fn_edge,
    class_names=cn_edge,
    discretize_continuous=True,
    random_state=42
)

print("LIME başlıyor (500 örnek)...")
t0        = time.time()
lime_vals = []
for i in range(500):
    exp = lime_explainer_dnn.explain_instance(
        X_orneklem[i], dnn_predict_proba,
        num_features=len(fn_edge), num_samples=1000
    )
    lime_vals.append(lime_to_vec(dict(exp.as_list()), fn_edge))
    if (i+1) % 100 == 0:
        print(f"  LIME {i+1}/500...")

lime_arr  = np.array(lime_vals)
lime_sure = (time.time() - t0) / 500 * 1000
print(f"✅ LIME tamamlandı — shape: {lime_arr.shape}, {lime_sure:.2f} ms/örnek")

# ── Kaydet ───────────────────────────────────────────────────────────
xai_dnn_edge = {
    'shap_values': shap_mean,
    'lime_values': lime_arr,
    'shap_sure'  : shap_sure,
    'lime_sure'  : lime_sure,
}
kaydet(xai_dnn_edge, os.path.join(DNN_DIR, "xai_dnn_edge.pkl"))
print(f"\n📊 Özet:")
print(f"  SHAP shape : {shap_mean.shape}")
print(f"  LIME shape : {lime_arr.shape}")
print(f"  SHAP süre  : {shap_sure:.2f} ms/örnek")
print(f"  LIME süre  : {lime_sure:.2f} ms/örnek")

In [ ]:
# ── RQ1: Pertürbasyon Analizi — DNN Edge-IIoT ────────────────────────
from scipy.stats import spearmanr

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

def dnn_predict_proba(X_np):
    ref_model.eval()
    with torch.no_grad():
        t = torch.FloatTensor(X_np).to(DEVICE)
        return torch.softmax(ref_model(t), dim=1).cpu().numpy()

import lime.lime_tabular

# Yükle
xai_dnn = yukle(os.path.join(DNN_DIR, "xai_dnn_edge.pkl"))
shap_orig = xai_dnn['shap_values']  # (500, 74)
lime_orig = xai_dnn['lime_values']  # (500, 74)

# Referans model
ref_model = None
for r in dnn_edge_sonuclar:
    if r['seed'] == 0 and r['fold'] == 0:
        ref_model = r['model_obj']
        break
ref_model = ref_model.to(DEVICE)
ref_model.eval()

# Explainerlar
np.random.seed(42)
orneklem_idx = np.random.choice(len(X_edge), size=500, replace=False)
X_orneklem   = X_edge[orneklem_idx]

np.random.seed(0)
bg_idx       = np.random.choice(len(X_edge), size=100, replace=False)
X_background = torch.FloatTensor(X_edge[bg_idx]).to(DEVICE)

explainer = shap.GradientExplainer(ref_model, X_background)

lime_explainer_dnn = lime.lime_tabular.LimeTabularExplainer(
    X_edge,
    feature_names=fn_edge,
    class_names=cn_edge,
    discretize_continuous=True,
    random_state=42
)

# ── Ana döngü ────────────────────────────────────────────────────────
ORANLAR      = {'pct1': 0.01, 'pct3': 0.03, 'pct5': 0.05}
PERTURB_PATH = os.path.join(DNN_DIR, "perturb_dnn_edge.pkl")

sonuc = {oran_key: {'shap': {'spearman': [], 'jaccard_5': [], 'jaccard_10': []},
                    'lime': {'spearman': [], 'jaccard_5': [], 'jaccard_10': []}}
         for oran_key in ORANLAR}

print("RQ1 Pertürbasyon başlıyor...")
t0 = time.time()

for oran_key, oran in ORANLAR.items():
    shap_sp, shap_j5, shap_j10 = [], [], []
    lime_sp, lime_j5, lime_j10 = [], [], []

    for i in range(500):
        x_orig = X_orneklem[i].copy()
        n_pert = max(1, int(len(fn_edge) * oran))
        pert_idx = np.random.choice(len(fn_edge), size=n_pert, replace=False)

        x_pert = x_orig.copy()
        x_pert[pert_idx] += np.random.normal(0, 0.1, size=n_pert)

        # SHAP pertürbe
        xp_t = torch.FloatTensor(x_pert.reshape(1, -1)).to(DEVICE)
        sv_p = np.array(explainer.shap_values(xp_t))  # (1, 74, 15) veya (1, 74)
        if sv_p.ndim == 3:
            sv_p = np.mean(np.abs(sv_p[0]), axis=1)   # (74,)
        else:
            sv_p = np.abs(sv_p[0])                     # (74,)

        sp_s, _ = spearmanr(shap_orig[i], sv_p)
        shap_sp.append(sp_s)
        shap_j5.append(jaccard_top_k(shap_orig[i], sv_p, 5))
        shap_j10.append(jaccard_top_k(shap_orig[i], sv_p, 10))

        # LIME pertürbe
        exp_p = lime_explainer_dnn.explain_instance(
            x_pert, dnn_predict_proba,
            num_features=len(fn_edge), num_samples=1000
        )
        lv_p    = lime_to_vec(dict(exp_p.as_list()), fn_edge)
        sp_l, _ = spearmanr(lime_orig[i], lv_p)
        lime_sp.append(sp_l)
        lime_j5.append(jaccard_top_k(lime_orig[i], lv_p, 5))
        lime_j10.append(jaccard_top_k(lime_orig[i], lv_p, 10))

        if (i+1) % 100 == 0:
            print(f"  [{oran_key}] {i+1}/500 ...")

    sonuc[oran_key]['shap']['spearman']   = shap_sp
    sonuc[oran_key]['shap']['jaccard_5']  = shap_j5
    sonuc[oran_key]['shap']['jaccard_10'] = shap_j10
    sonuc[oran_key]['lime']['spearman']   = lime_sp
    sonuc[oran_key]['lime']['jaccard_5']  = lime_j5
    sonuc[oran_key]['lime']['jaccard_10'] = lime_j10

    print(f"  ✅ %{int(oran*100):2d} → SHAP={np.nanmean(shap_sp):.4f}, LIME={np.nanmean(lime_sp):.4f}")

kaydet(sonuc, PERTURB_PATH)
print(f"\n⏱️  Toplam süre: {(time.time()-t0)/60:.1f} dk")
print("✅ RQ1 tamamlandı!")

In [ ]:
# ── RQ2: Model Varyasyonu — DNN Edge-IIoT ────────────────────────────
# 50 modelin (10 seed × 5 fold) SHAP değerlerini karşılaştır
from scipy.stats import spearmanr
from itertools import combinations

RQ2_PATH = os.path.join(DNN_DIR, "rq2_dnn_edge.pkl")
N_ORNEK_RQ2 = 100  # Her model için 100 örnek (hız için)

# Sabit örnek seti
np.random.seed(42)
rq2_idx = np.random.choice(len(X_edge), size=N_ORNEK_RQ2, replace=False)
X_rq2   = torch.FloatTensor(X_edge[rq2_idx]).to(DEVICE)

# Background
np.random.seed(0)
bg_idx       = np.random.choice(len(X_edge), size=100, replace=False)
X_background = torch.FloatTensor(X_edge[bg_idx]).to(DEVICE)

if kontrol(RQ2_PATH):
    shap_per_model = yukle(RQ2_PATH)
    print(f"Checkpoint yüklendi: {len(shap_per_model)} model")
else:
    shap_per_model = {}

print(f"RQ2: {len(dnn_edge_sonuclar)} model için SHAP hesaplanıyor...")
t0 = time.time()

for kayit in dnn_edge_sonuclar:
    key = (kayit['seed'], kayit['fold'])
    if key in shap_per_model:
        continue

    model = kayit['model_obj'].to(DEVICE)
    model.eval()

    exp_rq2  = shap.GradientExplainer(model, X_background)
    sv_raw   = np.array(exp_rq2.shap_values(X_rq2))  # (100, 74, 15) veya (15, 100, 74)

    if sv_raw.ndim == 3:
        if sv_raw.shape[0] == N_ORNEK_RQ2:
            sv = np.mean(np.abs(sv_raw), axis=2)   # (100, 74)
        else:
            sv = np.mean(np.abs(sv_raw), axis=0)   # (100, 74)
    else:
        sv = np.abs(sv_raw)

    shap_per_model[key] = sv
    model.cpu()

    elapsed = time.time() - t0
    done    = len(shap_per_model)
    total   = len(dnn_edge_sonuclar)
    eta     = elapsed / done * (total - done) / 60
    print(f"  ✅ seed={key[0]} fold={key[1]}  ({elapsed:.0f}s geçti, ~{eta:.0f} dk kaldı)")
    kaydet(shap_per_model, RQ2_PATH)

# ── Pairwise Spearman hesapla ─────────────────────────────────────────
print("\nPairwise Spearman hesaplanıyor...")
keys    = list(shap_per_model.keys())
sp_list = []
j5_list = []
j10_list = []

for (k1, k2) in combinations(keys, 2):
    sv1 = shap_per_model[k1]  # (100, 74)
    sv2 = shap_per_model[k2]  # (100, 74)
    for i in range(N_ORNEK_RQ2):
        sp, _ = spearmanr(sv1[i], sv2[i])
        sp_list.append(sp)
        j5_list.append(jaccard_top_k(sv1[i], sv2[i], 5))
        j10_list.append(jaccard_top_k(sv1[i], sv2[i], 10))

rq2_sonuc = {
    'spearman'  : float(np.nanmean(sp_list)),
    'jaccard_5' : float(np.nanmean(j5_list)),
    'jaccard_10': float(np.nanmean(j10_list)),
    'n_pairs'   : len(list(combinations(keys, 2))),
}
kaydet(rq2_sonuc, os.path.join(DNN_DIR, "rq2_dnn_edge_ozet.pkl"))

print(f"\n📊 DNN Edge-IIoT RQ2 Sonuçları:")
print(f"  Spearman   : {rq2_sonuc['spearman']:.4f}")
print(f"  Jaccard @5 : {rq2_sonuc['jaccard_5']:.4f}")
print(f"  Jaccard @10: {rq2_sonuc['jaccard_10']:.4f}")
print(f"  Çift sayısı: {rq2_sonuc['n_pairs']}")
print(f"  Toplam süre: {(time.time()-t0)/60:.1f} dk")

In [ ]:
# ── RQ3 + Fidelity — DNN Edge-IIoT ──────────────────────────────────
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

shap_per_model = yukle(os.path.join(DNN_DIR, "rq2_dnn_edge.pkl"))

# ── RQ3: F1 ~ SHAP tutarlılık korelasyonu ────────────────────────────
print("RQ3 hesaplanıyor...")
keys     = list(shap_per_model.keys())
f1_scores = []
shap_consistency = []

for key in keys:
    # Bu modelin F1'ini bul
    f1 = next(r['f1_macro'] for r in dnn_edge_sonuclar
              if r['seed'] == key[0] and r['fold'] == key[1])
    f1_scores.append(f1)

    # Bu modelin diğer tüm modellerle ortalama Spearman'ı
    sv1  = shap_per_model[key]
    sp_m = []
    for key2 in keys:
        if key2 == key:
            continue
        sv2 = shap_per_model[key2]
        for i in range(sv1.shape[0]):
            sp, _ = spearmanr(sv1[i], sv2[i])
            sp_m.append(sp)
    shap_consistency.append(np.nanmean(sp_m))

r_pearson,  p_pearson  = pearsonr(f1_scores, shap_consistency)
r_spearman, p_spearman = spearmanr(f1_scores, shap_consistency)

rq3_sonuc = {
    'f1_scores'         : f1_scores,
    'shap_consistency'  : shap_consistency,
    'pearson_r'         : r_pearson,
    'pearson_p'         : p_pearson,
    'spearman_r'        : r_spearman,
    'spearman_p'        : p_spearman,
}
kaydet(rq3_sonuc, os.path.join(DNN_DIR, "rq3_dnn_edge.pkl"))
print(f"  Pearson  r={r_pearson:.4f}  p={p_pearson:.4f}")
print(f"  Spearman r={r_spearman:.4f}  p={p_spearman:.4f}")

# ── Fidelity: Top-K özellik maskeleme ────────────────────────────────
print("\nFidelity hesaplanıyor...")
xai_dnn   = yukle(os.path.join(DNN_DIR, "xai_dnn_edge.pkl"))
shap_vals = xai_dnn['shap_values']   # (500, 74)
X_orn     = X_edge[orneklem_idx]     # (500, 74)
y_orn     = y_edge[orneklem_idx]     # (500,)

# Referans model
ref_model = next(r['model_obj'] for r in dnn_edge_sonuclar
                 if r['seed'] == 0 and r['fold'] == 0)
ref_model = ref_model.to(DEVICE)
ref_model.eval()

def dnn_predict(X_np):
    with torch.no_grad():
        t = torch.FloatTensor(X_np).to(DEVICE)
        return ref_model(t).argmax(dim=1).cpu().numpy()

# Orijinal F1
f1_orig = f1_score(y_orn, dnn_predict(X_orn), average='macro')
print(f"  Orijinal F1: {f1_orig:.4f}")

fidelity_sonuc = []
X_mean = np.mean(X_edge, axis=0)

for n_mask in range(1, 11):
    shap_masked = X_orn.copy()
    lime_masked = X_orn.copy()

    for i in range(len(X_orn)):
        # SHAP top-n maskele
        top_shap = np.argsort(shap_vals[i])[-n_mask:]
        shap_masked[i, top_shap] = X_mean[top_shap]

        # LIME top-n maskele
        top_lime = np.argsort(np.abs(xai_dnn['lime_values'][i]))[-n_mask:]
        lime_masked[i, top_lime] = X_mean[top_lime]

    f1_shap = f1_score(y_orn, dnn_predict(shap_masked), average='macro')
    f1_lime = f1_score(y_orn, dnn_predict(lime_masked), average='macro')

    fidelity_sonuc.append({
        'model'     : 'DNN',
        'xai'       : 'SHAP',
        'n_maskele' : n_mask,
        'f1_orig'   : round(f1_orig, 4),
        'f1_mask'   : round(f1_shap, 4),
        'delta_f1'  : round(f1_orig - f1_shap, 4),
    })
    fidelity_sonuc.append({
        'model'     : 'DNN',
        'xai'       : 'LIME',
        'n_maskele' : n_mask,
        'f1_orig'   : round(f1_orig, 4),
        'f1_mask'   : round(f1_lime, 4),
        'delta_f1'  : round(f1_orig - f1_lime, 4),
    })
    print(f"  n={n_mask:2d} → SHAP ΔF1={f1_orig-f1_shap:.4f}  LIME ΔF1={f1_orig-f1_lime:.4f}")

df_fid_dnn = pd.DataFrame(fidelity_sonuc)
kaydet(df_fid_dnn, os.path.join(DNN_DIR, "fidelity_dnn_edge.pkl"))

shap5 = df_fid_dnn[(df_fid_dnn['xai']=='SHAP') & (df_fid_dnn['n_maskele']==5)]['delta_f1'].values[0]
lime5 = df_fid_dnn[(df_fid_dnn['xai']=='LIME') & (df_fid_dnn['n_maskele']==5)]['delta_f1'].values[0]
print(f"\n📊 DNN Edge-IIoT Fidelity @Top-5:")
print(f"  SHAP ΔF1: {shap5:.4f}")
print(f"  LIME ΔF1: {lime5:.4f}")
print("✅ RQ3 + Fidelity tamamlandı!")

In [ ]:
# ── XAI-Score — DNN dahil tüm modeller (Edge-IIoT) ───────────────────
import numpy as np
import pandas as pd
import pickle, os

EDGE_DIR = str(EDGEIIOT_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

def yukle(yol):
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kaydet(nesne, yol):
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 {os.path.basename(yol)}")

def normalize(d, inverse=False):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.5 for k in d}
    normed = {k: (v - mn) / (mx - mn) for k, v in d.items()}
    if inverse:
        normed = {k: 1 - v for k, v in normed.items()}
    return normed

# ── Mevcut 7 modelin ham değerleri ───────────────────────────────────
df_xai_orig = yukle(os.path.join(EDGE_DIR, "df_xai_score.pkl"))

# Ham değerler sözlüğü (normalize edilmemiş)
perturb_ham  = dict(zip(df_xai_orig['Model'], df_xai_orig['Perturb']))
varyans_ham  = dict(zip(df_xai_orig['Model'], df_xai_orig['Varyans']))
fidelity_ham = dict(zip(df_xai_orig['Model'], df_xai_orig['Fidelity_d5']))
sure_ham     = dict(zip(df_xai_orig['Model'], df_xai_orig['Sure_ms']))

# ── DNN değerlerini ekle ──────────────────────────────────────────────
perturb_sonuc = yukle(os.path.join(DNN_DIR, "perturb_dnn_edge.pkl"))
rq2_ozet      = yukle(os.path.join(DNN_DIR, "rq2_dnn_edge_ozet.pkl"))
fidelity_dnn  = yukle(os.path.join(DNN_DIR, "fidelity_dnn_edge.pkl"))
xai_dnn       = yukle(os.path.join(DNN_DIR, "xai_dnn_edge.pkl"))

dnn_perturb  = float(np.nanmean(perturb_sonuc['pct3']['shap']['spearman']))
dnn_varyans  = rq2_ozet['spearman']
dnn_fidelity = float(fidelity_dnn[
    (fidelity_dnn['xai']=='SHAP') & (fidelity_dnn['n_maskele']==5)
]['delta_f1'].values[0])
dnn_sure     = xai_dnn['shap_sure']

perturb_ham['DNN']  = round(dnn_perturb,  4)
varyans_ham['DNN']  = round(dnn_varyans,  4)
fidelity_ham['DNN'] = round(dnn_fidelity, 4)
sure_ham['DNN']     = round(dnn_sure,     2)

print("DNN ham değerleri:")
print(f"  Perturb (pct3) : {perturb_ham['DNN']}")
print(f"  Varyans        : {varyans_ham['DNN']}")
print(f"  Fidelity @5    : {fidelity_ham['DNN']}")
print(f"  Sure_ms        : {sure_ham['DNN']}")

# ── Normalize et (8 model birlikte) ──────────────────────────────────
perturb_n  = normalize(perturb_ham)
varyans_n  = normalize(varyans_ham)
fidelity_n = normalize(fidelity_ham)
sure_n     = normalize(sure_ham, inverse=True)

# ── XAI-Score hesapla ─────────────────────────────────────────────────
W = 0.25
TUM_MODELLER = list(perturb_ham.keys())

xai_scores = {
    m: round(W*perturb_n[m] + W*varyans_n[m] + W*fidelity_n[m] + W*sure_n[m], 4)
    for m in TUM_MODELLER
}

df_xai_dnn = pd.DataFrame({
    'Model'      : TUM_MODELLER,
    'Perturb'    : [perturb_ham[m]  for m in TUM_MODELLER],
    'Varyans'    : [varyans_ham[m]  for m in TUM_MODELLER],
    'Fidelity_d5': [fidelity_ham[m] for m in TUM_MODELLER],
    'Sure_ms'    : [sure_ham[m]     for m in TUM_MODELLER],
    'XAI_Score'  : [xai_scores[m]   for m in TUM_MODELLER],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

print("\n📊 XAI-Score (DNN dahil, Edge-IIoT):")
print(df_xai_dnn.to_string(index=False))

kaydet(df_xai_dnn, os.path.join(DNN_DIR, "df_xai_score_dnn_edge.pkl"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DNN — CIC-IDS-2017 EĞİTİMİ
# ══════════════════════════════════════════════════════════════════════

# Veri yükle
veri_cic = yukle(os.path.join(CIC_DIR, "veri_cicids.pkl"))
X_cic    = veri_cic['X_scaled']
y_cic    = veri_cic['y']
fn_cic   = veri_cic['feature_names']
cn_cic   = list(veri_cic['sinif_isimleri'])

if hasattr(X_cic, 'values'): X_cic = X_cic.values
if hasattr(y_cic, 'values'): y_cic = y_cic.values

INPUT_DIM_CIC  = X_cic.shape[1]
NUM_CLASS_CIC  = len(np.unique(y_cic))
print(f"CIC-IDS-2017: {X_cic.shape}, {NUM_CLASS_CIC} sınıf, {INPUT_DIM_CIC} özellik")

# Eğitim
CHECKPOINT_PATH_CIC = os.path.join(DNN_DIR, "dnn_cic_modeller.pkl")

if kontrol(CHECKPOINT_PATH_CIC):
    dnn_cic_sonuclar = yukle(CHECKPOINT_PATH_CIC)
    print(f"Checkpoint yüklendi: {len(dnn_cic_sonuclar)} kayıt")
else:
    dnn_cic_sonuclar = []

tamamlanan_cic = {(r['seed'], r['fold']) for r in dnn_cic_sonuclar}

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cic, y_cic)):
        if (seed, fold) in tamamlanan_cic:
            print(f"  ♻  seed={seed} fold={fold} — atlandı")
            continue

        t0 = time.time()
        model, val_f1 = model_egit(
            X_cic[tr_idx], y_cic[tr_idx],
            X_cic[val_idx], y_cic[val_idx],
            INPUT_DIM_CIC, NUM_CLASS_CIC
        )
        sure = time.time() - t0

        model.cpu()
        dnn_cic_sonuclar.append({
            'seed'    : seed,
            'fold'    : fold,
            'f1_macro': round(val_f1, 4),
            'model_obj': model,
            'sure'    : round(sure, 1)
        })
        kaydet(dnn_cic_sonuclar, CHECKPOINT_PATH_CIC)
        print(f"  ✅ seed={seed} fold={fold}  F1={val_f1:.4f}  ({sure:.0f}s)")

f1_ler_cic = [r['f1_macro'] for r in dnn_cic_sonuclar]
print(f"\n📊 DNN CIC-IDS-2017 Özet:")
print(f"  Ortalama F1 : {np.mean(f1_ler_cic):.4f}")
print(f"  Std          : {np.std(f1_ler_cic):.4f}")
print(f"  Min/Max      : {np.min(f1_ler_cic):.4f} / {np.max(f1_ler_cic):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DNN CIC-IDS-2017 — XAI + RQ1 + RQ2 + RQ3 + Fidelity + XAI-Score
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import f1_score
from itertools import combinations
import lime.lime_tabular

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

# ── Referans model (seed=0, fold=0) ──────────────────────────────────
ref_cic = next(r['model_obj'] for r in dnn_cic_sonuclar
               if r['seed']==0 and r['fold']==0)
ref_cic = ref_cic.to(DEVICE)
ref_cic.eval()

# ── Örnek setleri ─────────────────────────────────────────────────────
np.random.seed(42)
orn_idx_cic  = np.random.choice(len(X_cic), size=500, replace=False)
X_orn_cic    = X_cic[orn_idx_cic]
y_orn_cic    = y_cic[orn_idx_cic]

np.random.seed(0)
bg_idx_cic   = np.random.choice(len(X_cic), size=100, replace=False)
X_bg_cic     = torch.FloatTensor(X_cic[bg_idx_cic]).to(DEVICE)
X_orn_cic_t  = torch.FloatTensor(X_orn_cic).to(DEVICE)

def dnn_cic_proba(X_np):
    ref_cic.eval()
    with torch.no_grad():
        return torch.softmax(ref_cic(torch.FloatTensor(X_np).to(DEVICE)), dim=1).cpu().numpy()

# ── SHAP ─────────────────────────────────────────────────────────────
print("SHAP hesaplanıyor...")
t0          = time.time()
exp_cic     = shap.GradientExplainer(ref_cic, X_bg_cic)
shap_raw    = np.array(exp_cic.shap_values(X_orn_cic_t))
if shap_raw.ndim == 3:
    shap_cic = np.mean(np.abs(shap_raw), axis=2) if shap_raw.shape[0]==500 \
               else np.mean(np.abs(shap_raw), axis=0)
else:
    shap_cic = np.abs(shap_raw)
shap_sure_cic = (time.time()-t0)/500*1000
print(f"  ✅ SHAP shape: {shap_cic.shape}, {shap_sure_cic:.1f} ms/örnek")

# ── LIME ─────────────────────────────────────────────────────────────
lime_exp_cic = lime.lime_tabular.LimeTabularExplainer(
    X_cic, feature_names=fn_cic, class_names=cn_cic,
    discretize_continuous=True, random_state=42)

print("LIME hesaplanıyor...")
t0 = time.time()
lime_cic = []
for i in range(500):
    exp = lime_exp_cic.explain_instance(X_orn_cic[i], dnn_cic_proba,
                                         num_features=len(fn_cic), num_samples=1000)
    lime_cic.append(lime_to_vec(dict(exp.as_list()), fn_cic))
    if (i+1)%100==0: print(f"  LIME {i+1}/500...")
lime_cic      = np.array(lime_cic)
lime_sure_cic = (time.time()-t0)/500*1000
print(f"  ✅ LIME shape: {lime_cic.shape}, {lime_sure_cic:.1f} ms/örnek")

xai_dnn_cic = {'shap_values': shap_cic, 'lime_values': lime_cic,
                'shap_sure': shap_sure_cic, 'lime_sure': lime_sure_cic}
kaydet(xai_dnn_cic, os.path.join(DNN_DIR, "xai_dnn_cic.pkl"))

# ── RQ1: Pertürbasyon ─────────────────────────────────────────────────
print("\nRQ1 başlıyor...")
ORANLAR   = {'pct1':0.01, 'pct3':0.03, 'pct5':0.05}
rq1_sonuc = {ok: {'shap':{'spearman':[],'jaccard_5':[],'jaccard_10':[]},
                   'lime':{'spearman':[],'jaccard_5':[],'jaccard_10':[]}}
             for ok in ORANLAR}
t0 = time.time()
for ok, oran in ORANLAR.items():
    ss,sj5,sj10,ls,lj5,lj10=[],[],[],[],[],[]
    for i in range(500):
        xp = X_orn_cic[i].copy()
        ni = max(1, int(len(fn_cic)*oran))
        pi = np.random.choice(len(fn_cic), size=ni, replace=False)
        xp[pi] += np.random.normal(0, 0.1, size=ni)

        sv = np.array(exp_cic.shap_values(torch.FloatTensor(xp.reshape(1,-1)).to(DEVICE)))
        sv = np.mean(np.abs(sv[0]),axis=1) if sv.ndim==3 else np.abs(sv[0])
        sp,_ = spearmanr(shap_cic[i], sv)
        ss.append(sp); sj5.append(jaccard_top_k(shap_cic[i],sv,5)); sj10.append(jaccard_top_k(shap_cic[i],sv,10))

        ep = lime_exp_cic.explain_instance(xp, dnn_cic_proba, num_features=len(fn_cic), num_samples=1000)
        lv = lime_to_vec(dict(ep.as_list()), fn_cic)
        sl,_ = spearmanr(lime_cic[i], lv)
        ls.append(sl); lj5.append(jaccard_top_k(lime_cic[i],lv,5)); lj10.append(jaccard_top_k(lime_cic[i],lv,10))
        if (i+1)%100==0: print(f"  [{ok}] {i+1}/500...")

    rq1_sonuc[ok]['shap'].update({'spearman':ss,'jaccard_5':sj5,'jaccard_10':sj10})
    rq1_sonuc[ok]['lime'].update({'spearman':ls,'jaccard_5':lj5,'jaccard_10':lj10})
    print(f"  ✅ %{int(oran*100):2d} SHAP={np.nanmean(ss):.4f} LIME={np.nanmean(ls):.4f}")

kaydet(rq1_sonuc, os.path.join(DNN_DIR, "perturb_dnn_cic.pkl"))
print(f"  ⏱ {(time.time()-t0)/60:.1f} dk")

# ── RQ2: Model Varyasyonu ─────────────────────────────────────────────
print("\nRQ2 başlıyor (50 model × 100 örnek)...")
RQ2_CIC_PATH = os.path.join(DNN_DIR, "rq2_dnn_cic.pkl")
N_RQ2 = 100
np.random.seed(42)
rq2_idx_cic = np.random.choice(len(X_cic), size=N_RQ2, replace=False)
X_rq2_cic   = torch.FloatTensor(X_cic[rq2_idx_cic]).to(DEVICE)

shap_per_cic = yukle(RQ2_CIC_PATH) if kontrol(RQ2_CIC_PATH) else {}
t0 = time.time()
for kayit in dnn_cic_sonuclar:
    key = (kayit['seed'], kayit['fold'])
    if key in shap_per_cic: continue
    m = kayit['model_obj'].to(DEVICE); m.eval()
    sv = np.array(shap.GradientExplainer(m, X_bg_cic).shap_values(X_rq2_cic))
    if sv.ndim==3:
        sv = np.mean(np.abs(sv),axis=2) if sv.shape[0]==N_RQ2 else np.mean(np.abs(sv),axis=0)
    else: sv = np.abs(sv)
    shap_per_cic[key] = sv; m.cpu()
    done = len(shap_per_cic); eta = (time.time()-t0)/done*(50-done)/60
    print(f"  ✅ seed={key[0]} fold={key[1]}  (~{eta:.0f} dk kaldı)")
    kaydet(shap_per_cic, RQ2_CIC_PATH)

keys_cic = list(shap_per_cic.keys())
sp_l,j5_l,j10_l=[],[],[]
for k1,k2 in combinations(keys_cic,2):
    for i in range(N_RQ2):
        sp,_=spearmanr(shap_per_cic[k1][i],shap_per_cic[k2][i])
        sp_l.append(sp); j5_l.append(jaccard_top_k(shap_per_cic[k1][i],shap_per_cic[k2][i],5))
        j10_l.append(jaccard_top_k(shap_per_cic[k1][i],shap_per_cic[k2][i],10))

rq2_cic_ozet = {'spearman':float(np.nanmean(sp_l)),'jaccard_5':float(np.nanmean(j5_l)),
                 'jaccard_10':float(np.nanmean(j10_l)),'n_pairs':len(list(combinations(keys_cic,2)))}
kaydet(rq2_cic_ozet, os.path.join(DNN_DIR, "rq2_dnn_cic_ozet.pkl"))
print(f"\n  RQ2: Spearman={rq2_cic_ozet['spearman']:.4f} J@5={rq2_cic_ozet['jaccard_5']:.4f}")

# ── RQ3: F1 ~ SHAP korelasyonu ────────────────────────────────────────
print("\nRQ3 hesaplanıyor...")
f1_cic_list, shap_con_cic = [], []
for key in keys_cic:
    f1 = next(r['f1_macro'] for r in dnn_cic_sonuclar if r['seed']==key[0] and r['fold']==key[1])
    f1_cic_list.append(f1)
    sp_m=[]
    for k2 in keys_cic:
        if k2==key: continue
        for i in range(N_RQ2):
            sp,_=spearmanr(shap_per_cic[key][i],shap_per_cic[k2][i]); sp_m.append(sp)
    shap_con_cic.append(np.nanmean(sp_m))

rp,pp = pearsonr(f1_cic_list, shap_con_cic)
rs,ps = spearmanr(f1_cic_list, shap_con_cic)
kaydet({'pearson_r':rp,'pearson_p':pp,'spearman_r':rs,'spearman_p':ps,
        'f1_scores':f1_cic_list,'shap_consistency':shap_con_cic},
       os.path.join(DNN_DIR,"rq3_dnn_cic.pkl"))
print(f"  Pearson r={rp:.4f} p={pp:.4f} | Spearman r={rs:.4f} p={ps:.4f}")

# ── Fidelity ──────────────────────────────────────────────────────────
print("\nFidelity hesaplanıyor...")
ref_cic.eval()
def dnn_cic_pred(X_np):
    with torch.no_grad():
        return ref_cic(torch.FloatTensor(X_np).to(DEVICE)).argmax(dim=1).cpu().numpy()

f1_orig_cic = f1_score(y_orn_cic, dnn_cic_pred(X_orn_cic), average='macro')
X_mean_cic  = np.mean(X_cic, axis=0)
fid_rows = []
for n in range(1,11):
    Xs = X_orn_cic.copy(); Xl = X_orn_cic.copy()
    for i in range(500):
        Xs[i, np.argsort(shap_cic[i])[-n:]]          = X_mean_cic[np.argsort(shap_cic[i])[-n:]]
        Xl[i, np.argsort(np.abs(lime_cic[i]))[-n:]]  = X_mean_cic[np.argsort(np.abs(lime_cic[i]))[-n:]]
    f1s = f1_score(y_orn_cic, dnn_cic_pred(Xs), average='macro')
    f1l = f1_score(y_orn_cic, dnn_cic_pred(Xl), average='macro')
    fid_rows.append({'model':'DNN','xai':'SHAP','n_maskele':n,'f1_orig':round(f1_orig_cic,4),'f1_mask':round(f1s,4),'delta_f1':round(f1_orig_cic-f1s,4)})
    fid_rows.append({'model':'DNN','xai':'LIME','n_maskele':n,'f1_orig':round(f1_orig_cic,4),'f1_mask':round(f1l,4),'delta_f1':round(f1_orig_cic-f1l,4)})
    print(f"  n={n:2d} SHAP ΔF1={f1_orig_cic-f1s:.4f}  LIME ΔF1={f1_orig_cic-f1l:.4f}")

df_fid_cic_dnn = pd.DataFrame(fid_rows)
kaydet(df_fid_cic_dnn, os.path.join(DNN_DIR,"fidelity_dnn_cic.pkl"))

# ── XAI-Score ─────────────────────────────────────────────────────────
CIC_DIR_CK = str(CICIDS2017_DIR)
df_xai_cic_orig = yukle(os.path.join(CIC_DIR_CK,"df_xai_score_cicids.pkl"))

def normalize(d, inverse=False):
    vals=np.array(list(d.values()),dtype=float); mn,mx=vals.min(),vals.max()
    if mx==mn: return {k:0.5 for k in d}
    n={k:(v-mn)/(mx-mn) for k,v in d.items()}
    return {k:1-v for k,v in n.items()} if inverse else n

perturb_c  = dict(zip(df_xai_cic_orig['Model'],df_xai_cic_orig['Perturb']))
varyans_c  = dict(zip(df_xai_cic_orig['Model'],df_xai_cic_orig['Varyans']))
fidelity_c = dict(zip(df_xai_cic_orig['Model'],df_xai_cic_orig['Fidelity_5']))
sure_c     = dict(zip(df_xai_cic_orig['Model'],df_xai_cic_orig['Sure_ms']))

perturb_c['DNN']  = round(float(np.nanmean(rq1_sonuc['pct3']['shap']['spearman'])),4)
varyans_c['DNN']  = round(rq2_cic_ozet['spearman'],4)
fidelity_c['DNN'] = round(float(df_fid_cic_dnn[(df_fid_cic_dnn['xai']=='SHAP')&(df_fid_cic_dnn['n_maskele']==5)]['delta_f1'].values[0]),4)
sure_c['DNN']     = round(xai_dnn_cic['shap_sure'],2)

pn=normalize(perturb_c); vn=normalize(varyans_c)
fn_=normalize(fidelity_c); sn=normalize(sure_c,inverse=True)

TUM=list(perturb_c.keys())
xai_sc={m:round(0.25*pn[m]+0.25*vn[m]+0.25*fn_[m]+0.25*sn[m],4) for m in TUM}
df_xai_cic_dnn=pd.DataFrame({'Model':TUM,'Perturb':[perturb_c[m] for m in TUM],
    'Varyans':[varyans_c[m] for m in TUM],'Fidelity_5':[fidelity_c[m] for m in TUM],
    'Sure_ms':[sure_c[m] for m in TUM],'XAI_Score':[xai_sc[m] for m in TUM]
}).sort_values('XAI_Score',ascending=False).reset_index(drop=True)

kaydet(df_xai_cic_dnn, os.path.join(DNN_DIR,"df_xai_score_dnn_cic.pkl"))
print(f"\n📊 XAI-Score (DNN dahil, CIC-IDS-2017):")
print(df_xai_cic_dnn.to_string(index=False))
print("\n✅ Tüm CIC-IDS-2017 DNN analizleri tamamlandı!")

In [ ]:
# ── Fidelity + XAI-Score — DNN CIC (düzeltme) ────────────────────────
from sklearn.metrics import f1_score

# Referans modeli GPU'ya taşı
ref_cic = next(r['model_obj'] for r in dnn_cic_sonuclar
               if r['seed']==0 and r['fold']==0)
ref_cic = ref_cic.to(DEVICE)
ref_cic.eval()

def dnn_cic_pred(X_np):
    ref_cic.eval()
    with torch.no_grad():
        return ref_cic(torch.FloatTensor(X_np).to(DEVICE)).argmax(dim=1).cpu().numpy()

f1_orig_cic = f1_score(y_orn_cic, dnn_cic_pred(X_orn_cic), average='macro')
X_mean_cic  = np.mean(X_cic, axis=0)
print(f"Orijinal F1: {f1_orig_cic:.4f}")

fid_rows = []
for n in range(1, 11):
    Xs = X_orn_cic.copy(); Xl = X_orn_cic.copy()
    for i in range(500):
        Xs[i, np.argsort(shap_cic[i])[-n:]]         = X_mean_cic[np.argsort(shap_cic[i])[-n:]]
        Xl[i, np.argsort(np.abs(lime_cic[i]))[-n:]] = X_mean_cic[np.argsort(np.abs(lime_cic[i]))[-n:]]
    f1s = f1_score(y_orn_cic, dnn_cic_pred(Xs), average='macro')
    f1l = f1_score(y_orn_cic, dnn_cic_pred(Xl), average='macro')
    fid_rows.append({'model':'DNN','xai':'SHAP','n_maskele':n,'f1_orig':round(f1_orig_cic,4),'f1_mask':round(f1s,4),'delta_f1':round(f1_orig_cic-f1s,4)})
    fid_rows.append({'model':'DNN','xai':'LIME','n_maskele':n,'f1_orig':round(f1_orig_cic,4),'f1_mask':round(f1l,4),'delta_f1':round(f1_orig_cic-f1l,4)})
    print(f"  n={n:2d}  SHAP ΔF1={f1_orig_cic-f1s:.4f}  LIME ΔF1={f1_orig_cic-f1l:.4f}")

df_fid_cic_dnn = pd.DataFrame(fid_rows)
kaydet(df_fid_cic_dnn, os.path.join(DNN_DIR, "fidelity_dnn_cic.pkl"))

# ── XAI-Score ─────────────────────────────────────────────────────────
def normalize(d, inverse=False):
    vals=np.array(list(d.values()),dtype=float); mn,mx=vals.min(),vals.max()
    if mx==mn: return {k:0.5 for k in d}
    n={k:(v-mn)/(mx-mn) for k,v in d.items()}
    return {k:1-v for k,v in n.items()} if inverse else n

CIC_DIR_CK = str(CICIDS2017_DIR)
df_xai_cic_orig = yukle(os.path.join(CIC_DIR_CK, "df_xai_score_cicids.pkl"))

perturb_c  = dict(zip(df_xai_cic_orig['Model'], df_xai_cic_orig['Perturb']))
varyans_c  = dict(zip(df_xai_cic_orig['Model'], df_xai_cic_orig['Varyans']))
fidelity_c = dict(zip(df_xai_cic_orig['Model'], df_xai_cic_orig['Fidelity_5']))
sure_c     = dict(zip(df_xai_cic_orig['Model'], df_xai_cic_orig['Sure_ms']))

rq1_sonuc    = yukle(os.path.join(DNN_DIR, "perturb_dnn_cic.pkl"))
rq2_cic_ozet = yukle(os.path.join(DNN_DIR, "rq2_dnn_cic_ozet.pkl"))
xai_dnn_cic  = yukle(os.path.join(DNN_DIR, "xai_dnn_cic.pkl"))

perturb_c['DNN']  = round(float(np.nanmean(rq1_sonuc['pct3']['shap']['spearman'])), 4)
varyans_c['DNN']  = round(rq2_cic_ozet['spearman'], 4)
fidelity_c['DNN'] = round(float(df_fid_cic_dnn[(df_fid_cic_dnn['xai']=='SHAP')&(df_fid_cic_dnn['n_maskele']==5)]['delta_f1'].values[0]), 4)
sure_c['DNN']     = round(xai_dnn_cic['shap_sure'], 2)

pn=normalize(perturb_c); vn=normalize(varyans_c)
fn_=normalize(fidelity_c); sn=normalize(sure_c, inverse=True)

TUM = list(perturb_c.keys())
xai_sc = {m: round(0.25*pn[m]+0.25*vn[m]+0.25*fn_[m]+0.25*sn[m], 4) for m in TUM}

df_xai_cic_dnn = pd.DataFrame({
    'Model'     : TUM,
    'Perturb'   : [perturb_c[m]  for m in TUM],
    'Varyans'   : [varyans_c[m]  for m in TUM],
    'Fidelity_5': [fidelity_c[m] for m in TUM],
    'Sure_ms'   : [sure_c[m]     for m in TUM],
    'XAI_Score' : [xai_sc[m]     for m in TUM],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

kaydet(df_xai_cic_dnn, os.path.join(DNN_DIR, "df_xai_score_dnn_cic.pkl"))
print(f"\n📊 XAI-Score (DNN dahil, CIC-IDS-2017):")
print(df_xai_cic_dnn.to_string(index=False))
print("\n✅ Tüm CIC-IDS-2017 DNN analizleri tamamlandı!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LSTM MİMARİSİ — Tabular IDS verisi için
# Her özellik = 1 zaman adımı (seq_len = n_features, input_size = 1)
# ══════════════════════════════════════════════════════════════════════

class IDS_LSTM(nn.Module):
    def __init__(self, seq_len, num_classes, hidden=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=False
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (batch, seq_len) → (batch, seq_len, 1)
        x = x.unsqueeze(-1)
        out, (hn, _) = self.lstm(x)
        return self.classifier(hn[-1])  # son katmanın son hidden state'i

    def predict_proba(self, X_np):
        self.eval()
        with torch.no_grad():
            t = torch.FloatTensor(X_np).to(DEVICE)
            return torch.softmax(self.forward(t), dim=1).cpu().numpy()


def lstm_egit(X_train, y_train, X_val, y_val,
              seq_len, num_classes,
              epochs=100, batch_size=256, patience=10, lr=1e-3):
    model = IDS_LSTM(seq_len, num_classes).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss()

    Xt = torch.FloatTensor(X_train).to(DEVICE)
    yt = torch.LongTensor(y_train).to(DEVICE)
    Xv = torch.FloatTensor(X_val).to(DEVICE)
    yv = torch.LongTensor(y_val).to(DEVICE)

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    en_iyi_val, patience_say, en_iyi_state = -1, 0, None

    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xv).argmax(dim=1).cpu().numpy()
        val_f1 = f1_score(yv.cpu().numpy(), val_pred, average='macro')
        scheduler.step(-val_f1)

        if val_f1 > en_iyi_val:
            en_iyi_val   = val_f1
            en_iyi_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_say = 0
        else:
            patience_say += 1
            if patience_say >= patience:
                break

    model.load_state_dict(en_iyi_state)
    return model, en_iyi_val

print("✅ LSTM mimarisi hazır")
print(f"  Edge-IIoT : seq_len=74, input_size=1")
print(f"  CIC-IDS   : seq_len=78, input_size=1")

# ── Edge-IIoT LSTM Eğitimi ────────────────────────────────────────────
LSTM_EDGE_PATH = os.path.join(DNN_DIR, "lstm_edge_modeller.pkl")

lstm_edge_sonuclar = yukle(LSTM_EDGE_PATH) if kontrol(LSTM_EDGE_PATH) else []
tamamlanan_lstm_edge = {(r['seed'], r['fold']) for r in lstm_edge_sonuclar}

print("\nLSTM Edge-IIoT eğitimi başlıyor...")
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_edge, y_edge)):
        if (seed, fold) in tamamlanan_lstm_edge:
            print(f"  ♻  seed={seed} fold={fold} — atlandı")
            continue

        t0 = time.time()
        model, val_f1 = lstm_egit(
            X_edge[tr_idx], y_edge[tr_idx],
            X_edge[val_idx], y_edge[val_idx],
            seq_len=INPUT_DIM_EDGE, num_classes=NUM_CLASS_EDGE
        )
        sure = time.time() - t0
        model.cpu()

        lstm_edge_sonuclar.append({
            'seed': seed, 'fold': fold,
            'f1_macro': round(val_f1, 4),
            'model_obj': model,
            'sure': round(sure, 1)
        })
        kaydet(lstm_edge_sonuclar, LSTM_EDGE_PATH)
        print(f"  ✅ seed={seed} fold={fold}  F1={val_f1:.4f}  ({sure:.0f}s)")

f1_lstm_edge = [r['f1_macro'] for r in lstm_edge_sonuclar]
print(f"\n📊 LSTM Edge-IIoT: F1={np.mean(f1_lstm_edge):.4f} ± {np.std(f1_lstm_edge):.4f}")

In [ ]:
# ── LSTM CIC-IDS-2017 Eğitimi ─────────────────────────────────────────
LSTM_CIC_PATH = os.path.join(DNN_DIR, "lstm_cic_modeller.pkl")

lstm_cic_sonuclar = yukle(LSTM_CIC_PATH) if kontrol(LSTM_CIC_PATH) else []
tamamlanan_lstm_cic = {(r['seed'], r['fold']) for r in lstm_cic_sonuclar}

print("LSTM CIC-IDS-2017 eğitimi başlıyor...")
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cic, y_cic)):
        if (seed, fold) in tamamlanan_lstm_cic:
            print(f"  ♻  seed={seed} fold={fold} — atlandı")
            continue

        t0 = time.time()
        model, val_f1 = lstm_egit(
            X_cic[tr_idx], y_cic[tr_idx],
            X_cic[val_idx], y_cic[val_idx],
            seq_len=INPUT_DIM_CIC, num_classes=NUM_CLASS_CIC
        )
        sure = time.time() - t0
        model.cpu()

        lstm_cic_sonuclar.append({
            'seed': seed, 'fold': fold,
            'f1_macro': round(val_f1, 4),
            'model_obj': model,
            'sure': round(sure, 1)
        })
        kaydet(lstm_cic_sonuclar, LSTM_CIC_PATH)
        print(f"  ✅ seed={seed} fold={fold}  F1={val_f1:.4f}  ({sure:.0f}s)")

f1_lstm_cic = [r['f1_macro'] for r in lstm_cic_sonuclar]
print(f"\n📊 LSTM CIC-IDS-2017: F1={np.mean(f1_lstm_cic):.4f} ± {np.std(f1_lstm_cic):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LSTM EDGE-IIoT — XAI + RQ1 + RQ2 + RQ3 + Fidelity + XAI-Score
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import f1_score
from itertools import combinations
import lime.lime_tabular

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean: vec[i] = fval; break
    return vec

def normalize(d, inverse=False):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    if mx == mn: return {k: 0.5 for k in d}
    n = {k: (v-mn)/(mx-mn) for k, v in d.items()}
    return {k: 1-v for k, v in n.items()} if inverse else n

def lstm_shap(model, background, samples):
    """LSTM için cuDNN fix: train mode + cudnn disable"""
    torch.backends.cudnn.enabled = False
    model.train()
    exp = shap.GradientExplainer(model, background)
    sv  = np.array(exp.shap_values(samples))
    torch.backends.cudnn.enabled = True
    model.eval()
    return exp, sv

# ── Referans model ────────────────────────────────────────────────────
ref_lstm_edge = next(r['model_obj'] for r in lstm_edge_sonuclar
                     if r['seed']==0 and r['fold']==0)
ref_lstm_edge = ref_lstm_edge.to(DEVICE)

# ── Örnek setleri ─────────────────────────────────────────────────────
np.random.seed(42)
orn_idx_e = np.random.choice(len(X_edge), size=500, replace=False)
X_orn_e   = X_edge[orn_idx_e]; y_orn_e = y_edge[orn_idx_e]

np.random.seed(0)
bg_idx_e  = np.random.choice(len(X_edge), size=100, replace=False)
X_bg_e_t  = torch.FloatTensor(X_edge[bg_idx_e]).to(DEVICE)
X_orn_e_t = torch.FloatTensor(X_orn_e).to(DEVICE)

def lstm_edge_proba(X_np):
    ref_lstm_edge.eval()
    with torch.no_grad():
        return torch.softmax(ref_lstm_edge(torch.FloatTensor(X_np).to(DEVICE)), dim=1).cpu().numpy()

# ── SHAP ─────────────────────────────────────────────────────────────
print("SHAP (Edge LSTM)...")
t0 = time.time()
exp_le, sv_raw = lstm_shap(ref_lstm_edge, X_bg_e_t, X_orn_e_t)
if sv_raw.ndim == 3:
    shap_le = np.mean(np.abs(sv_raw), axis=2) if sv_raw.shape[0]==500 \
              else np.mean(np.abs(sv_raw), axis=0)
else:
    shap_le = np.abs(sv_raw)
shap_sure_le = (time.time()-t0)/500*1000
print(f"  ✅ {shap_le.shape}, {shap_sure_le:.1f} ms/örnek")

# ── LIME ─────────────────────────────────────────────────────────────
lime_exp_le = lime.lime_tabular.LimeTabularExplainer(
    X_edge, feature_names=fn_edge, class_names=cn_edge,
    discretize_continuous=True, random_state=42)

print("LIME (Edge LSTM)...")
t0 = time.time()
lime_le = []
for i in range(500):
    exp = lime_exp_le.explain_instance(X_orn_e[i], lstm_edge_proba,
                                        num_features=len(fn_edge), num_samples=1000)
    lime_le.append(lime_to_vec(dict(exp.as_list()), fn_edge))
    if (i+1)%100==0: print(f"  LIME {i+1}/500...")
lime_le = np.array(lime_le)
lime_sure_le = (time.time()-t0)/500*1000
print(f"  ✅ {lime_le.shape}, {lime_sure_le:.1f} ms/örnek")

xai_lstm_edge = {'shap_values':shap_le, 'lime_values':lime_le,
                 'shap_sure':shap_sure_le, 'lime_sure':lime_sure_le}
kaydet(xai_lstm_edge, os.path.join(DNN_DIR, "xai_lstm_edge.pkl"))

# ── RQ1 ──────────────────────────────────────────────────────────────
print("\nRQ1 (Edge LSTM)...")
ORANLAR = {'pct1':0.01, 'pct3':0.03, 'pct5':0.05}
rq1_le  = {ok: {'shap':{'spearman':[],'jaccard_5':[],'jaccard_10':[]},
                'lime':{'spearman':[],'jaccard_5':[],'jaccard_10':[]}}
           for ok in ORANLAR}
t0 = time.time()
for ok, oran in ORANLAR.items():
    ss,sj5,sj10,ls,lj5,lj10 = [],[],[],[],[],[]
    for i in range(500):
        xp = X_orn_e[i].copy()
        ni = max(1, int(len(fn_edge)*oran))
        pi = np.random.choice(len(fn_edge), size=ni, replace=False)
        xp[pi] += np.random.normal(0, 0.1, size=ni)

        # SHAP pertürbe
        torch.backends.cudnn.enabled = False
        ref_lstm_edge.train()
        sv = np.array(exp_le.shap_values(torch.FloatTensor(xp.reshape(1,-1)).to(DEVICE)))
        torch.backends.cudnn.enabled = True
        ref_lstm_edge.eval()
        sv = np.mean(np.abs(sv[0]),axis=1) if sv.ndim==3 else np.abs(sv[0])

        sp,_ = spearmanr(shap_le[i], sv)
        ss.append(sp); sj5.append(jaccard_top_k(shap_le[i],sv,5))
        sj10.append(jaccard_top_k(shap_le[i],sv,10))

        # LIME pertürbe
        ep = lime_exp_le.explain_instance(xp, lstm_edge_proba,
                                           num_features=len(fn_edge), num_samples=1000)
        lv = lime_to_vec(dict(ep.as_list()), fn_edge)
        sl,_ = spearmanr(lime_le[i], lv)
        ls.append(sl); lj5.append(jaccard_top_k(lime_le[i],lv,5))
        lj10.append(jaccard_top_k(lime_le[i],lv,10))

        if (i+1)%100==0: print(f"  [{ok}] {i+1}/500...")

    rq1_le[ok]['shap'].update({'spearman':ss,'jaccard_5':sj5,'jaccard_10':sj10})
    rq1_le[ok]['lime'].update({'spearman':ls,'jaccard_5':lj5,'jaccard_10':lj10})
    print(f"  ✅ %{int(oran*100):2d} SHAP={np.nanmean(ss):.4f} LIME={np.nanmean(ls):.4f}")

kaydet(rq1_le, os.path.join(DNN_DIR, "perturb_lstm_edge.pkl"))
print(f"  ⏱ {(time.time()-t0)/60:.1f} dk")

# ── RQ2 ──────────────────────────────────────────────────────────────
print("\nRQ2 (Edge LSTM)...")
RQ2_LE_PATH = os.path.join(DNN_DIR, "rq2_lstm_edge.pkl")
N_RQ2 = 100
np.random.seed(42)
rq2_idx_e = np.random.choice(len(X_edge), size=N_RQ2, replace=False)
X_rq2_e   = torch.FloatTensor(X_edge[rq2_idx_e]).to(DEVICE)

shap_per_le = yukle(RQ2_LE_PATH) if kontrol(RQ2_LE_PATH) else {}
t0 = time.time()
for kayit in lstm_edge_sonuclar:
    key = (kayit['seed'], kayit['fold'])
    if key in shap_per_le: continue
    m = kayit['model_obj'].to(DEVICE)
    torch.backends.cudnn.enabled = False
    m.train()
    sv = np.array(shap.GradientExplainer(m, X_bg_e_t).shap_values(X_rq2_e))
    torch.backends.cudnn.enabled = True
    m.eval()
    if sv.ndim==3:
        sv = np.mean(np.abs(sv),axis=2) if sv.shape[0]==N_RQ2 else np.mean(np.abs(sv),axis=0)
    else: sv = np.abs(sv)
    shap_per_le[key] = sv; m.cpu()
    done = len(shap_per_le)
    eta  = (time.time()-t0)/done*(50-done)/60
    print(f"  ✅ seed={key[0]} fold={key[1]}  (~{eta:.0f} dk kaldı)")
    kaydet(shap_per_le, RQ2_LE_PATH)

keys_le = list(shap_per_le.keys())
sp_l,j5_l,j10_l = [],[],[]
for k1,k2 in combinations(keys_le,2):
    for i in range(N_RQ2):
        sp,_ = spearmanr(shap_per_le[k1][i], shap_per_le[k2][i])
        sp_l.append(sp)
        j5_l.append(jaccard_top_k(shap_per_le[k1][i],shap_per_le[k2][i],5))
        j10_l.append(jaccard_top_k(shap_per_le[k1][i],shap_per_le[k2][i],10))

rq2_le = {'spearman':float(np.nanmean(sp_l)), 'jaccard_5':float(np.nanmean(j5_l)),
          'jaccard_10':float(np.nanmean(j10_l)), 'n_pairs':len(list(combinations(keys_le,2)))}
kaydet(rq2_le, os.path.join(DNN_DIR, "rq2_lstm_edge_ozet.pkl"))
print(f"  RQ2: Spearman={rq2_le['spearman']:.4f} J@5={rq2_le['jaccard_5']:.4f}")

# ── RQ3 ──────────────────────────────────────────────────────────────
print("\nRQ3 (Edge LSTM)...")
f1_le_list, shap_con_le = [],[]
for key in keys_le:
    f1 = next(r['f1_macro'] for r in lstm_edge_sonuclar
              if r['seed']==key[0] and r['fold']==key[1])
    f1_le_list.append(f1)
    sp_m = []
    for k2 in keys_le:
        if k2==key: continue
        for i in range(N_RQ2):
            sp,_ = spearmanr(shap_per_le[key][i], shap_per_le[k2][i])
            sp_m.append(sp)
    shap_con_le.append(np.nanmean(sp_m))

rp,pp = pearsonr(f1_le_list, shap_con_le)
rs,ps = spearmanr(f1_le_list, shap_con_le)
kaydet({'pearson_r':rp,'pearson_p':pp,'spearman_r':rs,'spearman_p':ps,
        'f1_scores':f1_le_list,'shap_consistency':shap_con_le},
       os.path.join(DNN_DIR, "rq3_lstm_edge.pkl"))
print(f"  Pearson r={rp:.4f} p={pp:.4f} | Spearman r={rs:.4f} p={ps:.4f}")

# ── Fidelity ──────────────────────────────────────────────────────────
print("\nFidelity (Edge LSTM)...")
ref_lstm_edge.eval()
def lstm_edge_pred(X_np):
    ref_lstm_edge.eval()
    with torch.no_grad():
        return ref_lstm_edge(torch.FloatTensor(X_np).to(DEVICE)).argmax(dim=1).cpu().numpy()

f1_orig_e = f1_score(y_orn_e, lstm_edge_pred(X_orn_e), average='macro')
X_mean_e  = np.mean(X_edge, axis=0)
fid_le    = []
print(f"  Orijinal F1: {f1_orig_e:.4f}")
for n in range(1, 11):
    Xs = X_orn_e.copy(); Xl = X_orn_e.copy()
    for i in range(500):
        Xs[i, np.argsort(shap_le[i])[-n:]]          = X_mean_e[np.argsort(shap_le[i])[-n:]]
        Xl[i, np.argsort(np.abs(lime_le[i]))[-n:]]  = X_mean_e[np.argsort(np.abs(lime_le[i]))[-n:]]
    f1s = f1_score(y_orn_e, lstm_edge_pred(Xs), average='macro')
    f1l = f1_score(y_orn_e, lstm_edge_pred(Xl), average='macro')
    fid_le.append({'model':'LSTM','xai':'SHAP','n_maskele':n,'f1_orig':round(f1_orig_e,4),'f1_mask':round(f1s,4),'delta_f1':round(f1_orig_e-f1s,4)})
    fid_le.append({'model':'LSTM','xai':'LIME','n_maskele':n,'f1_orig':round(f1_orig_e,4),'f1_mask':round(f1l,4),'delta_f1':round(f1_orig_e-f1l,4)})
    print(f"  n={n:2d}  SHAP ΔF1={f1_orig_e-f1s:.4f}  LIME ΔF1={f1_orig_e-f1l:.4f}")

df_fid_lstm_edge = pd.DataFrame(fid_le)
kaydet(df_fid_lstm_edge, os.path.join(DNN_DIR, "fidelity_lstm_edge.pkl"))

# ── XAI-Score ─────────────────────────────────────────────────────────
df_xai_dnn_edge = yukle(os.path.join(DNN_DIR, "df_xai_score_dnn_edge.pkl"))
perturb_x  = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Perturb']))
varyans_x  = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Varyans']))
fidelity_x = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Fidelity_d5']))
sure_x     = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Sure_ms']))

perturb_x['LSTM']  = round(float(np.nanmean(rq1_le['pct3']['shap']['spearman'])), 4)
varyans_x['LSTM']  = round(rq2_le['spearman'], 4)
fidelity_x['LSTM'] = round(float(df_fid_lstm_edge[(df_fid_lstm_edge['xai']=='SHAP')&
                                                    (df_fid_lstm_edge['n_maskele']==5)]['delta_f1'].values[0]), 4)
sure_x['LSTM']     = round(shap_sure_le, 2)

TUM = list(perturb_x.keys())
pn=normalize(perturb_x); vn=normalize(varyans_x)
fn_=normalize(fidelity_x); sn=normalize(sure_x, inverse=True)
xai_sc = {m: round(0.25*pn[m]+0.25*vn[m]+0.25*fn_[m]+0.25*sn[m], 4) for m in TUM}

df_xai_lstm_edge = pd.DataFrame({
    'Model'      : TUM,
    'Perturb'    : [perturb_x[m]  for m in TUM],
    'Varyans'    : [varyans_x[m]  for m in TUM],
    'Fidelity_d5': [fidelity_x[m] for m in TUM],
    'Sure_ms'    : [sure_x[m]     for m in TUM],
    'XAI_Score'  : [xai_sc[m]     for m in TUM],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

kaydet(df_xai_lstm_edge, os.path.join(DNN_DIR, "df_xai_score_lstm_edge.pkl"))
print(f"\n📊 XAI-Score (DNN+LSTM, Edge-IIoT):")
print(df_xai_lstm_edge.to_string(index=False))
print("\n✅ LSTM Edge-IIoT tüm analizler tamamlandı!")

In [ ]:
# ── Fidelity + XAI-Score (Edge LSTM) — düzeltme ──────────────────────
from sklearn.metrics import f1_score

# Modeli GPU'ya geri taşı
ref_lstm_edge = next(r['model_obj'] for r in lstm_edge_sonuclar
                     if r['seed']==0 and r['fold']==0)
ref_lstm_edge = ref_lstm_edge.to(DEVICE)
ref_lstm_edge.eval()

def lstm_edge_pred(X_np):
    ref_lstm_edge.eval()
    with torch.no_grad():
        return ref_lstm_edge(torch.FloatTensor(X_np).to(DEVICE)).argmax(dim=1).cpu().numpy()

f1_orig_e = f1_score(y_orn_e, lstm_edge_pred(X_orn_e), average='macro')
X_mean_e  = np.mean(X_edge, axis=0)
fid_le    = []
print(f"Orijinal F1: {f1_orig_e:.4f}")

for n in range(1, 11):
    Xs = X_orn_e.copy(); Xl = X_orn_e.copy()
    for i in range(500):
        Xs[i, np.argsort(shap_le[i])[-n:]]          = X_mean_e[np.argsort(shap_le[i])[-n:]]
        Xl[i, np.argsort(np.abs(lime_le[i]))[-n:]]  = X_mean_e[np.argsort(np.abs(lime_le[i]))[-n:]]
    f1s = f1_score(y_orn_e, lstm_edge_pred(Xs), average='macro')
    f1l = f1_score(y_orn_e, lstm_edge_pred(Xl), average='macro')
    fid_le.append({'model':'LSTM','xai':'SHAP','n_maskele':n,'f1_orig':round(f1_orig_e,4),'f1_mask':round(f1s,4),'delta_f1':round(f1_orig_e-f1s,4)})
    fid_le.append({'model':'LSTM','xai':'LIME','n_maskele':n,'f1_orig':round(f1_orig_e,4),'f1_mask':round(f1l,4),'delta_f1':round(f1_orig_e-f1l,4)})
    print(f"  n={n:2d}  SHAP ΔF1={f1_orig_e-f1s:.4f}  LIME ΔF1={f1_orig_e-f1l:.4f}")

df_fid_lstm_edge = pd.DataFrame(fid_le)
kaydet(df_fid_lstm_edge, os.path.join(DNN_DIR, "fidelity_lstm_edge.pkl"))

# ── XAI-Score ─────────────────────────────────────────────────────────
df_xai_dnn_edge = yukle(os.path.join(DNN_DIR, "df_xai_score_dnn_edge.pkl"))
perturb_x  = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Perturb']))
varyans_x  = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Varyans']))
fidelity_x = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Fidelity_d5']))
sure_x     = dict(zip(df_xai_dnn_edge['Model'], df_xai_dnn_edge['Sure_ms']))

rq1_le   = yukle(os.path.join(DNN_DIR, "perturb_lstm_edge.pkl"))
rq2_le   = yukle(os.path.join(DNN_DIR, "rq2_lstm_edge_ozet.pkl"))
xai_le   = yukle(os.path.join(DNN_DIR, "xai_lstm_edge.pkl"))

perturb_x['LSTM']  = round(float(np.nanmean(rq1_le['pct3']['shap']['spearman'])), 4)
varyans_x['LSTM']  = round(rq2_le['spearman'], 4)
fidelity_x['LSTM'] = round(float(df_fid_lstm_edge[(df_fid_lstm_edge['xai']=='SHAP')&
                                                    (df_fid_lstm_edge['n_maskele']==5)]['delta_f1'].values[0]), 4)
sure_x['LSTM']     = round(xai_le['shap_sure'], 2)

def normalize(d, inverse=False):
    vals=np.array(list(d.values()),dtype=float); mn,mx=vals.min(),vals.max()
    if mx==mn: return {k:0.5 for k in d}
    n={k:(v-mn)/(mx-mn) for k,v in d.items()}
    return {k:1-v for k,v in n.items()} if inverse else n

TUM = list(perturb_x.keys())
pn=normalize(perturb_x); vn=normalize(varyans_x)
fn_=normalize(fidelity_x); sn=normalize(sure_x, inverse=True)
xai_sc = {m: round(0.25*pn[m]+0.25*vn[m]+0.25*fn_[m]+0.25*sn[m], 4) for m in TUM}

df_xai_lstm_edge = pd.DataFrame({
    'Model'      : TUM,
    'Perturb'    : [perturb_x[m]  for m in TUM],
    'Varyans'    : [varyans_x[m]  for m in TUM],
    'Fidelity_d5': [fidelity_x[m] for m in TUM],
    'Sure_ms'    : [sure_x[m]     for m in TUM],
    'XAI_Score'  : [xai_sc[m]     for m in TUM],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

kaydet(df_xai_lstm_edge, os.path.join(DNN_DIR, "df_xai_score_lstm_edge.pkl"))
print(f"\n📊 XAI-Score (DNN+LSTM, Edge-IIoT):")
print(df_xai_lstm_edge.to_string(index=False))
print("\n✅ LSTM Edge-IIoT tamamlandı!")

In [ ]:
import os, pickle, gc, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.stats import spearmanr, pearsonr
from itertools import combinations
import shap, lime, lime.lime_tabular
warnings.filterwarnings('ignore')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

def kaydet(nesne, yol):
    with open(yol, 'wb') as f: pickle.dump(nesne, f)
    print(f"  💾 {os.path.basename(yol)}")

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

def kontrol(yol): return os.path.exists(yol)

def jaccard_top_k(v1, v2, k):
    t1=set(np.argsort(np.abs(v1))[-k:]); t2=set(np.argsort(np.abs(v2))[-k:])
    return len(t1&t2)/len(t1|t2) if len(t1|t2)>0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec=np.zeros(len(feature_names))
    for fname,fval in lime_dict.items():
        clean=fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i,fn in enumerate(feature_names):
            if fn==clean: vec[i]=fval; break
    return vec

def normalize(d, inverse=False):
    vals=np.array(list(d.values()),dtype=float); mn,mx=vals.min(),vals.max()
    if mx==mn: return {k:0.5 for k in d}
    n={k:(v-mn)/(mx-mn) for k,v in d.items()}
    return {k:1-v for k,v in n.items()} if inverse else n

# ── Model mimarileri (yeniden tanımla) ───────────────────────────────
class IDS_DNN(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,num_classes))
    def forward(self, x): return self.network(x)

class IDS_LSTM(nn.Module):
    def __init__(self, seq_len, num_classes, hidden=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers>1 else 0)
        self.classifier = nn.Sequential(
            nn.Linear(hidden,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64,num_classes))
    def forward(self, x):
        x=x.unsqueeze(-1); out,(hn,_)=self.lstm(x); return self.classifier(hn[-1])

# ── Verileri yükle ────────────────────────────────────────────────────
print("Veriler yükleniyor...")
veri_edge = yukle(os.path.join(EDGE_DIR,"veri.pkl"))
X_edge    = veri_edge['X_multi_scaled']
y_edge    = veri_edge['y_multi']
fn_edge   = veri_edge['feature_names_multi']
cn_edge   = list(veri_edge['sinif_isimleri'])
if hasattr(X_edge,'values'): X_edge=X_edge.values
if hasattr(y_edge,'values'): y_edge=y_edge.values

veri_cic = yukle(os.path.join(CIC_DIR,"veri_cicids.pkl"))
X_cic    = veri_cic['X_scaled']
y_cic    = veri_cic['y']
fn_cic   = veri_cic['feature_names']
cn_cic   = list(veri_cic['sinif_isimleri'])
if hasattr(X_cic,'values'): X_cic=X_cic.values
if hasattr(y_cic,'values'): y_cic=y_cic.values

INPUT_DIM_CIC = X_cic.shape[1]
NUM_CLASS_CIC = len(np.unique(y_cic))

# ── LSTM CIC modellerini yükle ────────────────────────────────────────
print("LSTM CIC modelleri yükleniyor...")
lstm_cic_sonuclar = yukle(os.path.join(DNN_DIR,"lstm_cic_modeller.pkl"))
print(f"  ✅ {len(lstm_cic_sonuclar)} model yüklendi")
print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LSTM CIC-IDS-2017 — XAI + RQ1 + RQ2 + RQ3 + Fidelity + XAI-Score
# ══════════════════════════════════════════════════════════════════════

# ── Referans model ────────────────────────────────────────────────────
ref_lstm_cic = next(r['model_obj'] for r in lstm_cic_sonuclar
                    if r['seed']==0 and r['fold']==0)
ref_lstm_cic = ref_lstm_cic.to(DEVICE)

# ── Örnek setleri ─────────────────────────────────────────────────────
np.random.seed(42)
orn_idx_lc = np.random.choice(len(X_cic), size=500, replace=False)
X_orn_lc   = X_cic[orn_idx_lc]; y_orn_lc = y_cic[orn_idx_lc]

np.random.seed(0)
bg_idx_lc  = np.random.choice(len(X_cic), size=100, replace=False)
X_bg_lc_t  = torch.FloatTensor(X_cic[bg_idx_lc]).to(DEVICE)
X_orn_lc_t = torch.FloatTensor(X_orn_lc).to(DEVICE)

def lstm_cic_proba(X_np):
    ref_lstm_cic.eval()
    with torch.no_grad():
        return torch.softmax(ref_lstm_cic(torch.FloatTensor(X_np).to(DEVICE)), dim=1).cpu().numpy()

# ── SHAP ─────────────────────────────────────────────────────────────
print("SHAP (CIC LSTM)...")
t0 = time.time()
torch.backends.cudnn.enabled = False
ref_lstm_cic.train()
exp_lc  = shap.GradientExplainer(ref_lstm_cic, X_bg_lc_t)
sv_raw  = np.array(exp_lc.shap_values(X_orn_lc_t))
torch.backends.cudnn.enabled = True
ref_lstm_cic.eval()
if sv_raw.ndim==3:
    shap_lc = np.mean(np.abs(sv_raw),axis=2) if sv_raw.shape[0]==500 \
              else np.mean(np.abs(sv_raw),axis=0)
else: shap_lc = np.abs(sv_raw)
shap_sure_lc = (time.time()-t0)/500*1000
print(f"  ✅ {shap_lc.shape}, {shap_sure_lc:.1f} ms/örnek")

# ── LIME ─────────────────────────────────────────────────────────────
lime_exp_lc = lime.lime_tabular.LimeTabularExplainer(
    X_cic, feature_names=fn_cic, class_names=cn_cic,
    discretize_continuous=True, random_state=42)

print("LIME (CIC LSTM)...")
t0 = time.time()
lime_lc = []
for i in range(500):
    exp = lime_exp_lc.explain_instance(X_orn_lc[i], lstm_cic_proba,
                                        num_features=len(fn_cic), num_samples=1000)
    lime_lc.append(lime_to_vec(dict(exp.as_list()), fn_cic))
    if (i+1)%100==0: print(f"  LIME {i+1}/500...")
lime_lc = np.array(lime_lc)
lime_sure_lc = (time.time()-t0)/500*1000
print(f"  ✅ {lime_lc.shape}, {lime_sure_lc:.1f} ms/örnek")

xai_lstm_cic = {'shap_values':shap_lc,'lime_values':lime_lc,
                'shap_sure':shap_sure_lc,'lime_sure':lime_sure_lc}
kaydet(xai_lstm_cic, os.path.join(DNN_DIR,"xai_lstm_cic.pkl"))

# ── RQ1 ──────────────────────────────────────────────────────────────
print("\nRQ1 (CIC LSTM)...")
ORANLAR = {'pct1':0.01,'pct3':0.03,'pct5':0.05}
rq1_lc  = {ok:{'shap':{'spearman':[],'jaccard_5':[],'jaccard_10':[]},
               'lime':{'spearman':[],'jaccard_5':[],'jaccard_10':[]}} for ok in ORANLAR}
t0 = time.time()
for ok, oran in ORANLAR.items():
    ss,sj5,sj10,ls,lj5,lj10=[],[],[],[],[],[]
    for i in range(500):
        xp=X_orn_lc[i].copy(); ni=max(1,int(len(fn_cic)*oran))
        pi=np.random.choice(len(fn_cic),size=ni,replace=False); xp[pi]+=np.random.normal(0,0.1,size=ni)
        torch.backends.cudnn.enabled=False; ref_lstm_cic.train()
        sv=np.array(exp_lc.shap_values(torch.FloatTensor(xp.reshape(1,-1)).to(DEVICE)))
        torch.backends.cudnn.enabled=True; ref_lstm_cic.eval()
        sv=np.mean(np.abs(sv[0]),axis=1) if sv.ndim==3 else np.abs(sv[0])
        sp,_=spearmanr(shap_lc[i],sv); ss.append(sp)
        sj5.append(jaccard_top_k(shap_lc[i],sv,5)); sj10.append(jaccard_top_k(shap_lc[i],sv,10))
        ep=lime_exp_lc.explain_instance(xp,lstm_cic_proba,num_features=len(fn_cic),num_samples=1000)
        lv=lime_to_vec(dict(ep.as_list()),fn_cic); sl,_=spearmanr(lime_lc[i],lv)
        ls.append(sl); lj5.append(jaccard_top_k(lime_lc[i],lv,5)); lj10.append(jaccard_top_k(lime_lc[i],lv,10))
        if (i+1)%100==0: print(f"  [{ok}] {i+1}/500...")
    rq1_lc[ok]['shap'].update({'spearman':ss,'jaccard_5':sj5,'jaccard_10':sj10})
    rq1_lc[ok]['lime'].update({'spearman':ls,'jaccard_5':lj5,'jaccard_10':lj10})
    print(f"  ✅ %{int(oran*100):2d} SHAP={np.nanmean(ss):.4f} LIME={np.nanmean(ls):.4f}")
kaydet(rq1_lc, os.path.join(DNN_DIR,"perturb_lstm_cic.pkl"))
print(f"  ⏱ {(time.time()-t0)/60:.1f} dk")

# ── RQ2 ──────────────────────────────────────────────────────────────
print("\nRQ2 (CIC LSTM)...")
RQ2_LC_PATH = os.path.join(DNN_DIR,"rq2_lstm_cic.pkl")
N_RQ2=100
np.random.seed(42); rq2_idx_lc=np.random.choice(len(X_cic),size=N_RQ2,replace=False)
X_rq2_lc=torch.FloatTensor(X_cic[rq2_idx_lc]).to(DEVICE)
shap_per_lc = yukle(RQ2_LC_PATH) if kontrol(RQ2_LC_PATH) else {}
t0=time.time()
for kayit in lstm_cic_sonuclar:
    key=(kayit['seed'],kayit['fold'])
    if key in shap_per_lc: continue
    m=kayit['model_obj'].to(DEVICE)
    torch.backends.cudnn.enabled=False; m.train()
    sv=np.array(shap.GradientExplainer(m,X_bg_lc_t).shap_values(X_rq2_lc))
    torch.backends.cudnn.enabled=True; m.eval()
    if sv.ndim==3: sv=np.mean(np.abs(sv),axis=2) if sv.shape[0]==N_RQ2 else np.mean(np.abs(sv),axis=0)
    else: sv=np.abs(sv)
    shap_per_lc[key]=sv; m.cpu()
    done=len(shap_per_lc); eta=(time.time()-t0)/done*(50-done)/60
    print(f"  ✅ seed={key[0]} fold={key[1]}  (~{eta:.0f} dk kaldı)")
    kaydet(shap_per_lc,RQ2_LC_PATH)

keys_lc=list(shap_per_lc.keys()); sp_l,j5_l,j10_l=[],[],[]
for k1,k2 in combinations(keys_lc,2):
    for i in range(N_RQ2):
        sp,_=spearmanr(shap_per_lc[k1][i],shap_per_lc[k2][i]); sp_l.append(sp)
        j5_l.append(jaccard_top_k(shap_per_lc[k1][i],shap_per_lc[k2][i],5))
        j10_l.append(jaccard_top_k(shap_per_lc[k1][i],shap_per_lc[k2][i],10))
rq2_lc={'spearman':float(np.nanmean(sp_l)),'jaccard_5':float(np.nanmean(j5_l)),
        'jaccard_10':float(np.nanmean(j10_l)),'n_pairs':len(list(combinations(keys_lc,2)))}
kaydet(rq2_lc,os.path.join(DNN_DIR,"rq2_lstm_cic_ozet.pkl"))
print(f"  RQ2: Spearman={rq2_lc['spearman']:.4f} J@5={rq2_lc['jaccard_5']:.4f}")

# ── RQ3 ──────────────────────────────────────────────────────────────
print("\nRQ3 (CIC LSTM)...")
f1_lc_list,shap_con_lc=[],[]
for key in keys_lc:
    f1=next(r['f1_macro'] for r in lstm_cic_sonuclar if r['seed']==key[0] and r['fold']==key[1])
    f1_lc_list.append(f1)
    sp_m=[]
    for k2 in keys_lc:
        if k2==key: continue
        for i in range(N_RQ2):
            sp,_=spearmanr(shap_per_lc[key][i],shap_per_lc[k2][i]); sp_m.append(sp)
    shap_con_lc.append(np.nanmean(sp_m))
rp,pp=pearsonr(f1_lc_list,shap_con_lc); rs,ps=spearmanr(f1_lc_list,shap_con_lc)
kaydet({'pearson_r':rp,'pearson_p':pp,'spearman_r':rs,'spearman_p':ps,
        'f1_scores':f1_lc_list,'shap_consistency':shap_con_lc},
       os.path.join(DNN_DIR,"rq3_lstm_cic.pkl"))
print(f"  Pearson r={rp:.4f} p={pp:.4f} | Spearman r={rs:.4f} p={ps:.4f}")

# ── Fidelity ──────────────────────────────────────────────────────────
print("\nFidelity (CIC LSTM)...")
ref_lstm_cic = next(r['model_obj'] for r in lstm_cic_sonuclar
                    if r['seed']==0 and r['fold']==0)
ref_lstm_cic = ref_lstm_cic.to(DEVICE); ref_lstm_cic.eval()

def lstm_cic_pred(X_np):
    ref_lstm_cic.eval()
    with torch.no_grad():
        return ref_lstm_cic(torch.FloatTensor(X_np).to(DEVICE)).argmax(dim=1).cpu().numpy()

f1_orig_lc=f1_score(y_orn_lc,lstm_cic_pred(X_orn_lc),average='macro')
X_mean_lc=np.mean(X_cic,axis=0); fid_lc=[]
print(f"  Orijinal F1: {f1_orig_lc:.4f}")
for n in range(1,11):
    Xs=X_orn_lc.copy(); Xl=X_orn_lc.copy()
    for i in range(500):
        Xs[i,np.argsort(shap_lc[i])[-n:]]         =X_mean_lc[np.argsort(shap_lc[i])[-n:]]
        Xl[i,np.argsort(np.abs(lime_lc[i]))[-n:]] =X_mean_lc[np.argsort(np.abs(lime_lc[i]))[-n:]]
    f1s=f1_score(y_orn_lc,lstm_cic_pred(Xs),average='macro')
    f1l=f1_score(y_orn_lc,lstm_cic_pred(Xl),average='macro')
    fid_lc.append({'model':'LSTM','xai':'SHAP','n_maskele':n,'f1_orig':round(f1_orig_lc,4),'f1_mask':round(f1s,4),'delta_f1':round(f1_orig_lc-f1s,4)})
    fid_lc.append({'model':'LSTM','xai':'LIME','n_maskele':n,'f1_orig':round(f1_orig_lc,4),'f1_mask':round(f1l,4),'delta_f1':round(f1_orig_lc-f1l,4)})
    print(f"  n={n:2d}  SHAP ΔF1={f1_orig_lc-f1s:.4f}  LIME ΔF1={f1_orig_lc-f1l:.4f}")
df_fid_lstm_cic=pd.DataFrame(fid_lc)
kaydet(df_fid_lstm_cic,os.path.join(DNN_DIR,"fidelity_lstm_cic.pkl"))

# ── XAI-Score ─────────────────────────────────────────────────────────
df_xai_cic_dnn=yukle(os.path.join(DNN_DIR,"df_xai_score_dnn_cic.pkl"))
perturb_lc =dict(zip(df_xai_cic_dnn['Model'],df_xai_cic_dnn['Perturb']))
varyans_lc =dict(zip(df_xai_cic_dnn['Model'],df_xai_cic_dnn['Varyans']))
fidelity_lc=dict(zip(df_xai_cic_dnn['Model'],df_xai_cic_dnn['Fidelity_5']))
sure_lc    =dict(zip(df_xai_cic_dnn['Model'],df_xai_cic_dnn['Sure_ms']))

perturb_lc['LSTM'] =round(float(np.nanmean(rq1_lc['pct3']['shap']['spearman'])),4)
varyans_lc['LSTM'] =round(rq2_lc['spearman'],4)
fidelity_lc['LSTM']=round(float(df_fid_lstm_cic[(df_fid_lstm_cic['xai']=='SHAP')&(df_fid_lstm_cic['n_maskele']==5)]['delta_f1'].values[0]),4)
sure_lc['LSTM']    =round(xai_lstm_cic['shap_sure'],2)

TUM=list(perturb_lc.keys())
pn=normalize(perturb_lc); vn=normalize(varyans_lc)
fn_=normalize(fidelity_lc); sn=normalize(sure_lc,inverse=True)
xai_sc={m:round(0.25*pn[m]+0.25*vn[m]+0.25*fn_[m]+0.25*sn[m],4) for m in TUM}

df_xai_lstm_cic=pd.DataFrame({'Model':TUM,'Perturb':[perturb_lc[m] for m in TUM],
    'Varyans':[varyans_lc[m] for m in TUM],'Fidelity_5':[fidelity_lc[m] for m in TUM],
    'Sure_ms':[sure_lc[m] for m in TUM],'XAI_Score':[xai_sc[m] for m in TUM]
}).sort_values('XAI_Score',ascending=False).reset_index(drop=True)

kaydet(df_xai_lstm_cic,os.path.join(DNN_DIR,"df_xai_score_lstm_cic.pkl"))
print(f"\n📊 XAI-Score (DNN+LSTM, CIC-IDS-2017):")
print(df_xai_lstm_cic.to_string(index=False))
print("\n✅ LSTM CIC-IDS-2017 tüm analizler tamamlandı!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FGSM ADVERSARİAL ANALİZİ
# ══════════════════════════════════════════════════════════════════════
import os, pickle, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap, lime, lime.lime_tabular
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DNN_DIR = str(DNN_LSTM_DIR)
EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

def kaydet(nesne, yol):
    with open(yol, 'wb') as f: pickle.dump(nesne, f)
    print(f"  💾 {os.path.basename(yol)}")

def jaccard_top_k(v1, v2, k):
    t1=set(np.argsort(np.abs(v1))[-k:]); t2=set(np.argsort(np.abs(v2))[-k:])
    return len(t1&t2)/len(t1|t2) if len(t1|t2)>0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec=np.zeros(len(feature_names))
    for fname,fval in lime_dict.items():
        clean=fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i,fn in enumerate(feature_names):
            if fn==clean: vec[i]=fval; break
    return vec

# ── Model mimarileri ──────────────────────────────────────────────────
class IDS_DNN(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(64,num_classes))
    def forward(self, x): return self.network(x)

class IDS_LSTM(nn.Module):
    def __init__(self, seq_len, num_classes, hidden=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm=nn.LSTM(input_size=1,hidden_size=hidden,num_layers=n_layers,
                          batch_first=True,dropout=dropout if n_layers>1 else 0)
        self.classifier=nn.Sequential(
            nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,num_classes))
    def forward(self, x):
        x=x.unsqueeze(-1); out,(hn,_)=self.lstm(x); return self.classifier(hn[-1])

# ── Verileri yükle ────────────────────────────────────────────────────
veri_edge=yukle(os.path.join(EDGE_DIR,"veri.pkl"))
X_edge=veri_edge['X_multi_scaled']; y_edge=veri_edge['y_multi']
fn_edge=veri_edge['feature_names_multi']; cn_edge=list(veri_edge['sinif_isimleri'])
if hasattr(X_edge,'values'): X_edge=X_edge.values
if hasattr(y_edge,'values'): y_edge=y_edge.values

veri_cic=yukle(os.path.join(CIC_DIR,"veri_cicids.pkl"))
X_cic=veri_cic['X_scaled']; y_cic=veri_cic['y']
fn_cic=veri_cic['feature_names']; cn_cic=list(veri_cic['sinif_isimleri'])
if hasattr(X_cic,'values'): X_cic=X_cic.values
if hasattr(y_cic,'values'): y_cic=y_cic.values

print("✅ Setup tamamlandı")
print(f"   Edge: {X_edge.shape} | CIC: {X_cic.shape} | GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FGSM FONKSİYONLARI + EDGE-IIoT ANALİZİ
# ══════════════════════════════════════════════════════════════════════
from sklearn.metrics import f1_score

EPSILONLAR  = [0.01, 0.05, 0.10]
N_FGSM      = 200   # Her model için 200 örnek
CRITERION   = nn.CrossEntropyLoss()

# ── FGSM: Sinir ağları için (DNN / LSTM) ─────────────────────────────
def fgsm_nn(model, x_np, y_true, epsilon, is_lstm=False):
    """Tek örnek için FGSM adversarial örnek üret."""
    model.eval()
    x_t = torch.FloatTensor(x_np).unsqueeze(0).to(DEVICE).requires_grad_(True)
    y_t = torch.LongTensor([y_true]).to(DEVICE)
    if is_lstm:
        torch.backends.cudnn.enabled = False
        model.train()
    loss = CRITERION(model(x_t), y_t)
    model.zero_grad(); loss.backward()
    x_adv = (x_t + epsilon * x_t.grad.sign()).detach().cpu().numpy()[0]
    if is_lstm:
        torch.backends.cudnn.enabled = True; model.eval()
    return x_adv

# ── FGSM: Ağaç modelleri için (sayısal gradyan) ───────────────────────
def fgsm_tree(predict_proba_fn, x_np, y_true, epsilon, h=1e-3):
    """Finite differences ile FGSM."""
    grad = np.zeros_like(x_np, dtype=float)
    p0   = predict_proba_fn(x_np.reshape(1,-1))[0]
    for i in range(len(x_np)):
        xp = x_np.copy(); xp[i] += h
        pp = predict_proba_fn(xp.reshape(1,-1))[0]
        grad[i] = (pp[y_true] - p0[y_true]) / h
    return x_np + epsilon * np.sign(grad)

# ── FGSM Analiz Döngüsü ───────────────────────────────────────────────
def fgsm_analiz(model_adi, model_obj, predict_proba_fn, shap_orig, lime_orig,
                X_orneklem, y_orneklem, feature_names, lime_explainer,
                shap_explainer, epsilonlar, n_fgsm, is_lstm=False, is_nn=False):
    sonuclar = []
    np.random.seed(42)
    idx = np.random.choice(len(X_orneklem), size=n_fgsm, replace=False)

    for eps in epsilonlar:
        sp_shap, sp_lime = [], []
        j5_shap, j5_lime = [], []
        atk_basari       = []

        for i in idx:
            x_orig = X_orneklem[i]
            y_true = int(y_orneklem[i])

            # Adversarial örnek üret
            if is_nn:
                x_adv = fgsm_nn(model_obj, x_orig, y_true, eps, is_lstm=is_lstm)
            else:
                x_adv = fgsm_tree(predict_proba_fn, x_orig, y_true, eps)

            # Saldırı başarısı
            pred_orig = np.argmax(predict_proba_fn(x_orig.reshape(1,-1))[0])
            pred_adv  = np.argmax(predict_proba_fn(x_adv.reshape(1,-1))[0])
            atk_basari.append(int(pred_orig != pred_adv))

            # SHAP adversarial
            if is_nn:
                if is_lstm:
                    torch.backends.cudnn.enabled = False; model_obj.train()
                sv_adv = np.array(shap_explainer.shap_values(
                    torch.FloatTensor(x_adv.reshape(1,-1)).to(DEVICE)))
                if is_lstm:
                    torch.backends.cudnn.enabled = True; model_obj.eval()
                sv_adv = np.mean(np.abs(sv_adv[0]),axis=1) if sv_adv.ndim==3 else np.abs(sv_adv[0])
            else:
                sv_adv = np.array(shap_explainer.shap_values(x_adv.reshape(1,-1)))
                if isinstance(sv_adv, list):
                    sv_adv = np.mean([np.abs(s[0]) for s in sv_adv], axis=0)
                elif sv_adv.ndim == 3:
                    sv_adv = np.mean(np.abs(sv_adv[:,0,:]), axis=0)
                else:
                    sv_adv = np.abs(sv_adv[0])

            sp,_ = spearmanr(shap_orig[i], sv_adv)
            sp_shap.append(sp)
            j5_shap.append(jaccard_top_k(shap_orig[i], sv_adv, 5))

            # LIME adversarial
            exp_adv = lime_explainer.explain_instance(
                x_adv, predict_proba_fn, num_features=len(feature_names), num_samples=500)
            lv_adv  = lime_to_vec(dict(exp_adv.as_list()), feature_names)
            sl,_ = spearmanr(lime_orig[i], lv_adv)
            sp_lime.append(sl)
            j5_lime.append(jaccard_top_k(lime_orig[i], lv_adv, 5))

        sonuclar.append({
            'model'         : model_adi,
            'epsilon'       : eps,
            'atk_basari_pct': round(np.mean(atk_basari)*100, 1),
            'shap_spearman' : round(float(np.nanmean(sp_shap)), 4),
            'shap_jaccard5' : round(float(np.nanmean(j5_shap)), 4),
            'lime_spearman' : round(float(np.nanmean(sp_lime)), 4),
            'lime_jaccard5' : round(float(np.nanmean(j5_lime)), 4),
        })
        print(f"  [{model_adi}] ε={eps:.2f} → Atk%={np.mean(atk_basari)*100:.1f}%  "
              f"SHAP={np.nanmean(sp_shap):.4f}  LIME={np.nanmean(sp_lime):.4f}")
    return sonuclar

# ══════════════════════════════════════════════════════════════════════
# EDGE-IIoT — TÜM MODELLER
# ══════════════════════════════════════════════════════════════════════
np.random.seed(42)
orn_idx_fe = np.random.choice(len(X_edge), size=500, replace=False)
X_orn_fe   = X_edge[orn_idx_fe]; y_orn_fe = y_edge[orn_idx_fe]

np.random.seed(0)
bg_idx_fe  = np.random.choice(len(X_edge), size=100, replace=False)
X_bg_fe    = torch.FloatTensor(X_edge[bg_idx_fe]).to(DEVICE)

lime_exp_fe = lime.lime_tabular.LimeTabularExplainer(
    X_edge, feature_names=fn_edge, class_names=cn_edge,
    discretize_continuous=True, random_state=42)

FGSM_EDGE_PATH = os.path.join(DNN_DIR, "fgsm_edge_sonuclar.pkl")
fgsm_edge_sonuclar = yukle(FGSM_EDGE_PATH) if os.path.exists(FGSM_EDGE_PATH) else []
tamamlanan_fgsm_edge = {(r['model'], r['epsilon']) for r in fgsm_edge_sonuclar}

# ── 1. DNN Edge ───────────────────────────────────────────────────────
print("\n=== DNN Edge-IIoT ===")
dnn_edge_sonuclar = yukle(os.path.join(DNN_DIR,"dnn_edge_modeller.pkl"))
xai_dnn_edge      = yukle(os.path.join(DNN_DIR,"xai_dnn_edge.pkl"))
ref_dnn_e = next(r['model_obj'] for r in dnn_edge_sonuclar if r['seed']==0 and r['fold']==0)
ref_dnn_e = ref_dnn_e.to(DEVICE); ref_dnn_e.eval()

def dnn_e_proba(X_np):
    ref_dnn_e.eval()
    with torch.no_grad():
        return torch.softmax(ref_dnn_e(torch.FloatTensor(X_np).to(DEVICE)),dim=1).cpu().numpy()

exp_dnn_e = shap.GradientExplainer(ref_dnn_e, X_bg_fe)

eps_todo = [e for e in EPSILONLAR if ('DNN',e) not in tamamlanan_fgsm_edge]
if eps_todo:
    res = fgsm_analiz('DNN', ref_dnn_e, dnn_e_proba,
                      xai_dnn_edge['shap_values'], xai_dnn_edge['lime_values'],
                      X_orn_fe, y_orn_fe, fn_edge, lime_exp_fe, exp_dnn_e,
                      eps_todo, N_FGSM, is_nn=True, is_lstm=False)
    fgsm_edge_sonuclar.extend(res)
    kaydet(fgsm_edge_sonuclar, FGSM_EDGE_PATH)
ref_dnn_e.cpu(); del ref_dnn_e, dnn_edge_sonuclar; import gc; gc.collect()

# ── 2. LSTM Edge ──────────────────────────────────────────────────────
print("\n=== LSTM Edge-IIoT ===")
lstm_edge_sonuclar = yukle(os.path.join(DNN_DIR,"lstm_edge_modeller.pkl"))
xai_lstm_edge      = yukle(os.path.join(DNN_DIR,"xai_lstm_edge.pkl"))
ref_lstm_e = next(r['model_obj'] for r in lstm_edge_sonuclar if r['seed']==0 and r['fold']==0)
ref_lstm_e = ref_lstm_e.to(DEVICE)

def lstm_e_proba(X_np):
    ref_lstm_e.eval()
    with torch.no_grad():
        return torch.softmax(ref_lstm_e(torch.FloatTensor(X_np).to(DEVICE)),dim=1).cpu().numpy()

torch.backends.cudnn.enabled=False; ref_lstm_e.train()
exp_lstm_e = shap.GradientExplainer(ref_lstm_e, X_bg_fe)
torch.backends.cudnn.enabled=True; ref_lstm_e.eval()

eps_todo = [e for e in EPSILONLAR if ('LSTM',e) not in tamamlanan_fgsm_edge]
if eps_todo:
    res = fgsm_analiz('LSTM', ref_lstm_e, lstm_e_proba,
                      xai_lstm_edge['shap_values'], xai_lstm_edge['lime_values'],
                      X_orn_fe, y_orn_fe, fn_edge, lime_exp_fe, exp_lstm_e,
                      eps_todo, N_FGSM, is_nn=True, is_lstm=True)
    fgsm_edge_sonuclar.extend(res)
    kaydet(fgsm_edge_sonuclar, FGSM_EDGE_PATH)
ref_lstm_e.cpu(); del ref_lstm_e, lstm_edge_sonuclar; gc.collect()

# ── 3. Ağaç Modelleri Edge ────────────────────────────────────────────
print("\n=== Ağaç Modelleri Edge-IIoT ===")
MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

seed0_edge = yukle(os.path.join(EDGE_DIR,"modeller_seed0.pkl"))

for model_adi in MODEL_SIRASI:
    eps_todo = [e for e in EPSILONLAR if (model_adi,e) not in tamamlanan_fgsm_edge]
    if not eps_todo:
        print(f"  ♻  {model_adi} — atlandı"); continue

    # Modeli yükle
    model_obj = next(r['model_obj'] for r in seed0_edge
                     if r['model_adi']==model_adi and r['fold']==0)
    xai_obj   = yukle(os.path.join(EDGE_DIR, f"xai_{model_adi}.pkl"))

    # SHAP explainer (TreeExplainer)
    if model_adi == 'LogisticRegression':
        shap_exp = shap.LinearExplainer(model_obj, X_edge[bg_idx_fe])
    elif model_adi == 'CatBoost':
        from catboost import Pool
        shap_exp = None  # CatBoost için özel
    else:
        shap_exp = shap.TreeExplainer(model_obj)

    def make_proba(m):
        def fn(X): return m.predict_proba(X)
        return fn
    proba_fn = make_proba(model_obj)

    # CatBoost için özel SHAP
    if model_adi == 'CatBoost':
        def catboost_shap_adv(x):
            from catboost import Pool
            sv = model_obj.get_feature_importance(Pool(x.reshape(1,-1)), type='ShapValues')
            sv = np.array(sv)
            if sv.ndim==3: return np.mean(np.abs(sv[0,:,:-1]),axis=0)
            elif sv.ndim==2: return np.abs(sv[0,:-1])
        # Geçici explainer objesi
        class CatBoostExp:
            def shap_values(self, x):
                return catboost_shap_adv(x.reshape(-1) if hasattr(x,'reshape') else x)
        shap_exp_obj = CatBoostExp()
    else:
        shap_exp_obj = shap_exp

    print(f"\n  {model_adi}...")
    res = fgsm_analiz(model_adi, model_obj, proba_fn,
                      xai_obj['shap_values'], xai_obj['lime_values'],
                      X_orn_fe, y_orn_fe, fn_edge, lime_exp_fe, shap_exp_obj,
                      eps_todo, N_FGSM, is_nn=False, is_lstm=False)
    fgsm_edge_sonuclar.extend(res)
    tamamlanan_fgsm_edge = {(r['model'],r['epsilon']) for r in fgsm_edge_sonuclar}
    kaydet(fgsm_edge_sonuclar, FGSM_EDGE_PATH)
    del model_obj, xai_obj; gc.collect()

del seed0_edge; gc.collect()

# ── Özet ─────────────────────────────────────────────────────────────
df_fgsm_edge = pd.DataFrame(fgsm_edge_sonuclar)
kaydet(df_fgsm_edge, os.path.join(DNN_DIR,"df_fgsm_edge.pkl"))
print("\n📊 FGSM Edge-IIoT Özet:")
print(df_fgsm_edge.pivot_table(
    index='model', columns='epsilon',
    values=['atk_basari_pct','shap_spearman','lime_spearman']).to_string())
print("\n✅ FGSM Edge-IIoT tamamlandı!")

In [ ]:
def extract_shap_tree(sv_raw, model_adi):
    """TreeExplainer çıktısını (74,) vektörüne çevir."""
    arr = np.array(sv_raw)
    if isinstance(sv_raw, list):
        # liste: [n_sinif x (1, 74)]
        return np.mean([np.abs(s[0]) for s in sv_raw], axis=0)
    elif arr.ndim == 3 and arr.shape[-1] > arr.shape[0]:
        # (1, 74, 15) → mean over classes
        return np.mean(np.abs(arr[0]), axis=1)
    elif arr.ndim == 3:
        # (15, 1, 74) → mean over classes
        return np.mean(np.abs(arr[:, 0, :]), axis=0)
    else:
        return np.abs(arr[0])

def fgsm_analiz_tree(model_adi, model_obj, predict_proba_fn, shap_orig, lime_orig,
                     X_orneklem, y_orneklem, feature_names, lime_explainer,
                     shap_explainer, epsilonlar, n_fgsm):
    sonuclar = []
    np.random.seed(42)
    idx = np.random.choice(len(X_orneklem), size=n_fgsm, replace=False)

    for eps in epsilonlar:
        sp_shap,sp_lime,j5_shap,j5_lime,atk_basari=[],[],[],[],[]

        for i in idx:
            x_orig = X_orneklem[i]; y_true = int(y_orneklem[i])
            x_adv  = fgsm_tree(predict_proba_fn, x_orig, y_true, eps)

            pred_orig = np.argmax(predict_proba_fn(x_orig.reshape(1,-1))[0])
            pred_adv  = np.argmax(predict_proba_fn(x_adv.reshape(1,-1))[0])
            atk_basari.append(int(pred_orig != pred_adv))

            # SHAP adversarial
            if model_adi == 'CatBoost':
                sv_adv = shap_explainer.shap_values(x_adv)
            else:
                sv_raw = shap_explainer.shap_values(x_adv.reshape(1,-1))
                sv_adv = extract_shap_tree(sv_raw, model_adi)

            sp,_ = spearmanr(shap_orig[i], sv_adv)
            sp_shap.append(sp); j5_shap.append(jaccard_top_k(shap_orig[i],sv_adv,5))

            # LIME adversarial
            exp_adv = lime_explainer.explain_instance(
                x_adv, predict_proba_fn, num_features=len(feature_names), num_samples=500)
            lv_adv = lime_to_vec(dict(exp_adv.as_list()), feature_names)
            sl,_ = spearmanr(lime_orig[i], lv_adv)
            sp_lime.append(sl); j5_lime.append(jaccard_top_k(lime_orig[i],lv_adv,5))

        sonuclar.append({
            'model'         : model_adi,
            'epsilon'       : eps,
            'atk_basari_pct': round(np.mean(atk_basari)*100, 1),
            'shap_spearman' : round(float(np.nanmean(sp_shap)), 4),
            'shap_jaccard5' : round(float(np.nanmean(j5_shap)), 4),
            'lime_spearman' : round(float(np.nanmean(sp_lime)), 4),
            'lime_jaccard5' : round(float(np.nanmean(j5_lime)), 4),
        })
        print(f"  [{model_adi}] ε={eps:.2f} → Atk%={np.mean(atk_basari)*100:.1f}%  "
              f"SHAP={np.nanmean(sp_shap):.4f}  LIME={np.nanmean(sp_lime):.4f}")
    return sonuclar

# ── Ağaç modelleri — yeniden başlat ──────────────────────────────────
fgsm_edge_sonuclar = yukle(FGSM_EDGE_PATH)
tamamlanan_fgsm_edge = {(r['model'],r['epsilon']) for r in fgsm_edge_sonuclar}

for model_adi in MODEL_SIRASI:
    eps_todo = [e for e in EPSILONLAR if (model_adi,e) not in tamamlanan_fgsm_edge]
    if not eps_todo:
        print(f"  ♻  {model_adi} — atlandı"); continue

    model_obj = next(r['model_obj'] for r in seed0_edge
                     if r['model_adi']==model_adi and r['fold']==0)
    xai_obj   = yukle(os.path.join(EDGE_DIR, f"xai_{model_adi}.pkl"))

    if model_adi == 'LogisticRegression':
        shap_exp = shap.LinearExplainer(model_obj, X_edge[bg_idx_fe])
    elif model_adi == 'CatBoost':
        class CatBoostExp:
            def __init__(self, m): self.m = m
            def shap_values(self, x):
                from catboost import Pool
                sv = self.m.get_feature_importance(Pool(x.reshape(1,-1)), type='ShapValues')
                sv = np.array(sv)
                if sv.ndim==3: return np.mean(np.abs(sv[0,:,:-1]), axis=0)
                return np.abs(sv[0,:-1])
        shap_exp = CatBoostExp(model_obj)
    else:
        shap_exp = shap.TreeExplainer(model_obj)

    def make_proba(m):
        def fn(X): return m.predict_proba(X)
        return fn

    print(f"\n  {model_adi}...")
    res = fgsm_analiz_tree(
        model_adi, model_obj, make_proba(model_obj),
        xai_obj['shap_values'], xai_obj['lime_values'],
        X_orn_fe, y_orn_fe, fn_edge, lime_exp_fe, shap_exp,
        eps_todo, N_FGSM)
    fgsm_edge_sonuclar.extend(res)
    tamamlanan_fgsm_edge = {(r['model'],r['epsilon']) for r in fgsm_edge_sonuclar}
    kaydet(fgsm_edge_sonuclar, FGSM_EDGE_PATH)
    del model_obj, xai_obj; gc.collect()

df_fgsm_edge = pd.DataFrame(fgsm_edge_sonuclar)
kaydet(df_fgsm_edge, os.path.join(DNN_DIR,"df_fgsm_edge.pkl"))
print("\n📊 FGSM Edge-IIoT Özet:")
print(df_fgsm_edge.pivot_table(
    index='model', columns='epsilon',
    values=['atk_basari_pct','shap_spearman','lime_spearman']).to_string())
print("\n✅ FGSM Edge-IIoT tamamlandı!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FGSM — CIC-IDS-2017
# ══════════════════════════════════════════════════════════════════════
np.random.seed(42)
orn_idx_fc = np.random.choice(len(X_cic), size=500, replace=False)
X_orn_fc   = X_cic[orn_idx_fc]; y_orn_fc = y_cic[orn_idx_fc]

np.random.seed(0)
bg_idx_fc  = np.random.choice(len(X_cic), size=100, replace=False)
X_bg_fc    = torch.FloatTensor(X_cic[bg_idx_fc]).to(DEVICE)

lime_exp_fc = lime.lime_tabular.LimeTabularExplainer(
    X_cic, feature_names=fn_cic, class_names=cn_cic,
    discretize_continuous=True, random_state=42)

FGSM_CIC_PATH = os.path.join(DNN_DIR, "fgsm_cic_sonuclar.pkl")
fgsm_cic_sonuclar = yukle(FGSM_CIC_PATH) if os.path.exists(FGSM_CIC_PATH) else []
tamamlanan_fgsm_cic = {(r['model'],r['epsilon']) for r in fgsm_cic_sonuclar}

# ── 1. DNN CIC ────────────────────────────────────────────────────────
print("=== DNN CIC-IDS-2017 ===")
dnn_cic_sonuclar = yukle(os.path.join(DNN_DIR,"dnn_cic_modeller.pkl"))
xai_dnn_cic      = yukle(os.path.join(DNN_DIR,"xai_dnn_cic.pkl"))
ref_dnn_c = next(r['model_obj'] for r in dnn_cic_sonuclar if r['seed']==0 and r['fold']==0)
ref_dnn_c = ref_dnn_c.to(DEVICE); ref_dnn_c.eval()

def dnn_c_proba(X_np):
    ref_dnn_c.eval()
    with torch.no_grad():
        return torch.softmax(ref_dnn_c(torch.FloatTensor(X_np).to(DEVICE)),dim=1).cpu().numpy()

exp_dnn_c = shap.GradientExplainer(ref_dnn_c, X_bg_fc)

eps_todo = [e for e in EPSILONLAR if ('DNN',e) not in tamamlanan_fgsm_cic]
if eps_todo:
    res = fgsm_analiz('DNN', ref_dnn_c, dnn_c_proba,
                      xai_dnn_cic['shap_values'], xai_dnn_cic['lime_values'],
                      X_orn_fc, y_orn_fc, fn_cic, lime_exp_fc, exp_dnn_c,
                      eps_todo, N_FGSM, is_nn=True, is_lstm=False)
    fgsm_cic_sonuclar.extend(res)
    kaydet(fgsm_cic_sonuclar, FGSM_CIC_PATH)
ref_dnn_c.cpu(); del ref_dnn_c, dnn_cic_sonuclar; gc.collect()

# ── 2. LSTM CIC ───────────────────────────────────────────────────────
print("\n=== LSTM CIC-IDS-2017 ===")
lstm_cic_sonuclar = yukle(os.path.join(DNN_DIR,"lstm_cic_modeller.pkl"))
xai_lstm_cic      = yukle(os.path.join(DNN_DIR,"xai_lstm_cic.pkl"))
ref_lstm_c = next(r['model_obj'] for r in lstm_cic_sonuclar if r['seed']==0 and r['fold']==0)
ref_lstm_c = ref_lstm_c.to(DEVICE)

def lstm_c_proba(X_np):
    ref_lstm_c.eval()
    with torch.no_grad():
        return torch.softmax(ref_lstm_c(torch.FloatTensor(X_np).to(DEVICE)),dim=1).cpu().numpy()

torch.backends.cudnn.enabled=False; ref_lstm_c.train()
exp_lstm_c = shap.GradientExplainer(ref_lstm_c, X_bg_fc)
torch.backends.cudnn.enabled=True; ref_lstm_c.eval()

eps_todo = [e for e in EPSILONLAR if ('LSTM',e) not in tamamlanan_fgsm_cic]
if eps_todo:
    res = fgsm_analiz('LSTM', ref_lstm_c, lstm_c_proba,
                      xai_lstm_cic['shap_values'], xai_lstm_cic['lime_values'],
                      X_orn_fc, y_orn_fc, fn_cic, lime_exp_fc, exp_lstm_c,
                      eps_todo, N_FGSM, is_nn=True, is_lstm=True)
    fgsm_cic_sonuclar.extend(res)
    kaydet(fgsm_cic_sonuclar, FGSM_CIC_PATH)
ref_lstm_c.cpu(); del ref_lstm_c, lstm_cic_sonuclar; gc.collect()

# ── 3. Ağaç Modelleri CIC ─────────────────────────────────────────────
print("\n=== Ağaç Modelleri CIC-IDS-2017 ===")
tamamlanan_fgsm_cic = {(r['model'],r['epsilon']) for r in fgsm_cic_sonuclar}

seed0_cic = yukle(os.path.join(CIC_DIR,"modeller_cicids_seed0.pkl"))

for model_adi in MODEL_SIRASI:
    eps_todo = [e for e in EPSILONLAR if (model_adi,e) not in tamamlanan_fgsm_cic]
    if not eps_todo:
        print(f"  ♻  {model_adi} — atlandı"); continue

    model_obj = next(r['model_obj'] for r in seed0_cic
                     if r['model_adi']==model_adi and r['fold']==0)
    xai_obj   = yukle(os.path.join(CIC_DIR, f"xai_cicids_{model_adi}.pkl"))

    if model_adi == 'LogisticRegression':
        shap_exp = shap.LinearExplainer(model_obj, X_cic[bg_idx_fc])
    elif model_adi == 'CatBoost':
        class CatBoostExpCIC:
            def __init__(self, m): self.m = m
            def shap_values(self, x):
                from catboost import Pool
                sv = self.m.get_feature_importance(Pool(x.reshape(1,-1)), type='ShapValues')
                sv = np.array(sv)
                if sv.ndim==3: return np.mean(np.abs(sv[0,:,:-1]), axis=0)
                return np.abs(sv[0,:-1])
        shap_exp = CatBoostExpCIC(model_obj)
    else:
        shap_exp = shap.TreeExplainer(model_obj)

    def make_proba(m):
        def fn(X): return m.predict_proba(X)
        return fn

    print(f"\n  {model_adi}...")
    res = fgsm_analiz_tree(
        model_adi, model_obj, make_proba(model_obj),
        xai_obj['shap_values'], xai_obj['lime_values'],
        X_orn_fc, y_orn_fc, fn_cic, lime_exp_fc, shap_exp,
        eps_todo, N_FGSM)
    fgsm_cic_sonuclar.extend(res)
    tamamlanan_fgsm_cic = {(r['model'],r['epsilon']) for r in fgsm_cic_sonuclar}
    kaydet(fgsm_cic_sonuclar, FGSM_CIC_PATH)
    del model_obj, xai_obj; gc.collect()

del seed0_cic; gc.collect()

df_fgsm_cic = pd.DataFrame(fgsm_cic_sonuclar)
kaydet(df_fgsm_cic, os.path.join(DNN_DIR,"df_fgsm_cic.pkl"))
print("\n📊 FGSM CIC-IDS-2017 Özet:")
print(df_fgsm_cic.pivot_table(
    index='model', columns='epsilon',
    values=['atk_basari_pct','shap_spearman','lime_spearman']).to_string())
print("\n✅ FGSM CIC-IDS-2017 tamamlandı!")

In [ ]:
# ── FGSM Görsel ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

GORSEL_DIR = str(CICIDS2017_DIR)
RENKLER = {
    'RandomForest':'#2196F3','ExtraTrees':'#4CAF50','DecisionTree':'#FF9800',
    'XGBoost':'#F44336','LightGBM':'#9C27B0','CatBoost':'#00BCD4',
    'LogisticRegression':'#795548','DNN':'#E91E63','LSTM':'#FF6F00'
}
KISALTMA = {
    'RandomForest':'RF','ExtraTrees':'ET','DecisionTree':'DT','XGBoost':'XGB',
    'LightGBM':'LGBM','CatBoost':'CB','LogisticRegression':'LR','DNN':'DNN','LSTM':'LSTM'
}
MODEL_SIRASI9 = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
                 'LightGBM','CatBoost','LogisticRegression','DNN','LSTM']

df_edge = yukle(os.path.join(DNN_DIR,"df_fgsm_edge.pkl"))
df_cic  = yukle(os.path.join(DNN_DIR,"df_fgsm_cic.pkl"))

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('FGSM Adversarial Analizi — SHAP ve LIME Kararlılığı\nEdge-IIoT vs CIC-IDS-2017',
             fontsize=14, fontweight='bold')

EPSILONLAR = [0.01, 0.05, 0.10]
renk_listesi = [RENKLER[m] for m in MODEL_SIRASI9]
kisalt_listesi = [KISALTMA[m] for m in MODEL_SIRASI9]
x = np.arange(len(MODEL_SIRASI9)); bar_w = 0.28

for col, eps in enumerate(EPSILONLAR):
    for row, (df, dataset) in enumerate([(df_edge,'Edge-IIoT'), (df_cic,'CIC-IDS-2017')]):
        ax = axes[row][col]
        df_eps = df[df['epsilon']==eps].set_index('model')

        shap_vals = [df_eps.loc[m,'shap_spearman'] if m in df_eps.index else 0 for m in MODEL_SIRASI9]
        lime_vals = [df_eps.loc[m,'lime_spearman'] if m in df_eps.index else 0 for m in MODEL_SIRASI9]
        atk_vals  = [df_eps.loc[m,'atk_basari_pct'] if m in df_eps.index else 0 for m in MODEL_SIRASI9]

        b1 = ax.bar(x - bar_w/2, shap_vals, bar_w, color=renk_listesi,
                    alpha=1.0, edgecolor='white', linewidth=0.8, label='SHAP')
        b2 = ax.bar(x + bar_w/2, lime_vals, bar_w, color=renk_listesi,
                    alpha=0.4, edgecolor='grey', linewidth=0.8, hatch='//', label='LIME')

        # Saldırı başarısını üst kısımda göster
        for xi, atk in zip(x, atk_vals):
            ax.text(xi, 1.02, f'%{atk:.0f}', ha='center', va='bottom',
                    fontsize=6.5, color='red', fontweight='bold')

        ax.set_xticks(x); ax.set_xticklabels(kisalt_listesi, fontsize=8)
        ax.set_ylim(-0.15, 1.12)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='-')
        ax.axhline(0.9, color='green', linewidth=0.8, linestyle='--', alpha=0.5)
        ax.set_title(f'{dataset} — ε={eps}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Spearman r', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.spines[['top','right']].set_visible(False)

        if row==0 and col==0:
            leg = [mpatches.Patch(facecolor='grey',alpha=1.0,label='SHAP (düz)'),
                   mpatches.Patch(facecolor='grey',alpha=0.4,hatch='//',label='LIME (taralı)'),
                   plt.Line2D([0],[0],color='red',label='Kırmızı % = Saldırı Başarısı')]
            ax.legend(handles=leg, fontsize=7, loc='lower right')

plt.tight_layout()
kayit = os.path.join(GORSEL_DIR,'fgsm_adversarial_analiz.png')
plt.savefig(kayit, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {kayit}")

# ── Özet tablo ────────────────────────────────────────────────────────
print("\n📊 FGSM ε=0.10 Karşılaştırma:")
print(f"{'Model':<20} {'Edge Atk%':>9} {'Edge SHAP':>10} {'CIC Atk%':>9} {'CIC SHAP':>9}")
print("-"*60)
for m in MODEL_SIRASI9:
    e_row = df_edge[(df_edge['model']==m)&(df_edge['epsilon']==0.10)]
    c_row = df_cic[(df_cic['model']==m)&(df_cic['epsilon']==0.10)]
    if len(e_row) and len(c_row):
        print(f"{KISALTMA[m]:<20} {e_row['atk_basari_pct'].values[0]:>9.1f} "
              f"{e_row['shap_spearman'].values[0]:>10.4f} "
              f"{c_row['atk_basari_pct'].values[0]:>9.1f} "
              f"{c_row['shap_spearman'].values[0]:>9.4f}")

In [ ]:
import pickle, os
import pandas as pd
import numpy as np

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

# Klasik modeller LIME Fidelity@5
df_fid_edge = yukle(os.path.join(EDGE_DIR, "df_fidelity.pkl"))
df_fid_cic  = yukle(os.path.join(CIC_DIR,  "df_fidelity_cicids.pkl"))

print("=== Klasik Modeller LIME Fidelity@5 (Edge) ===")
lime5_edge = df_fid_edge[(df_fid_edge['xai']=='LIME') & (df_fid_edge['n_maskele']==5)]
print(lime5_edge.groupby('model')['delta_f1'].mean().round(4))

print("\n=== Klasik Modeller LIME Fidelity@5 (CIC) ===")
lime5_cic = df_fid_cic[(df_fid_cic['xai']=='LIME') & (df_fid_cic['n_maskele']==5)]
print(lime5_cic.groupby('model')['delta_f1'].mean().round(4))

# LSTM Edge Jaccard@10
rq2_lstm_edge = yukle(os.path.join(DNN_DIR, "rq2_lstm_edge.pkl"))
from scipy.stats import spearmanr
from itertools import combinations

def jaccard_top_k(v1, v2, k):
    t1=set(np.argsort(np.abs(v1))[-k:]); t2=set(np.argsort(np.abs(v2))[-k:])
    return len(t1&t2)/len(t1|t2) if len(t1|t2)>0 else 0.0

keys = list(rq2_lstm_edge.keys())
j10_list = []
for k1,k2 in combinations(keys,2):
    for i in range(100):
        j10_list.append(jaccard_top_k(rq2_lstm_edge[k1][i], rq2_lstm_edge[k2][i], 10))

print(f"\n=== LSTM Edge Jaccard@10 ===")
print(f"  {np.nanmean(j10_list):.4f}")

In [ ]:
import pickle, os
import numpy as np

DNN_DIR = str(DNN_LSTM_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

# DNN ve LSTM Jaccard@10 değerleri
dosyalar = {
    'DNN Edge' : 'rq2_dnn_edge_ozet.pkl',
    'DNN CIC'  : 'rq2_dnn_cic_ozet.pkl',
    'LSTM Edge': 'rq2_lstm_edge_ozet.pkl',
    'LSTM CIC' : 'rq2_lstm_cic_ozet.pkl',
}

print("=== RQ2 Jaccard@10 Değerleri ===")
for isim, dosya in dosyalar.items():
    d = yukle(os.path.join(DNN_DIR, dosya))
    print(f"  {isim:<12} Spearman={d['spearman']:.4f}  J@5={d['jaccard_5']:.4f}  J@10={d.get('jaccard_10', 'YOK')}")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

# CIC Jaccard@10 klasik modeller
df_var_cic = yukle(os.path.join(CIC_DIR, "df_var_cicids.pkl"))
print("=== CIC J@10 Klasik Modeller ===")
print(df_var_cic.groupby('model_adi')['jaccard_10'].mean().round(4))

# RQ3 Spearman r genel (Edge)
df_rq3_edge = yukle(os.path.join(EDGE_DIR, "df_rq3.pkl"))
print(f"\n=== df_rq3_edge sütunları ===")
print(df_rq3_edge.columns.tolist())
print(df_rq3_edge.head(3))

# RQ3 Spearman r genel (CIC)
df_rq3_cic = yukle(os.path.join(CIC_DIR, "df_rq3_cicids.pkl"))
print(f"\n=== df_rq3_cic sütunları ===")
print(df_rq3_cic.columns.tolist())
print(df_rq3_cic.head(3))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

# ── Edge-IIoT RQ3 (df_var'dan hesapla) ───────────────────────────────
df_var_edge = yukle(os.path.join(EDGE_DIR, "df_var.pkl"))
print("df_var_edge sütunları:", df_var_edge.columns.tolist())

# Her seed/fold için F1 ve SHAP tutarlılığı
f1_edge   = df_var_edge['f1_macro'].values
shap_edge = df_var_edge['spearman'].values

rp_e, pp_e = pearsonr(f1_edge, shap_edge)
rs_e, ps_e = spearmanr(f1_edge, shap_edge)
print(f"\n=== Edge-IIoT RQ3 (7 klasik model genel) ===")
print(f"  Pearson  r={rp_e:.4f}  p={pp_e:.4f}")
print(f"  Spearman r={rs_e:.4f}  p={ps_e:.4f}")
print(f"  N={len(f1_edge)}")

# ── CIC-IDS-2017 RQ3 ──────────────────────────────────────────────────
df_rq3_cic = yukle(os.path.join(CIC_DIR, "df_rq3_cicids.pkl"))
print(f"\ndf_rq3_cic sütunları: {df_rq3_cic.columns.tolist()}")
print(df_rq3_cic.head(3))

f1_cic   = df_rq3_cic['f1_macro'].values if 'f1_macro' in df_rq3_cic.columns else None
shap_cic = df_rq3_cic['spearman'].values  if 'spearman'  in df_rq3_cic.columns else None

if f1_cic is not None and shap_cic is not None:
    rp_c, pp_c = pearsonr(f1_cic, shap_cic)
    rs_c, ps_c = spearmanr(f1_cic, shap_cic)
    print(f"\n=== CIC-IDS-2017 RQ3 (7 klasik model genel) ===")
    print(f"  Pearson  r={rp_c:.4f}  p={pp_c:.4f}")
    print(f"  Spearman r={rs_c:.4f}  p={ps_c:.4f}")
    print(f"  N={len(f1_cic)}")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def yukle(yol):
    with open(yol, 'rb') as f: return pickle.load(f)

def kaydet(nesne, yol):
    with open(yol, 'wb') as f: pickle.dump(nesne, f)
    print(f"  💾 {os.path.basename(yol)}")

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ── EDGE-IIoT ─────────────────────────────────────────────────────────
print("=== Edge-IIoT LIME Fidelity ===")
veri_edge = yukle(os.path.join(EDGE_DIR, "veri.pkl"))
X_edge    = veri_edge['X_multi_scaled']
y_edge    = veri_edge['y_multi']
fn_edge   = veri_edge['feature_names_multi']
if hasattr(X_edge,'values'): X_edge=X_edge.values
if hasattr(y_edge,'values'): y_edge=y_edge.values

seed0_edge = yukle(os.path.join(EDGE_DIR, "modeller_seed0.pkl"))
ref_edge   = {r['model_adi']:r['model_obj'] for r in seed0_edge if r['fold']==0}

np.random.seed(42)
orn_idx  = np.random.choice(len(X_edge), size=500, replace=False)
X_orn_e  = X_edge[orn_idx]; y_orn_e = y_edge[orn_idx]
X_mean_e = np.mean(X_edge, axis=0)

kategorik_prefixler = ['http.request.method-','http.referer-','http.request.version-',
    'dns.qry.name.len-','mqtt.conack.flags-','mqtt.protoname-','mqtt.topic-']

lime_fid_edge = []
for m in MODEL_SIRASI:
    model     = ref_edge[m]
    xai       = yukle(os.path.join(EDGE_DIR, f"xai_{m}.pkl"))
    lime_vals = xai['lime_values']  # (500, 74)

    y_orig = model.predict(X_orn_e)
    f1_orig= f1_score(y_orn_e, y_orig, average='macro')

    for n in range(1, 11):
        Xl = X_orn_e.copy()
        for i in range(500):
            top_lime = np.argsort(np.abs(lime_vals[i]))[-n:]
            for idx in top_lime:
                col = fn_edge[idx]
                Xl[i,idx] = 0 if any(col.startswith(p) for p in kategorik_prefixler) else X_mean_e[idx]

        f1l = f1_score(y_orn_e, model.predict(Xl), average='macro')
        lime_fid_edge.append({'model':m,'xai':'LIME','n_maskele':n,
                              'f1_orig':round(f1_orig,4),'f1_mask':round(f1l,4),
                              'delta_f1':round(f1_orig-f1l,4)})

    d5 = next(r['delta_f1'] for r in lime_fid_edge if r['model']==m and r['n_maskele']==5)
    print(f"  {m:<22} ΔF1@5(LIME)={d5:.4f}")
    del xai

df_lime_fid_edge = pd.DataFrame(lime_fid_edge)
kaydet(df_lime_fid_edge, os.path.join(EDGE_DIR, "df_fidelity_lime_edge.pkl"))

# ── CIC-IDS-2017 ──────────────────────────────────────────────────────
print("\n=== CIC-IDS-2017 LIME Fidelity ===")
veri_cic = yukle(os.path.join(CIC_DIR, "veri_cicids.pkl"))
X_cic    = veri_cic['X_scaled']; y_cic = veri_cic['y']
if hasattr(X_cic,'values'): X_cic=X_cic.values
if hasattr(y_cic,'values'): y_cic=y_cic.values

seed0_cic = yukle(os.path.join(CIC_DIR, "modeller_cicids_seed0.pkl"))
ref_cic   = {r['model_adi']:r['model_obj'] for r in seed0_cic if r['fold']==0}

np.random.seed(42)
orn_idx_c = np.random.choice(len(X_cic), size=500, replace=False)
X_orn_c   = X_cic[orn_idx_c]; y_orn_c = y_cic[orn_idx_c]
X_mean_c  = np.mean(X_cic, axis=0)

lime_fid_cic = []
for m in MODEL_SIRASI:
    model     = ref_cic[m]
    xai       = yukle(os.path.join(CIC_DIR, f"xai_cicids_{m}.pkl"))
    lime_vals = xai['lime_values']  # (500, 78)

    y_orig = model.predict(X_orn_c)
    f1_orig= f1_score(y_orn_c, y_orig, average='macro')

    for n in range(1, 11):
        Xl = X_orn_c.copy()
        for i in range(500):
            top_lime = np.argsort(np.abs(lime_vals[i]))[-n:]
            Xl[i, top_lime] = X_mean_c[top_lime]

        f1l = f1_score(y_orn_c, model.predict(Xl), average='macro')
        lime_fid_cic.append({'model':m,'xai':'LIME','n_maskele':n,
                             'f1_orig':round(f1_orig,4),'f1_mask':round(f1l,4),
                             'delta_f1':round(f1_orig-f1l,4)})

    d5 = next(r['delta_f1'] for r in lime_fid_cic if r['model']==m and r['n_maskele']==5)
    print(f"  {m:<22} ΔF1@5(LIME)={d5:.4f}")
    del xai

df_lime_fid_cic = pd.DataFrame(lime_fid_cic)
kaydet(df_lime_fid_cic, os.path.join(CIC_DIR, "df_fidelity_lime_cic.pkl"))
print("\n✅ Tamamlandı!")

In [ ]:
import pickle, os
import numpy as np

CIC_DIR = str(CICIDS2017_DIR)

# CIC tüm sonuçlar
with open(os.path.join(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl"), 'rb') as f:
    cic = pickle.load(f)

print("CIC — SHAP Spearman ortalamaları (pct1 / pct3 / pct5):")
print(f"{'Model':<20} {'pct1':>8} {'pct3':>8} {'pct5':>8}")
for model, veri in cic.items():
    s1 = np.mean(veri['pct1']['shap']['spearman'])
    s3 = np.mean(veri['pct3']['shap']['spearman'])
    s5 = np.mean(veri['pct5']['shap']['spearman'])
    print(f"{model:<20} {s1:>8.4f} {s3:>8.4f} {s5:>8.4f}")

print("\nCIC — LIME Spearman ortalamaları (pct1 / pct3 / pct5):")
print(f"{'Model':<20} {'pct1':>8} {'pct3':>8} {'pct5':>8}")
for model, veri in cic.items():
    l1 = np.mean(veri['pct1']['lime']['spearman'])
    l3 = np.mean(veri['pct3']['lime']['spearman'])
    l5 = np.mean(veri['pct5']['lime']['spearman'])
    print(f"{model:<20} {l1:>8.4f} {l3:>8.4f} {l5:>8.4f}")

# --- DT LIME NaN incelemesi ---
print("\n--- Edge DT LIME NaN incelemesi ---")
EDGE_DIR = str(EDGEIIOT_DIR)
with open(os.path.join(EDGE_DIR, "tum_perturb_sonuclar.pkl"), 'rb') as f:
    edge = pickle.load(f)

dt_lime_pct5 = edge['DecisionTree']['pct5']['lime']['spearman']
arr = np.array(dt_lime_pct5)
print(f"Toplam değer sayısı : {len(arr)}")
print(f"NaN sayısı          : {np.sum(np.isnan(arr))}")
print(f"Sıfır sayısı        : {np.sum(arr == 0)}")
print(f"NaN olmayan ortalama: {np.nanmean(arr):.4f}")
print(f"İlk 10 değer        : {arr[:10]}")

In [ ]:
import pickle, os
import numpy as np

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

with open(os.path.join(EDGE_DIR, "tum_perturb_sonuclar.pkl"), 'rb') as f:
    edge = pickle.load(f)
with open(os.path.join(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl"), 'rb') as f:
    cic = pickle.load(f)

modeller = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
            'LightGBM','CatBoost','LogisticRegression']

print("=" * 95)
print(f"{'RQ1 PERTÜRBASYON STABİLİTESİ — TEMİZ TABLO (@%5 Gürültü)':^95}")
print("=" * 95)
print(f"{'Model':<20} {'EDGE SHAP':>10} {'EDGE LIME':>12} {'CIC SHAP':>10} {'CIC LIME':>12}")
print("-" * 95)

for m in modeller:
    # Edge SHAP
    e_shap = np.nanmean(edge[m]['pct5']['shap']['spearman'])
    # Edge LIME (DT için Jaccard kullan)
    e_lime_arr = np.array(edge[m]['pct5']['lime']['spearman'])
    nan_oran = np.mean(np.isnan(e_lime_arr))
    if nan_oran > 0.5:
        e_lime_str = f"J@5={np.mean(edge[m]['pct5']['lime']['jaccard_5']):.4f}*"
    else:
        e_lime_str = f"{np.nanmean(e_lime_arr):.4f}"

    # CIC SHAP
    c_shap = np.nanmean(cic[m]['pct5']['shap']['spearman'])
    # CIC LIME
    c_lime = np.nanmean(cic[m]['pct5']['lime']['spearman'])

    print(f"{m:<20} {e_shap:>10.4f} {e_lime_str:>12} {c_shap:>10.4f} {c_lime:>12.4f}")

print("-" * 95)
print("* Spearman NaN (sabit açıklama vektörü) — Jaccard@5 ile raporlandı")

# DNN/LSTM
print("\nDNN/LSTM (@%5 Gürültü):")
print(f"{'Model':<8} {'EDGE SHAP':>10} {'EDGE LIME':>12} {'CIC SHAP':>10} {'CIC LIME':>12}")
print("-" * 55)
for tag, fname_e, fname_c in [
    ('DNN',  'perturb_dnn_edge.pkl',  'perturb_dnn_cic.pkl'),
    ('LSTM', 'perturb_lstm_edge.pkl', 'perturb_lstm_cic.pkl')
]:
    with open(os.path.join(DNN_DIR, fname_e), 'rb') as f: de = pickle.load(f)
    with open(os.path.join(DNN_DIR, fname_c), 'rb') as f: dc = pickle.load(f)
    es = np.nanmean(de['pct5']['shap']['spearman'])
    el = np.nanmean(de['pct5']['lime']['spearman'])
    cs = np.nanmean(dc['pct5']['shap']['spearman'])
    cl = np.nanmean(dc['pct5']['lime']['spearman'])
    print(f"{tag:<8} {es:>10.4f} {el:>12.4f} {cs:>10.4f} {cl:>12.4f}")

In [ ]:
import pickle, os
import numpy as np
from scipy.stats import pearsonr, spearmanr
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# --- Edge: df_var.pkl'den RQ3 hesapla ---
with open(os.path.join(EDGE_DIR, "df_var.pkl"), 'rb') as f:
    df_edge = pickle.load(f)

print("Edge df_var sütunları:", df_edge.columns.tolist())
print("Edge df_var shape:", df_edge.shape)
print(df_edge.head(3))
print()

# --- CIC: df_rq3_cicids.pkl ---
with open(os.path.join(CIC_DIR, "df_rq3_cicids.pkl"), 'rb') as f:
    df_cic = pickle.load(f)

print("CIC df_rq3 sütunları:", df_cic.columns.tolist())
print("CIC df_rq3 shape:", df_cic.shape)
print(df_cic.head(3))
print()

# --- Genel korelasyon ---
# Edge
edge_f1 = df_edge['f1_macro'].values
edge_sp  = df_edge['spearman'].values
mask_e   = ~np.isnan(edge_f1) & ~np.isnan(edge_sp)
pr_e, pp_e = pearsonr(edge_f1[mask_e], edge_sp[mask_e])
sr_e, sp_e = spearmanr(edge_f1[mask_e], edge_sp[mask_e])

# CIC
cic_f1 = df_cic['f1_macro'].values
cic_sp  = df_cic['spearman'].values
mask_c  = ~np.isnan(cic_f1) & ~np.isnan(cic_sp)
pr_c, pp_c = pearsonr(cic_f1[mask_c], cic_sp[mask_c])
sr_c, sp_c = spearmanr(cic_f1[mask_c], cic_sp[mask_c])

print("=" * 60)
print("RQ3 GENEL KORElASYON")
print("=" * 60)
print(f"Edge  — Pearson r={pr_e:.3f} (p={pp_e:.4f}), Spearman r={sr_e:.3f} (p={sp_e:.4f})")
print(f"CIC   — Pearson r={pr_c:.3f} (p={pp_c:.4f}), Spearman r={sr_c:.3f} (p={sp_c:.4f})")

# --- Model bazında kırılım ---
print("\nEDGE — Model bazında Pearson r (F1 ~ SHAP Spearman):")
print(f"{'Model':<22} {'n':>4} {'Pearson r':>10} {'p':>8} {'F1 min':>8} {'F1 max':>8} {'Sp min':>8} {'Sp max':>8}")
print("-" * 80)
for model, grp in df_edge.groupby('model_adi'):
    f = grp['f1_macro'].values
    s = grp['spearman'].values
    mask = ~np.isnan(f) & ~np.isnan(s)
    if mask.sum() < 5:
        print(f"{model:<22} {mask.sum():>4} {'yetersiz':>10}")
        continue
    r, p = pearsonr(f[mask], s[mask])
    print(f"{model:<22} {mask.sum():>4} {r:>10.3f} {p:>8.4f} "
          f"{f[mask].min():>8.4f} {f[mask].max():>8.4f} "
          f"{s[mask].min():>8.4f} {s[mask].max():>8.4f}")

print("\nCIC — Model bazında Pearson r (F1 ~ SHAP Spearman):")
print(f"{'Model':<22} {'n':>4} {'Pearson r':>10} {'p':>8} {'F1 min':>8} {'F1 max':>8} {'Sp min':>8} {'Sp max':>8}")
print("-" * 80)
for model, grp in df_cic.groupby('model_adi'):
    f = grp['f1_macro'].values
    s = grp['spearman'].values
    mask = ~np.isnan(f) & ~np.isnan(s)
    if mask.sum() < 5:
        print(f"{model:<22} {mask.sum():>4} {'yetersiz':>10}")
        continue
    r, p = pearsonr(f[mask], s[mask])
    print(f"{model:<22} {mask.sum():>4} {r:>10.3f} {p:>8.4f} "
          f"{f[mask].min():>8.4f} {f[mask].max():>8.4f} "
          f"{s[mask].min():>8.4f} {s[mask].max():>8.4f}")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

with open(os.path.join(EDGE_DIR, "df_var.pkl"), 'rb') as f:
    df_edge = pickle.load(f)
with open(os.path.join(CIC_DIR, "df_rq3_cicids.pkl"), 'rb') as f:
    df_cic = pickle.load(f)

# Model bazında ortalamalar — Simpson'ı görmek için
print("EDGE — Model ortalamalar (global pattern'ı açıklıyor):")
print(f"{'Model':<22} {'F1 ort':>8} {'Sp ort':>8} {'Sp std':>8}")
print("-" * 52)
for model, grp in df_edge.groupby('model_adi'):
    print(f"{model:<22} {grp['f1_macro'].mean():>8.4f} "
          f"{grp['spearman'].mean():>8.4f} "
          f"{grp['spearman'].std():>8.4f}")

print("\nCIC — Model ortalamalar:")
print(f"{'Model':<22} {'F1 ort':>8} {'Sp ort':>8} {'Sp std':>8}")
print("-" * 52)
for model, grp in df_cic.groupby('model_adi'):
    print(f"{model:<22} {grp['f1_macro'].mean():>8.4f} "
          f"{grp['spearman'].mean():>8.4f} "
          f"{grp['spearman'].std():>8.4f}")

# Edge Spearman dağılımı — neden bu kadar çok 1.0 var?
print("\nEDGE Spearman değer dağılımı:")
for model, grp in df_edge.groupby('model_adi'):
    sp = grp['spearman'].values
    n_tam1 = np.sum(sp == 1.0)
    print(f"  {model:<22}: mean={sp.mean():.4f}, "
          f"min={sp.min():.4f}, max={sp.max():.4f}, "
          f"==1.0 sayısı: {n_tam1}/50")

# LR'nin etkisini izole et
print("\n--- LR çıkarılınca Edge global Pearson ---")
df_no_lr = df_edge[df_edge['model_adi'] != 'LogisticRegression']
f = df_no_lr['f1_macro'].values
s = df_no_lr['spearman'].values
r, p = pearsonr(f, s)
print(f"LR hariç Edge: Pearson r={r:.3f}, p={p:.4f}")

print("\n--- LR çıkarılınca CIC global Pearson ---")
df_no_lr_c = df_cic[df_cic['model_adi'] != 'LogisticRegression']
f = df_no_lr_c['f1_macro'].values
s = df_no_lr_c['spearman'].values
r, p = pearsonr(f, s)
print(f"LR hariç CIC: Pearson r={r:.3f}, p={p:.4f}")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

with open(os.path.join(EDGE_DIR, "df_var.pkl"), 'rb') as f:
    df_edge = pickle.load(f)
with open(os.path.join(CIC_DIR, "df_rq3_cicids.pkl"), 'rb') as f:
    df_cic = pickle.load(f)

def bootstrap_ci(x, y, func, n=2000, alpha=0.95):
    rng = np.random.default_rng(42)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, len(x), len(x))
        try:
            r, _ = func(x[idx], y[idx])
            stats.append(r)
        except:
            pass
    lo = np.percentile(stats, (1-alpha)/2*100)
    hi = np.percentile(stats, (1+alpha)/2*100)
    return np.mean(stats), lo, hi

def rq3_rapor(df, isim):
    print(f"\n{'='*65}")
    print(f"  {isim}")
    print(f"{'='*65}")

    senaryolar = [
        ("Tüm modeller",        df),
        ("LR hariç",            df[df['model_adi'] != 'LogisticRegression']),
        ("Sadece ağaç modeller",df[df['model_adi'].isin(
            ['RandomForest','ExtraTrees','DecisionTree',
             'XGBoost','LightGBM','CatBoost'])]),
    ]

    for etiket, sub in senaryolar:
        f = sub['f1_macro'].values
        s = sub['spearman'].values
        mask = ~np.isnan(f) & ~np.isnan(s)
        f, s = f[mask], s[mask]

        pr, pp = pearsonr(f, s)
        sr, sp = spearmanr(f, s)
        _, plo, phi = bootstrap_ci(f, s, pearsonr)

        print(f"\n  [{etiket}] n={len(f)}")
        print(f"    Pearson  r = {pr:+.3f}  (p={pp:.4f})  95% CI [{plo:+.3f}, {phi:+.3f}]")
        print(f"    Spearman r = {sr:+.3f}  (p={sp:.4f})")

    # LR'nin konumunu göster
    print(f"\n  --- LR'nin aykırı konumu ---")
    lr_e = df[df['model_adi'] == 'LogisticRegression']
    oth  = df[df['model_adi'] != 'LogisticRegression']
    print(f"  LR   : F1={lr_e['f1_macro'].mean():.4f}, Spearman={lr_e['spearman'].mean():.4f} ±{lr_e['spearman'].std():.4f}")
    print(f"  Diğer: F1={oth['f1_macro'].mean():.4f}, Spearman={oth['spearman'].mean():.4f} ±{oth['spearman'].std():.4f}")

rq3_rapor(df_edge, "EDGE-IIoT RQ3")
rq3_rapor(df_cic,  "CIC-IDS-2017 RQ3")

# Her iki veri setini birleştir — LR hariç
print(f"\n{'='*65}")
print("  BİRLEŞİK (her iki dataset, LR hariç)")
print(f"{'='*65}")
df_e2 = df_edge[df_edge['model_adi'] != 'LogisticRegression'].copy()
df_c2 = df_cic[df_cic['model_adi']  != 'LogisticRegression'].copy()
df_e2['dataset'] = 'Edge'; df_c2['dataset'] = 'CIC'
df_all = pd.concat([df_e2, df_c2], ignore_index=True)
f = df_all['f1_macro'].values
s = df_all['spearman'].values
mask = ~np.isnan(f) & ~np.isnan(s)
pr, pp = pearsonr(f[mask], s[mask])
_, lo, hi = bootstrap_ci(f[mask], s[mask], pearsonr)
print(f"  Pearson r = {pr:+.3f}  (p={pp:.4f})  95% CI [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
import pickle, os, gc
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

# Mevcut fidelity checkpoint'lerini yükle — yapıyı görelim
for fname, label in [
    (os.path.join(EDGE_DIR, "df_fidelity.pkl"),      "Edge SHAP fidelity"),
    (os.path.join(CIC_DIR,  "df_fidelity_cicids.pkl"),"CIC SHAP fidelity"),
]:
    with open(fname, 'rb') as f:
        df = pickle.load(f)
    print(f"\n{label}:")
    print(f"  Sütunlar : {df.columns.tolist()}")
    print(f"  Shape    : {df.shape}")
    print(f"  n_maskele: {df['n_maskele'].unique() if 'n_maskele' in df.columns else 'YOK'}")
    print(df.head(3).to_string())

# Veri checkpoint yapısını kontrol et
print("\n\nEdge veri checkpoint kontrol:")
with open(os.path.join(EDGE_DIR, "veri.pkl"), 'rb') as f:
    veri = pickle.load(f)
print(f"  Anahtarlar: {list(veri.keys())}")
X = veri['X_multi_scaled']
y = veri['y_multi']
feature_names = veri['feature_names_multi']
print(f"  X tipi: {type(X)}, shape: {X.shape if hasattr(X,'shape') else len(X)}")
print(f"  y tipi: {type(y)}")

In [ ]:
import pickle, os, gc
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def maskele_ve_hesapla(model_obj, X_test, y_test, shap_vals,
                       feature_names, k_list, n_tekrar=30, seed=42):
    """Her k için top-k, random ve least-important baseline hesapla."""
    rng = np.random.default_rng(seed)
    X = X_test if isinstance(X_test, np.ndarray) else X_test.values
    feat_means = X.mean(axis=0)

    # Orijinal F1
    y_pred_orig = model_obj.predict(X)
    f1_orig = f1_score(y_test, y_pred_orig, average='macro', zero_division=0)

    # SHAP ortalama önem sırası
    shap_mean = np.abs(shap_vals).mean(axis=0) if shap_vals.ndim == 2 else shap_vals
    top_k_idx    = np.argsort(shap_mean)[::-1]   # yüksekten düşüğe
    least_k_idx  = np.argsort(shap_mean)          # düşükten yükseğe

    sonuclar = []
    for k in k_list:
        # --- Top-k (zaten var ama kontrol için) ---
        X_top = X.copy()
        for i in top_k_idx[:k]:
            X_top[:, i] = feat_means[i]
        f1_top = f1_score(y_test, model_obj.predict(X_top),
                          average='macro', zero_division=0)

        # --- Least-important baseline ---
        X_least = X.copy()
        for i in least_k_idx[:k]:
            X_least[:, i] = feat_means[i]
        f1_least = f1_score(y_test, model_obj.predict(X_least),
                            average='macro', zero_division=0)

        # --- Random baseline (n_tekrar ortalama) ---
        f1_rand_list = []
        all_idx = np.arange(X.shape[1])
        for _ in range(n_tekrar):
            rand_idx = rng.choice(all_idx, size=k, replace=False)
            X_rand = X.copy()
            for i in rand_idx:
                X_rand[:, i] = feat_means[i]
            f1_rand_list.append(f1_score(y_test, model_obj.predict(X_rand),
                                         average='macro', zero_division=0))

        sonuclar.append({
            'k':          k,
            'f1_orig':    f1_orig,
            'f1_topk':    f1_top,
            'f1_least':   f1_least,
            'f1_random':  np.mean(f1_rand_list),
            'delta_topk':   f1_orig - f1_top,
            'delta_least':  f1_orig - f1_least,
            'delta_random': f1_orig - np.mean(f1_rand_list),
        })
    return sonuclar

# ── EDGE ──────────────────────────────────────────────────────
print("Edge baseline hesaplanıyor...")
with open(os.path.join(EDGE_DIR, "veri.pkl"), 'rb') as f:
    veri = pickle.load(f)
X_edge = veri['X_multi_scaled'].values
y_edge = veri['y_multi']
feature_names_edge = veri['feature_names_multi']

modeller_klasik = ['RandomForest','ExtraTrees','DecisionTree',
                   'XGBoost','LightGBM','CatBoost','LogisticRegression']

k_list = list(range(1, 21))
edge_baseline_rows = []

for model_adi in modeller_klasik:
    print(f"  {model_adi}...", end=' ')

    # XAI değerlerini yükle
    xai_path = os.path.join(EDGE_DIR, f"xai_{model_adi}.pkl")
    if not os.path.exists(xai_path):
        print("XAI YOK, atlandı"); continue
    with open(xai_path, 'rb') as f:
        xai = pickle.load(f)
    shap_vals = xai['shap_values']  # (500, 74)

    # Seed 0 fold 0 modeli yükle
    model_path = os.path.join(EDGE_DIR, "modeller_seed0.pkl")
    with open(model_path, 'rb') as f:
        modeller_list = pickle.load(f)

    kayit = next((m for m in modeller_list
                  if m['model_adi'] == model_adi and m['fold'] == 0), None)
    if kayit is None:
        print("MODEL YOK, atlandı"); continue

    model_obj = kayit['model_obj']
    X_test    = kayit['X_test']
    y_test    = kayit['y_test']

    rows = maskele_ve_hesapla(model_obj, X_test, y_test,
                              shap_vals, feature_names_edge, k_list)
    for r in rows:
        r['model'] = model_adi
        r['dataset'] = 'Edge'
    edge_baseline_rows.extend(rows)
    print("OK")

# ── CIC ───────────────────────────────────────────────────────
print("\nCIC baseline hesaplanıyor...")
with open(os.path.join(CIC_DIR, "veri_cicids.pkl"), 'rb') as f:
    veri_cic = pickle.load(f)
X_cic = veri_cic['X_scaled']
y_cic = veri_cic['y']
if hasattr(X_cic, 'values'): X_cic = X_cic.values
if hasattr(y_cic, 'values'): y_cic = y_cic.values

cic_baseline_rows = []

for model_adi in modeller_klasik:
    print(f"  {model_adi}...", end=' ')

    xai_path = os.path.join(CIC_DIR, f"xai_cicids_{model_adi}.pkl")
    if not os.path.exists(xai_path):
        print("XAI YOK, atlandı"); continue
    with open(xai_path, 'rb') as f:
        xai = pickle.load(f)
    shap_vals = xai['shap_values']

    model_path = os.path.join(CIC_DIR, "modeller_cicids_seed0.pkl")
    with open(model_path, 'rb') as f:
        modeller_list = pickle.load(f)

    kayit = next((m for m in modeller_list
                  if m['model_adi'] == model_adi and m['fold'] == 0), None)
    if kayit is None:
        print("MODEL YOK, atlandı"); continue

    model_obj = kayit['model_obj']
    X_test    = kayit['X_test']
    y_test    = kayit['y_test']

    rows = maskele_ve_hesapla(model_obj, X_test, y_test,
                              shap_vals, None, k_list)
    for r in rows:
        r['model'] = model_adi
        r['dataset'] = 'CIC'
    cic_baseline_rows.extend(rows)
    print("OK")

# ── Kaydet ve önizle ─────────────────────────────────────────
df_baseline = pd.DataFrame(edge_baseline_rows + cic_baseline_rows)
out_path = os.path.join(EDGE_DIR, "df_fidelity_baseline.pkl")
with open(out_path, 'wb') as f:
    pickle.dump(df_baseline, f)
print(f"\nKaydedildi: {out_path}")

# @k=5 karşılaştırma tablosu
print("\n@k=5 Karşılaştırma (ΔF1 — yüksek = o özellikler önemli):")
print(f"{'Model':<22} {'Dataset':<6} {'Top-k':>8} {'Random':>8} {'Least':>8} {'Oran*':>8}")
print("-" * 62)
for _, row in df_baseline[df_baseline['k']==5].iterrows():
    oran = row['delta_topk'] / row['delta_random'] if row['delta_random'] > 0.001 else float('inf')
    print(f"{row['model']:<22} {row['dataset']:<6} "
          f"{row['delta_topk']:>8.4f} {row['delta_random']:>8.4f} "
          f"{row['delta_least']:>8.4f} {oran:>8.1f}x")
print("* Top-k ΔF1 / Random ΔF1 — SHAP'in rastgeleden kaç kat daha etkili olduğu")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# Yükle
with open(os.path.join(EDGE_DIR, "df_xai_score.pkl"), 'rb') as f:
    df_e7 = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_xai_score_cicids.pkl"), 'rb') as f:
    df_c7 = pickle.load(f)
with open(os.path.join(DNN_DIR,  "df_xai_score_dnn_edge.pkl"), 'rb') as f:
    df_dnn_e = pickle.load(f)
with open(os.path.join(DNN_DIR,  "df_xai_score_lstm_edge.pkl"), 'rb') as f:
    df_lstm_e = pickle.load(f)
with open(os.path.join(DNN_DIR,  "df_xai_score_dnn_cic.pkl"), 'rb') as f:
    df_dnn_c = pickle.load(f)
with open(os.path.join(DNN_DIR,  "df_xai_score_lstm_cic.pkl"), 'rb') as f:
    df_lstm_c = pickle.load(f)

print("Edge 7-model sütunları:", df_e7.columns.tolist())
print(df_e7.to_string(index=False))
print("\nDNN Edge sütunları:", df_dnn_e.columns.tolist())
print(df_dnn_e.to_string(index=False))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# Tüm dosyaları yükle
with open(os.path.join(DNN_DIR, "df_xai_score_lstm_edge.pkl"), 'rb') as f:
    df_lstm_e = pickle.load(f)
with open(os.path.join(CIC_DIR, "df_xai_score_cicids.pkl"), 'rb') as f:
    df_c7 = pickle.load(f)
with open(os.path.join(DNN_DIR, "df_xai_score_dnn_cic.pkl"), 'rb') as f:
    df_dnn_c = pickle.load(f)
with open(os.path.join(DNN_DIR, "df_xai_score_lstm_cic.pkl"), 'rb') as f:
    df_lstm_c = pickle.load(f)

# CIC sütun adını kontrol et
print("CIC sütunları:", df_c7.columns.tolist())
print("LSTM Edge sütunları:", df_lstm_e.columns.tolist())
print(df_lstm_e.to_string(index=False))
print("\nCIC 7-model:")
print(df_c7.to_string(index=False))
print("\nDNN CIC:")
print(df_dnn_c.to_string(index=False))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

DNN_DIR = str(DNN_LSTM_DIR)
CIC_DIR = str(CICIDS2017_DIR)

with open(os.path.join(DNN_DIR, "df_xai_score_lstm_cic.pkl"), 'rb') as f:
    df_lstm_c = pickle.load(f)

# df_lstm_e ve df_lstm_c zaten 9 modeli içeriyor (en kapsamlı dosyalar)
# CIC için LSTM dosyası 9 modeli içeriyor mu kontrol et
print("LSTM CIC (9 model?):")
print(df_lstm_c.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

# 9'lu tablolar hazır — Edge: df_lstm_e, CIC: df_lstm_c
# Sütun adı farkını normalize et
df_edge9 = df_lstm_e.copy().rename(columns={'Fidelity_d5': 'Fidelity'})
df_cic9  = df_lstm_c.copy().rename(columns={'Fidelity_5': 'Fidelity'})

def normalize(vals, inverse=False):
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return pd.Series([0.5] * len(vals), index=vals.index)
    n = (vals - mn) / (mx - mn)
    return 1 - n if inverse else n

def xai_score_hesapla(df, w_perturb, w_varyans, w_fidelity, w_sure):
    d = df.copy()
    d['n_Perturb']  = normalize(d['Perturb'])
    d['n_Varyans']  = normalize(d['Varyans'])
    d['n_Fidelity'] = normalize(d['Fidelity'])
    d['n_Sure']     = normalize(d['Sure_ms'], inverse=True)
    d['Score'] = (w_perturb  * d['n_Perturb'] +
                  w_varyans  * d['n_Varyans'] +
                  w_fidelity * d['n_Fidelity'] +
                  w_sure     * d['n_Sure'])
    return d[['Model', 'Score']].sort_values('Score', ascending=False).reset_index(drop=True)

# 5 ağırlık senaryosu
senaryolar = {
    'S1: Eşit (%25×4)':         (0.25, 0.25, 0.25, 0.25),
    'S2: Hız hariç (%33×3)':    (0.33, 0.33, 0.34, 0.00),
    'S3: Stabilite ağırlıklı':  (0.40, 0.30, 0.20, 0.10),
    'S4: Fidelity ağırlıklı':   (0.20, 0.20, 0.40, 0.20),
    'S5: Güvenilirlik (%50×2)': (0.50, 0.50, 0.00, 0.00),
}

for dataset_adi, df9 in [('EDGE-IIoT', df_edge9), ('CIC-IDS-2017', df_cic9)]:
    print(f"\n{'='*85}")
    print(f"  {dataset_adi} — Sensitivity Analizi (Sıralama)")
    print(f"{'='*85}")

    # Her senaryo için sıralama
    tablo = pd.DataFrame({'Model': df9['Model']})
    for s_adi, (wp, wv, wf, ws) in senaryolar.items():
        sonuc = xai_score_hesapla(df9, wp, wv, wf, ws)
        sonuc['Sira'] = range(1, len(sonuc)+1)
        tablo = tablo.merge(
            sonuc[['Model','Sira']].rename(columns={'Sira': s_adi}),
            on='Model')

    # Mevcut sıralamaya göre sırala (S1)
    tablo = tablo.sort_values('S1: Eşit (%25×4)').reset_index(drop=True)
    print(tablo.to_string(index=False))

    # Sıralama değişimi analizi
    print(f"\n  Maksimum sıralama değişimi (en kötü durum):")
    sira_cols = [c for c in tablo.columns if c != 'Model']
    for _, row in tablo.iterrows():
        siralar = [row[c] for c in sira_cols]
        delta = max(siralar) - min(siralar)
        if delta > 1:
            print(f"    {row['Model']:<22}: min={min(siralar)} max={max(siralar)} → Δ={delta}")

    # Üst 3 istikrarı
    print(f"\n  Üst-3 her senaryoda:")
    for s_adi in sira_cols:
        top3 = tablo.nsmallest(3, s_adi)['Model'].tolist()
        print(f"    {s_adi:<30}: {', '.join(top3)}")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

# Mevcut istatistik dosyalarını kontrol et
for fname, label in [
    (os.path.join(EDGE_DIR, "df_istat.pkl"),       "Edge istat"),
    (os.path.join(CIC_DIR,  "df_istat_cicids.pkl"),"CIC istat"),
]:
    with open(fname, 'rb') as f:
        df = pickle.load(f)
    print(f"\n{label}:")
    print(f"  Sütunlar: {df.columns.tolist()}")
    print(f"  Shape   : {df.shape}")
    print(df.head(4).to_string(index=False))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

with open(os.path.join(EDGE_DIR, "tum_perturb_sonuclar.pkl"), 'rb') as f:
    edge = pickle.load(f)
with open(os.path.join(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl"), 'rb') as f:
    cic = pickle.load(f)
with open(os.path.join(EDGE_DIR, "df_istat.pkl"), 'rb') as f:
    df_istat_e = pickle.load(f)
with open(os.path.join(CIC_DIR, "df_istat_cicids.pkl"), 'rb') as f:
    df_istat_c = pickle.load(f)

def rank_biserial_r_paired(x, y):
    """Paired rank-biserial r — sadece her ikisi de geçerli olan indeksleri kullan."""
    x, y = np.array(x), np.array(y)
    mask = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[mask], y[mask]
    diff = x - y
    diff = diff[diff != 0]
    n = len(diff)
    if n == 0: return np.nan
    ranks = np.argsort(np.abs(diff)) + 1
    W_plus  = ranks[diff > 0].sum()
    W_minus = ranks[diff < 0].sum()
    return (W_plus - W_minus) / (n * (n + 1) / 2)

def bootstrap_mean_ci(arr, n=2000, alpha=0.95):
    arr = arr[~np.isnan(arr)]
    rng = np.random.default_rng(42)
    means = [rng.choice(arr, size=len(arr), replace=True).mean()
             for _ in range(n)]
    lo = np.percentile(means, (1-alpha)/2*100)
    hi = np.percentile(means, (1+alpha)/2*100)
    return lo, hi

modeller = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
            'LightGBM','CatBoost','LogisticRegression']

def istat_tablosu(perturb_data, df_istat, gurultu_col, gurultu_val, dataset_adi):
    print(f"\n{'='*100}")
    print(f"  {dataset_adi} — İstatistiksel Özet (@%5 Gürültü, SHAP vs LIME)")
    print(f"{'='*100}")
    print(f"{'Model':<22} {'SHAP ort':>9} {'LIME ort':>9} "
          f"{'CI SHAP':>16} {'CI LIME':>16} "
          f"{'Cohen d':>9} {'RB-r':>7} {'p-bonf':>12} {'Sig':>4}")
    print("-"*100)

    rows = []
    for model in modeller:
        shap_raw = np.array(perturb_data[model]['pct5']['shap']['spearman'])
        lime_raw = np.array(perturb_data[model]['pct5']['lime']['spearman'])

        # DT LIME özel durum: Jaccard ile raporla, RBr hesaplama
        lime_all_nan = np.mean(np.isnan(lime_raw)) > 0.5

        shap_clean = shap_raw[~np.isnan(shap_raw)]
        lime_clean = lime_raw[~np.isnan(lime_raw)]

        shap_ci = bootstrap_mean_ci(shap_raw)
        lime_ci = bootstrap_mean_ci(lime_raw) if not lime_all_nan else (np.nan, np.nan)

        rbr = rank_biserial_r_paired(shap_raw, lime_raw)

        # İstatistik satırı
        row_df = df_istat[(df_istat['Model']==model) &
                           (df_istat[gurultu_col]==gurultu_val)]
        if len(row_df) == 0:
            print(f"  {model}: istat satırı bulunamadı"); continue
        row = row_df.iloc[0]

        cd  = row.get('cohen_d', row.get('Cohens_d', np.nan))
        pb  = row.get('p_wilcoxon', row.get('p_bonf', np.nan))
        if gurultu_col == 'Gürültü':
            pb = min(pb * 21, 1.0)  # Edge'de bonf düzeltmesi elle uygula
        sig = '✅' if pb < 0.05 else '❌'

        shap_str = f"{shap_clean.mean():.4f}"
        lime_str = f"J@5={np.mean(perturb_data[model]['pct5']['lime']['jaccard_5']):.4f}*" \
                   if lime_all_nan else f"{lime_clean.mean():.4f}"
        ci_lime  = f"[N/A—NaN]" if lime_all_nan else \
                   f"[{lime_ci[0]:.4f},{lime_ci[1]:.4f}]"
        rbr_str  = "N/A*" if np.isnan(rbr) else f"{rbr:.4f}"

        print(f"{model:<22} {shap_str:>9} {lime_str:>9} "
              f"[{shap_ci[0]:.4f},{shap_ci[1]:.4f}] "
              f"{ci_lime:>16} "
              f"{cd:>9.3f} {rbr_str:>7} {pb:>12.2e} {sig:>4}")

        rows.append({'Model':model,
                     'SHAP_mean':shap_clean.mean(),
                     'LIME_mean':lime_clean.mean() if not lime_all_nan else np.nan,
                     'SHAP_CI_lo':shap_ci[0], 'SHAP_CI_hi':shap_ci[1],
                     'Cohen_d':cd, 'RB_r':rbr, 'p_bonf':pb,
                     'DT_lime_nan': lime_all_nan})
    if lime_all_nan:
        print("  * DT LIME: 474/500 NaN — Spearman hesaplanamadı, Jaccard@5 ile raporlandı")
    return pd.DataFrame(rows)

df_final_e = istat_tablosu(edge, df_istat_e, 'Gürültü', '%5', 'EDGE-IIoT')
df_final_c = istat_tablosu(cic,  df_istat_c, 'Gurultu', 'pct5', 'CIC-IDS-2017')

# Kaydet
with open(os.path.join(EDGE_DIR, "df_istat_final.pkl"), 'wb') as f:
    pickle.dump(df_final_e, f)
with open(os.path.join(CIC_DIR,  "df_istat_final_cicids.pkl"), 'wb') as f:
    pickle.dump(df_final_c, f)
print("\n✅ Kaydedildi.")

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# Tüm fidelity dosyalarını yükle
with open(os.path.join(EDGE_DIR, "df_fidelity.pkl"), 'rb') as f:
    df_shap_e = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_fidelity_cicids.pkl"), 'rb') as f:
    df_shap_c = pickle.load(f)
with open(os.path.join(EDGE_DIR, "df_fidelity_lime_edge.pkl"), 'rb') as f:
    df_lime_e = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_fidelity_lime_cic.pkl"), 'rb') as f:
    df_lime_c = pickle.load(f)
with open(os.path.join(EDGE_DIR, "df_fidelity_baseline.pkl"), 'rb') as f:
    df_base = pickle.load(f)

# DNN/LSTM fidelity kontrol
print("DNN_DIR fidelity dosyaları:")
import glob
for f in glob.glob(os.path.join(DNN_DIR, "*fidelity*")):
    print(" ", os.path.basename(f))

print("\nEdge SHAP fidelity sütunları:", df_shap_e.columns.tolist())
print("Edge LIME fidelity sütunları:", df_lime_e.columns.tolist())
print("Baseline sütunları:", df_base.columns.tolist())
print("\nBaseline shape:", df_base.shape)
print(df_base[df_base['dataset']=='Edge'].head(3).to_string(index=False))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# DNN/LSTM yapısını kontrol et
with open(os.path.join(DNN_DIR, "fidelity_dnn_edge.pkl"), 'rb') as f:
    dnn_e = pickle.load(f)
print("DNN Edge fidelity tipi:", type(dnn_e))
if isinstance(dnn_e, pd.DataFrame):
    print("Sütunlar:", dnn_e.columns.tolist())
    print(dnn_e.head(3).to_string(index=False))
elif isinstance(dnn_e, dict):
    print("Anahtarlar:", list(dnn_e.keys()))

In [ ]:
import pickle, os
import numpy as np
import pandas as pd

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)
DNN_DIR  = str(DNN_LSTM_DIR)

# Tüm fidelity yükle
with open(os.path.join(EDGE_DIR, "df_fidelity.pkl"),          'rb') as f: df_shap_e  = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_fidelity_cicids.pkl"),   'rb') as f: df_shap_c  = pickle.load(f)
with open(os.path.join(EDGE_DIR, "df_fidelity_lime_edge.pkl"),'rb') as f: df_lime_e  = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_fidelity_lime_cic.pkl"), 'rb') as f: df_lime_c  = pickle.load(f)
with open(os.path.join(EDGE_DIR, "df_fidelity_baseline.pkl"), 'rb') as f: df_base    = pickle.load(f)
with open(os.path.join(DNN_DIR,  "fidelity_dnn_edge.pkl"),    'rb') as f: dnn_e      = pickle.load(f)
with open(os.path.join(DNN_DIR,  "fidelity_lstm_edge.pkl"),   'rb') as f: lstm_e     = pickle.load(f)
with open(os.path.join(DNN_DIR,  "fidelity_dnn_cic.pkl"),     'rb') as f: dnn_c      = pickle.load(f)
with open(os.path.join(DNN_DIR,  "fidelity_lstm_cic.pkl"),    'rb') as f: lstm_c     = pickle.load(f)

# Unified format: model, dataset, xai, k, delta_f1
def std_format(df, dataset, model_col='model', xai_col='xai',
               k_col='n_maskele', delta_col='delta_f1'):
    d = df[[model_col, xai_col, k_col, delta_col]].copy()
    d.columns = ['model','xai','k','delta_f1']
    d['dataset'] = dataset
    return d

rows = []
rows.append(std_format(df_shap_e, 'Edge'))
rows.append(std_format(df_lime_e, 'Edge'))
rows.append(std_format(df_shap_c, 'CIC'))
rows.append(std_format(df_lime_c, 'CIC'))
rows.append(std_format(dnn_e,  'Edge'))
rows.append(std_format(lstm_e, 'Edge'))
rows.append(std_format(dnn_c,  'CIC'))
rows.append(std_format(lstm_c, 'CIC'))
df_all = pd.concat(rows, ignore_index=True)

# Baseline'ı ekle
base_rows = []
for _, row in df_base.iterrows():
    base_rows.append({'model':row['model'],'xai':'Random',
                      'k':row['k'],'delta_f1':row['delta_random'],'dataset':row['dataset']})
    base_rows.append({'model':row['model'],'xai':'Least',
                      'k':row['k'],'delta_f1':row['delta_least'],'dataset':row['dataset']})
df_all = pd.concat([df_all, pd.DataFrame(base_rows)], ignore_index=True)

# Kaydet
out = os.path.join(EDGE_DIR, "df_deletion_curve_full.pkl")
with open(out, 'wb') as f: pickle.dump(df_all, f)
print(f"Kaydedildi: {out}")
print(f"Toplam satır: {len(df_all)}")
print(f"Modeller: {sorted(df_all['model'].unique())}")
print(f"XAI tipleri: {sorted(df_all['xai'].unique())}")

# Özet tablo — anahtar k değerleri
K_LIST = [1, 3, 5, 10, 15, 20]
modeller9 = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
             'LightGBM','CatBoost','LogisticRegression','DNN','LSTM']

for dataset in ['Edge','CIC']:
    print(f"\n{'='*100}")
    print(f"  {dataset} — Deletion Curve ΔF1 (SHAP / LIME / Random / Least) @ anahtar k değerleri")
    print(f"{'='*100}")
    header = f"{'Model':<22} {'k':>3}  {'SHAP':>7}  {'LIME':>7}  {'Random':>7}  {'Least':>7}  {'SHAP/Rand':>10}"
    print(header)
    print("-"*100)

    sub = df_all[df_all['dataset']==dataset]
    for model in modeller9:
        m_sub = sub[sub['model']==model]
        if len(m_sub) == 0: continue
        for k in [3, 5, 10]:
            shap_v  = m_sub[(m_sub['xai']=='SHAP')  & (m_sub['k']==k)]['delta_f1'].values
            lime_v  = m_sub[(m_sub['xai']=='LIME')  & (m_sub['k']==k)]['delta_f1'].values
            rand_v  = m_sub[(m_sub['xai']=='Random')& (m_sub['k']==k)]['delta_f1'].values
            least_v = m_sub[(m_sub['xai']=='Least') & (m_sub['k']==k)]['delta_f1'].values

            sv = shap_v[0]  if len(shap_v)  else np.nan
            lv = lime_v[0]  if len(lime_v)  else np.nan
            rv = rand_v[0]  if len(rand_v)  else np.nan
            lsv= least_v[0] if len(least_v) else np.nan
            ratio = sv/rv if (not np.isnan(sv) and rv > 0.001) else np.nan
            ratio_str = f"{ratio:.1f}x" if not np.isnan(ratio) else "—"

            model_str = model if k==3 else ""
            print(f"{model_str:<22} {k:>3}  {sv:>7.4f}  {lv:>7.4f}  "
                  f"{rv:>7.4f}  {lsv:>7.4f}  {ratio_str:>10}")
        print()

In [ ]:
import pickle, os
import numpy as np
import pandas as pd
import lime, lime.lime_tabular
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

EDGE_DIR = str(EDGEIIOT_DIR)
CIC_DIR  = str(CICIDS2017_DIR)

def lime_to_vec(lime_exp, n_features):
    vec = np.zeros(n_features)
    for feat_idx, val in lime_exp.local_exp[lime_exp.available_labels()[0]]:
        vec[feat_idx] = val
    return vec

def lime_stochastic(model_obj, X_background, X_samples, n_features,
                    n_tekrar=10, num_samples=1000):
    explainer = lime.lime_tabular.LimeTabularExplainer(
        X_background,
        mode='classification',
        discretize_continuous=False,
        verbose=False
    )
    tum_spearman = []
    for i, x in enumerate(X_samples):
        aciklamalar = []
        for t in range(n_tekrar):
            np.random.seed(t * 100)  # her tekrarda farklı seed
            exp = explainer.explain_instance(
                x,
                model_obj.predict_proba,
                num_features=n_features,
                num_samples=num_samples
            )
            vec = lime_to_vec(exp, n_features)
            aciklamalar.append(vec)

        pairs = []
        for a in range(n_tekrar):
            for b in range(a+1, n_tekrar):
                r, _ = spearmanr(aciklamalar[a], aciklamalar[b])
                if not np.isnan(r):
                    pairs.append(r)
        if pairs:
            tum_spearman.append(np.mean(pairs))

        if (i+1) % 10 == 0:
            print(f"    {i+1}/{len(X_samples)} örnek tamamlandı", end='\r')

    return np.array(tum_spearman)

# Veri yükle
with open(os.path.join(EDGE_DIR, "veri.pkl"), 'rb') as f:
    veri = pickle.load(f)
X_edge = veri['X_multi_scaled'].values

with open(os.path.join(CIC_DIR, "veri_cicids.pkl"), 'rb') as f:
    veri_c = pickle.load(f)
X_cic = veri_c['X_scaled']
if hasattr(X_cic, 'values'): X_cic = X_cic.values

modeller_test = ['RandomForest', 'XGBoost', 'LogisticRegression']
N_ORNEK = 30

print("LIME Stochastic Stability Analizi başlıyor...\n")
sonuclar = []

for dataset_adi, X_data, model_dir, model_prefix in [
    ('Edge', X_edge, EDGE_DIR, 'modeller_seed0.pkl'),
    ('CIC',  X_cic,  CIC_DIR,  'modeller_cicids_seed0.pkl'),
]:
    with open(os.path.join(model_dir, model_prefix), 'rb') as f:
        modeller_list = pickle.load(f)

    rng = np.random.default_rng(42)
    idx_sample = rng.choice(len(X_data), size=N_ORNEK, replace=False)
    X_samples  = X_data[idx_sample]
    X_bg       = X_data[:500]

    for model_adi in modeller_test:
        kayit = next((m for m in modeller_list
                      if m['model_adi']==model_adi and m['fold']==0), None)
        if kayit is None:
            print(f"  {model_adi} bulunamadı"); continue

        print(f"  {dataset_adi} — {model_adi}...")
        sp_arr = lime_stochastic(
            kayit['model_obj'], X_bg, X_samples,
            n_features=X_data.shape[1],
            n_tekrar=10, num_samples=1000
        )
        mean_sp = np.mean(sp_arr)
        std_sp  = np.std(sp_arr)
        print(f"    → mean={mean_sp:.4f}  std={std_sp:.4f}  "
              f"min={sp_arr.min():.4f}  max={sp_arr.max():.4f}")
        sonuclar.append({
            'Dataset': dataset_adi, 'Model': model_adi,
            'mean': mean_sp, 'std': std_sp,
            'min': sp_arr.min(), 'max': sp_arr.max()
        })

df_stoch = pd.DataFrame(sonuclar)
with open(os.path.join(EDGE_DIR, "df_lime_stochastic.pkl"), 'wb') as f:
    pickle.dump(df_stoch, f)
print(f"\n✅ Kaydedildi")
print(df_stoch.to_string(index=False))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

SHAP_COL  = '#1565C0'
LIME_COL  = '#E53935'
GORSEL_DIR = str(CICIDS2017_DIR)

def plot_panel(ax, values, feat_names, title, color, top_k=10):
    values = np.array(values).flatten()
    idx    = np.argsort(np.abs(values))[-top_k:][::-1]
    vals   = values[idx]
    names  = [str(feat_names[int(i)])[:32] for i in idx]
    colors = [color if v >= 0 else '#90A4AE' for v in vals]
    ax.barh(range(top_k), np.abs(vals), color=colors,
            edgecolor='white', linewidth=0.5, height=0.7)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(names, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_xlabel('|Importance|', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', labelsize=8)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('white')

plot_panel(axes[0,0], shap_cb, feat_edge_list,
           'Edge-IIoT · CatBoost · SHAP\n(ΔF1@Top-5 = 0.792)', SHAP_COL)
plot_panel(axes[0,1], lime_cb, feat_edge_list,
           'Edge-IIoT · CatBoost · LIME\n(ΔF1@Top-5 = 0.037)', LIME_COL)
plot_panel(axes[1,0], shap_dt, feat_cic_list,
           'CIC-IDS-2017 · DecisionTree · SHAP\n(ΔF1@Top-5 = 0.905)', SHAP_COL)
plot_panel(axes[1,1], lime_dt, feat_cic_list,
           'CIC-IDS-2017 · DecisionTree · LIME\n(ΔF1@Top-5 = 0.842)', LIME_COL)

# Satır başlıkları — subplot dışında, sol kenarda
for ax, label in [(axes[0,0], 'Case A: LIME fails\n(Edge-IIoT)'),
                  (axes[1,0], 'Case B: LIME works\n(CIC-IDS-2017)')]:
    ax.annotate(label, xy=(-0.35, 0.5), xycoords='axes fraction',
                fontsize=9.5, fontweight='bold', color='#37474F',
                va='center', ha='center', rotation=90,
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#ECEFF1',
                          edgecolor='#90A4AE', linewidth=0.8))

# Jaccard badge — her satırın altında subplot içinde
for ax, j, ok in [(axes[0,1], j_cb, False), (axes[1,1], j_dt, True)]:
    clr  = '#B71C1C' if not ok else '#1B5E20'
    bg   = '#FFEBEE' if not ok else '#E8F5E9'
    edge = '#EF9A9A' if not ok else '#A5D6A7'
    txt  = f'Top-5 Jaccard = {j:.2f}  ·  {"Low agreement — LIME selects irrelevant features" if not ok else "High agreement — both methods agree"}'
    ax.text(0.98, 0.03, txt, transform=ax.transAxes,
            fontsize=8, color=clr, ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.35', facecolor=bg,
                      edgecolor=edge, linewidth=0.8))

fig.suptitle('SHAP vs. LIME Feature Attribution: Dataset-Dependent Reliability',
             fontsize=12.5, fontweight='bold', y=1.01)

plt.tight_layout(rect=[0.06, 0, 1, 1])
out_path = os.path.join(GORSEL_DIR, "shap_vs_lime_ornek_karsilastirma.png")
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Görsel güncellendi: {out_path}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

EDGE_DIR  = str(EDGEIIOT_DIR)
CIC_DIR   = str(CICIDS2017_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(EDGE_DIR, "tum_perturb_sonuclar.pkl"), 'rb') as f:
    edge = pickle.load(f)
with open(os.path.join(CIC_DIR, "tum_perturb_sonuclar_cicids.pkl"), 'rb') as f:
    cic = pickle.load(f)

modeller   = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
              'LightGBM','CatBoost','LogisticRegression']
kisaltma   = ['RF','ET','DT*','XGB','LGBM','CB','LR']
pct_keys   = ['pct1','pct3','pct5']
pct_labels = ['1%','3%','5%']
DT_IDX     = 2

SHAP_COLS  = ['#1565C0','#1976D2','#64B5F6']
LIME_COLS  = ['#B71C1C','#E53935','#EF9A9A']
SHAP_MARKS = ['o','s','D']
LIME_MARKS = ['o','s','D']
SHAP_LW    = [2.2, 1.8, 1.4]
LIME_LW    = [2.2, 1.8, 1.4]

def get_mean(data, model, pk, xai):
    arr = np.array(data[model][pk][xai]['spearman'])
    if np.mean(np.isnan(arr)) > 0.5:
        return np.mean(data[model][pk][xai]['jaccard_5']), True
    return np.nanmean(arr), False

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
fig.patch.set_facecolor('white')
x = np.arange(len(modeller))

for ax, (ds_name, data) in zip(axes, [('Edge-IIoT', edge),
                                       ('CIC-IDS-2017', cic)]):
    # ── SHAP ────────────────────────────────────────────────
    shap_all = []
    for pi, (pk, label, col, mk, lw) in enumerate(
            zip(pct_keys, pct_labels, SHAP_COLS, SHAP_MARKS, SHAP_LW)):
        means = [get_mean(data, m, pk, 'shap')[0] for m in modeller]
        shap_all.append(means)
        ax.plot(x, means, color=col, marker=mk, markersize=6.5,
                linewidth=lw, label=f'SHAP {label}', zorder=4,
                markerfacecolor='white', markeredgewidth=1.8,
                markeredgecolor=col)
    ax.fill_between(x, shap_all[0], shap_all[2],
                    color='#BBDEFB', alpha=0.20, zorder=1)

    # ── LIME ────────────────────────────────────────────────
    lime_all = []
    for pi, (pk, label, col, mk, lw) in enumerate(
            zip(pct_keys, pct_labels, LIME_COLS, LIME_MARKS, LIME_LW)):
        means, jac_flags = [], []
        for model in modeller:
            m, is_jac = get_mean(data, model, pk, 'lime')
            means.append(m)
            jac_flags.append(is_jac)
        lime_all.append(means)

        left_x  = x[:DT_IDX];    left_y  = means[:DT_IDX]
        right_x = x[DT_IDX+1:];  right_y = means[DT_IDX+1:]
        kw = dict(color=col, marker=mk, markersize=6.5,
                  linewidth=lw, linestyle='--', zorder=4,
                  markerfacecolor='white', markeredgewidth=1.8,
                  markeredgecolor=col)
        lbl = f'LIME {label}'
        if len(left_x):
            ax.plot(left_x, left_y, **kw, label=lbl)
        if len(right_x):
            ax.plot(right_x, right_y, **kw, label='_')
        if jac_flags[DT_IDX]:
            ax.scatter([DT_IDX], [means[DT_IDX]], color=col,
                       marker='*', s=150, zorder=6,
                       edgecolors='white', linewidths=0.5)

    ax.fill_between(x, lime_all[0], lime_all[2],
                    color='#FFCDD2', alpha=0.18, zorder=1)

    # ── Threshold ───────────────────────────────────────────
    ax.axhline(0.9, color='#90A4AE', lw=1.0, ls=':', zorder=2)
    ax.text(6.45, 0.902, '0.90', fontsize=8,
            color='#78909C', va='bottom', ha='right')

    # ── Δ annotation ────────────────────────────────────────
    shap_lr = shap_all[2][-1]
    lime_lr = lime_all[2][-1]
    ax.annotate('', xy=(6.38, lime_lr), xytext=(6.38, shap_lr),
                arrowprops=dict(arrowstyle='<->',color='#37474F',
                                lw=1.3, mutation_scale=12))
    ax.text(6.50, (shap_lr + lime_lr) / 2,
            f'Δ = {shap_lr - lime_lr:.2f}',
            fontsize=8.5, color='#263238', va='center', ha='left')

    # ── DT* notu — sol alt köşe ─────────────────────────────
    ax.text(0.02, 0.04,
            '★ DT*: Spearman undefined (near-constant LIME output)\n'
            '   Jaccard@5 shown instead. LIME line broken at DT.',
            transform=ax.transAxes, fontsize=7.5,
            color='#616161', style='italic',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='#F5F5F5',
                      edgecolor='#CFD8DC', linewidth=0.6))

    ax.set_xticks(x)
    ax.set_xticklabels(kisaltma, fontsize=11)
    ax.set_ylabel('Spearman r', fontsize=11)
    ax.set_ylim(-0.02, 1.13)
    ax.set_xlim(-0.5, 6.85)
    ax.set_title(ds_name, fontsize=12.5, fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=9.5)

# ── Legend ──────────────────────────────────────────────────
shap_h = [Line2D([0],[0], color=c, marker=m, markersize=6,
                  linewidth=2, markerfacecolor='white',
                  markeredgecolor=c, markeredgewidth=1.8,
                  label=f'SHAP {l}')
          for c,m,l in zip(SHAP_COLS, SHAP_MARKS, pct_labels)]
lime_h = [Line2D([0],[0], color=c, marker=m, markersize=6,
                  linewidth=2, linestyle='--',
                  markerfacecolor='white',
                  markeredgecolor=c, markeredgewidth=1.8,
                  label=f'LIME {l}')
          for c,m,l in zip(LIME_COLS, LIME_MARKS, pct_labels)]
shap_band = mpatches.Patch(color='#BBDEFB', alpha=0.6,
                            label='SHAP range (1%–5%)')
lime_band = mpatches.Patch(color='#FFCDD2', alpha=0.6,
                            label='LIME range (1%–5%)')

fig.legend(handles=shap_h + lime_h + [shap_band, lime_band],
           loc='lower center', ncol=8, fontsize=9,
           frameon=True, framealpha=0.93,
           edgecolor='#CFD8DC', bbox_to_anchor=(0.5, -0.09))

fig.suptitle(
    'RQ1: Explanation Stability Under Input Perturbation\n'
    'SHAP vs. LIME — Spearman Correlation across Noise Levels'
    ' (n = 500 samples/model)',
    fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0.06, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig1_rq1_perturbation_stability.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

EDGE_DIR  = str(EDGEIIOT_DIR)
CIC_DIR   = str(CICIDS2017_DIR)
DNN_DIR   = str(DNN_LSTM_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(EDGE_DIR, "df_var.pkl"),        'rb') as f: df_edge = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_var_cicids.pkl"), 'rb') as f: df_cic  = pickle.load(f)

rq2 = {}
for k, fname in [('DNN_Edge','rq2_dnn_edge_ozet.pkl'),
                 ('LSTM_Edge','rq2_lstm_edge_ozet.pkl'),
                 ('DNN_CIC', 'rq2_dnn_cic_ozet.pkl'),
                 ('LSTM_CIC','rq2_lstm_cic_ozet.pkl')]:
    with open(os.path.join(DNN_DIR, fname), 'rb') as f:
        rq2[k] = pickle.load(f)

modeller9  = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
              'LightGBM','CatBoost','LogisticRegression','DNN','LSTM']
kisaltma9  = ['RF','ET','DT','XGB','LGBM','CB','LR','DNN','LSTM']
renkler    = {'RandomForest':'#2196F3','ExtraTrees':'#4CAF50',
              'DecisionTree':'#FF9800','XGBoost':'#F44336',
              'LightGBM':'#9C27B0','CatBoost':'#00BCD4',
              'LogisticRegression':'#795548','DNN':'#E91E63','LSTM':'#FF6F00'}

def get_stats(df, model, metric):
    sub = df[df['model_adi'] == model][metric]
    return sub.mean(), sub.std()

def build_data(df_klasik, rq2_dnn, rq2_lstm, metric):
    means, stds = [], []
    for m in modeller9:
        if m == 'DNN':
            means.append(rq2_dnn[metric]); stds.append(0)
        elif m == 'LSTM':
            means.append(rq2_lstm[metric]); stds.append(0)
        else:
            mn, sd = get_stats(df_klasik, m, metric)
            means.append(mn); stds.append(sd)
    return np.array(means), np.array(stds)

# ── Data ────────────────────────────────────────────────────
sp_e_m,  sp_e_s  = build_data(df_edge, rq2['DNN_Edge'], rq2['LSTM_Edge'], 'spearman')
sp_c_m,  sp_c_s  = build_data(df_cic,  rq2['DNN_CIC'],  rq2['LSTM_CIC'],  'spearman')
j5_e_m,  j5_e_s  = build_data(df_edge, rq2['DNN_Edge'], rq2['LSTM_Edge'], 'jaccard_5')
j5_c_m,  j5_c_s  = build_data(df_cic,  rq2['DNN_CIC'],  rq2['LSTM_CIC'],  'jaccard_5')
j10_e_m, j10_e_s = build_data(df_edge, rq2['DNN_Edge'], rq2['LSTM_Edge'], 'jaccard_10')
j10_c_m, j10_c_s = build_data(df_cic,  rq2['DNN_CIC'],  rq2['LSTM_CIC'],  'jaccard_10')

# ── Figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.patch.set_facecolor('white')

x   = np.arange(len(modeller9))
bw  = 0.35
col_edge = [renkler[m] for m in modeller9]
col_cic  = [renkler[m] for m in modeller9]

configs = [
    (axes[0], 'Spearman r  (SHAP consistency)',
     sp_e_m,  sp_e_s,  sp_c_m,  sp_c_s,  0.55, 1.04),
    (axes[1], 'Jaccard Similarity @ Top-5',
     j5_e_m,  j5_e_s,  j5_c_m,  j5_c_s,  0.0,  1.04),
    (axes[2], 'Jaccard Similarity @ Top-10',
     j10_e_m, j10_e_s, j10_c_m, j10_c_s, 0.0,  1.04),
]

for ax, ylabel, em, es, cm, cs, ylo, yhi in configs:
    for i in range(len(modeller9)):
        # Edge bar (düz)
        ax.bar(x[i]-bw/2, em[i], bw,
               color=col_edge[i], alpha=0.90,
               yerr=es[i] if es[i]>0 else None,
               error_kw=dict(elinewidth=0.9, ecolor='#333',
                             capsize=3, capthick=0.9),
               zorder=3)
        # CIC bar (taralı)
        ax.bar(x[i]+bw/2, cm[i], bw,
               color=col_cic[i], alpha=0.55,
               hatch='///', edgecolor=col_cic[i],
               yerr=cs[i] if cs[i]>0 else None,
               error_kw=dict(elinewidth=0.9, ecolor='#333',
                             capsize=3, capthick=0.9),
               zorder=3)

    ax.axhline(0.9, color='#90A4AE', lw=1.0, ls=':', zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(kisaltma9, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10.5)
    ax.set_ylim(ylo, yhi)
    ax.set_title(ylabel, fontsize=11, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, alpha=0.25, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=9)

    # DNN/LSTM bölge arkaplanı
    ax.axvspan(6.5, 8.5, color='#F3E5F5', alpha=0.25, zorder=0)
    ax.text(7.5, yhi*0.97, 'DL', fontsize=8, color='#7B1FA2',
            ha='center', va='top', style='italic')

# Model renk legend
model_handles = [mpatches.Patch(color=renkler[m], label=kisaltma9[i])
                 for i, m in enumerate(modeller9)]
edge_h = mpatches.Patch(color='#616161', alpha=0.90,
                         label='Edge-IIoT (solid)')
cic_h  = mpatches.Patch(color='#616161', alpha=0.55,
                         hatch='///', label='CIC-IDS-2017 (hatched)')

fig.legend(handles=model_handles + [edge_h, cic_h],
           loc='lower center', ncol=11, fontsize=9,
           frameon=True, framealpha=0.93,
           edgecolor='#CFD8DC', bbox_to_anchor=(0.5, -0.09))

fig.suptitle(
    'RQ2: Explanation Consistency Across Model Variations\n'
    'Pairwise SHAP Stability — 50 models × 100 samples → 1,225 pairs/algorithm',
    fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0.06, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig2_rq2_model_variation.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

EDGE_DIR  = str(EDGEIIOT_DIR)
CIC_DIR   = str(CICIDS2017_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(EDGE_DIR, "df_var.pkl"),        'rb') as f: df_edge = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_rq3_cicids.pkl"), 'rb') as f: df_cic  = pickle.load(f)

modeller = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
            'LightGBM','CatBoost','LogisticRegression']
kisaltma = {'RandomForest':'RF','ExtraTrees':'ET','DecisionTree':'DT',
            'XGBoost':'XGB','LightGBM':'LGBM','CatBoost':'CB',
            'LogisticRegression':'LR'}
renkler  = {'RandomForest':'#2196F3','ExtraTrees':'#4CAF50',
            'DecisionTree':'#FF9800','XGBoost':'#F44336',
            'LightGBM':'#9C27B0','CatBoost':'#00BCD4',
            'LogisticRegression':'#795548'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
fig.patch.set_facecolor('white')

for ax, (ds_name, df) in zip(axes, [('Edge-IIoT', df_edge),
                                     ('CIC-IDS-2017', df_cic)]):
    # Her model scatter
    for model in modeller:
        sub = df[df['model_adi'] == model]
        f1  = sub['f1_macro'].values
        sp  = sub['spearman'].values
        mask = ~np.isnan(f1) & ~np.isnan(sp)

        is_lr = (model == 'LogisticRegression')
        ax.scatter(f1[mask], sp[mask],
                   color=renkler[model],
                   s=28 if not is_lr else 45,
                   alpha=0.75 if not is_lr else 0.95,
                   label=kisaltma[model],
                   zorder=4 if not is_lr else 5,
                   edgecolors='white' if not is_lr else renkler[model],
                   linewidths=0.3,
                   marker='o' if not is_lr else 'D')

    f1_all = df['f1_macro'].values
    sp_all = df['spearman'].values
    mask_all = ~np.isnan(f1_all) & ~np.isnan(sp_all)

    # Global trend (tüm modeller)
    r_all, p_all = pearsonr(f1_all[mask_all], sp_all[mask_all])
    z = np.polyfit(f1_all[mask_all], sp_all[mask_all], 1)
    p_fn = np.poly1d(z)
    x_line = np.linspace(f1_all[mask_all].min(),
                          f1_all[mask_all].max(), 100)
    ax.plot(x_line, p_fn(x_line), color='#263238',
            lw=2.0, ls='--', zorder=6,
            label=f'All models: r = {r_all:+.3f}')

    # LR hariç trend
    df_nolr = df[df['model_adi'] != 'LogisticRegression']
    f1_nolr = df_nolr['f1_macro'].values
    sp_nolr = df_nolr['spearman'].values
    mask_nolr = ~np.isnan(f1_nolr) & ~np.isnan(sp_nolr)
    r_nolr, _ = pearsonr(f1_nolr[mask_nolr], sp_nolr[mask_nolr])
    z2 = np.polyfit(f1_nolr[mask_nolr], sp_nolr[mask_nolr], 1)
    p2 = np.poly1d(z2)
    x2 = np.linspace(f1_nolr[mask_nolr].min(),
                      f1_nolr[mask_nolr].max(), 100)
    ax.plot(x2, p2(x2), color='#1565C0',
            lw=2.0, ls='-', zorder=6,
            label=f'w/o LR: r = {r_nolr:+.3f}')

    # LR bölge vurgusu
    lr_sub = df[df['model_adi'] == 'LogisticRegression']
    lr_f1 = lr_sub['f1_macro'].values
    lr_sp = lr_sub['spearman'].values
    mask_lr = ~np.isnan(lr_f1) & ~np.isnan(lr_sp)
    ax.annotate('LR outlier\n(Simpson\'s Paradox driver)',
                xy=(lr_f1[mask_lr].mean(), lr_sp[mask_lr].mean()),
                xytext=(lr_f1[mask_lr].mean() + 0.015,
                        lr_sp[mask_lr].mean() - 0.06),
                fontsize=8, color='#795548',
                arrowprops=dict(arrowstyle='->', color='#795548',
                                lw=1.2),
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='#EFEBE9',
                          edgecolor='#A1887F', linewidth=0.8))

    ax.set_xlabel('F1-macro', fontsize=11)
    ax.set_ylabel('SHAP Spearman r (RQ2)', fontsize=11)
    ax.set_title(ds_name, fontsize=12.5, fontweight='bold', pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, alpha=0.22, zorder=0)
    ax.xaxis.grid(True, alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=9.5)

    # Trend legend
    trend_handles = [
        Line2D([0],[0], color='#263238', lw=2, ls='--',
               label=f'All models: r = {r_all:+.3f} (p < 0.001)'),
        Line2D([0],[0], color='#1565C0', lw=2, ls='-',
               label=f'Tree models only (w/o LR): r = {r_nolr:+.3f} (p < 0.001)'),
    ]
    ax.legend(handles=trend_handles, fontsize=8.5,
              loc='upper left', frameon=True,
              framealpha=0.92, edgecolor='#CFD8DC')

# Model renk legend (alta)
model_handles = [mpatches.Patch(color=renkler[m],
                                 label=kisaltma[m])
                 for m in modeller]
fig.legend(handles=model_handles, loc='lower center',
           ncol=7, fontsize=9.5, frameon=True,
           framealpha=0.93, edgecolor='#CFD8DC',
           bbox_to_anchor=(0.5, -0.07))

fig.suptitle(
    "RQ3: Relationship Between Classification Performance and Explanation Consistency\n"
    "Simpson's Paradox: LR drives sign reversal — tree models show consistent positive correlation",
    fontsize=12.5, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0.06, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig3_rq3_simpsons_paradox.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

EDGE_DIR  = str(EDGEIIOT_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(EDGE_DIR, "df_deletion_curve_full.pkl"), 'rb') as f:
    df = pickle.load(f)

# Temsili 6 model seç — hikayeyi en iyi anlatan
modeller_sec = ['XGBoost','LightGBM','CatBoost',
                'DecisionTree','RandomForest','LogisticRegression']
kisaltma = {'XGBoost':'XGB','LightGBM':'LGBM','CatBoost':'CB',
            'DecisionTree':'DT','RandomForest':'RF',
            'LogisticRegression':'LR'}
renkler  = {'XGBoost':'#F44336','LightGBM':'#9C27B0','CatBoost':'#00BCD4',
            'DecisionTree':'#FF9800','RandomForest':'#2196F3',
            'LogisticRegression':'#795548'}

k_vals = list(range(1, 21))

XAI_STYLE = {
    'SHAP':   dict(ls='-',  lw=2.2, alpha=0.95),
    'LIME':   dict(ls='--', lw=1.8, alpha=0.85),
    'Random': dict(ls=':',  lw=1.4, alpha=0.70),
    'Least':  dict(ls='-.', lw=1.2, alpha=0.60),
}
XAI_COLS = {
    'SHAP':   None,   # model rengi kullan
    'LIME':   None,
    'Random': '#78909C',
    'Least':  '#B0BEC5',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('white')

for ax, dataset in zip(axes, ['Edge', 'CIC']):
    sub = df[df['dataset'] == dataset]

    for model in modeller_sec:
        col = renkler[model]
        ms  = sub[sub['model'] == model]

        for xai, style in XAI_STYLE.items():
            ms_xai = ms[ms['xai'] == xai].sort_values('k')
            if len(ms_xai) == 0:
                continue
            c = col if XAI_COLS[xai] is None else XAI_COLS[xai]
            lbl = f'{kisaltma[model]} – {xai}' if xai == 'SHAP' else '_'
            ax.plot(ms_xai['k'], ms_xai['delta_f1'],
                    color=c, label=lbl, **style, zorder=4)

    # Random ve Least ortalama bandı (tüm modeller)
    for xai, face_col in [('Random','#ECEFF1'), ('Least','#F5F5F5')]:
        vals = []
        for k in k_vals:
            kv = sub[(sub['xai']==xai) & (sub['k']==k)]['delta_f1']
            vals.append(kv.mean() if len(kv) else np.nan)
        vals = np.array(vals)
        ax.fill_between(k_vals, 0, vals,
                        color=face_col, alpha=0.5, zorder=1)

    # k=5 dikey referans
    ax.axvline(5, color='#37474F', lw=1.0, ls=':', zorder=3, alpha=0.6)
    ax.text(5.2, 0.97, 'k = 5', fontsize=8,
            color='#37474F', va='top', transform=ax.get_xaxis_transform())

    ds_label = 'Edge-IIoT' if dataset == 'Edge' else 'CIC-IDS-2017'
    ax.set_title(ds_label, fontsize=12.5, fontweight='bold', pad=10)
    ax.set_xlabel('Number of masked features (k)', fontsize=11)
    ax.set_ylabel('ΔF1-macro (↑ = more important features)', fontsize=11)
    ax.set_xlim(1, 20)
    ax.set_ylim(-0.02, 1.05)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=9.5)
    ax.set_xticks([1,3,5,7,10,13,15,18,20])

# ── Legend ──────────────────────────────────────────────────
model_handles = [Line2D([0],[0], color=renkler[m], lw=2.5,
                         label=kisaltma[m])
                 for m in modeller_sec]

style_handles = [
    Line2D([0],[0], color='#555', lw=2.2, ls='-',  label='SHAP (top-k)'),
    Line2D([0],[0], color='#555', lw=1.8, ls='--', label='LIME (top-k)'),
    Line2D([0],[0], color='#78909C', lw=1.4, ls=':', label='Random baseline'),
    Line2D([0],[0], color='#B0BEC5', lw=1.2, ls='-.', label='Least-important baseline'),
]

sep = mpatches.Patch(color='none', label='  ')
fig.legend(handles=model_handles + [sep] + style_handles,
           loc='lower center', ncol=11, fontsize=9,
           frameon=True, framealpha=0.93,
           edgecolor='#CFD8DC', bbox_to_anchor=(0.5, -0.09))

fig.suptitle(
    'RQ1–Fidelity: Feature Importance Deletion Curve\n'
    'ΔF1 when masking top-k SHAP / LIME features vs. random and least-important baselines',
    fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0.06, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig4_fidelity_deletion_curve.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

EDGE_DIR  = str(EDGEIIOT_DIR)
CIC_DIR   = str(CICIDS2017_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(EDGE_DIR, "df_istat_final.pkl"),        'rb') as f: df_e = pickle.load(f)
with open(os.path.join(CIC_DIR,  "df_istat_final_cicids.pkl"), 'rb') as f: df_c = pickle.load(f)

modeller = ['RandomForest','ExtraTrees','DecisionTree','XGBoost',
            'LightGBM','CatBoost','LogisticRegression']
kisaltma = ['RF','ET','DT*','XGB','LGBM','CB','LR']
renkler  = {'RandomForest':'#2196F3','ExtraTrees':'#4CAF50',
            'DecisionTree':'#FF9800','XGBoost':'#F44336',
            'LightGBM':'#9C27B0','CatBoost':'#00BCD4',
            'LogisticRegression':'#795548'}
DT_IDX = 2

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')
gs  = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.28)

# Üst satır: Cohen's d — Alt satır: SHAP vs LIME Spearman
ax_cd_e  = fig.add_subplot(gs[0, 0])
ax_cd_c  = fig.add_subplot(gs[0, 1])
ax_sp_e  = fig.add_subplot(gs[1, 0])
ax_sp_c  = fig.add_subplot(gs[1, 1])

x = np.arange(len(modeller))

for (ax_cd, ax_sp), (ds_name, df) in zip(
        [(ax_cd_e, ax_sp_e), (ax_cd_c, ax_sp_c)],
        [('Edge-IIoT', df_e), ('CIC-IDS-2017', df_c)]):

    df = df.set_index('Model') if 'Model' in df.columns else df

    cd_vals, shap_m, lime_m, lime_is_jac = [], [], [], []
    for m in modeller:
        row = df.loc[m] if m in df.index else None
        cd_vals.append(row['Cohen_d'] if row is not None else 0)
        shap_m.append(row['SHAP_mean'] if row is not None else 0)
        is_jac = (m == 'DecisionTree' and ds_name == 'Edge-IIoT')
        lime_is_jac.append(is_jac)
        if row is not None and not np.isnan(row['LIME_mean']):
            lime_m.append(row['LIME_mean'])
        else:
            lime_m.append(np.nan)

    # ── Üst panel: Cohen's d ─────────────────────────────────
    bars = ax_cd.bar(x, cd_vals, 0.58,
                     color=[renkler[m] for m in modeller],
                     alpha=0.88, zorder=3)

    for bar, val in zip(bars, cd_vals):
        ax_cd.text(bar.get_x() + bar.get_width()/2,
                   bar.get_height() + 0.25,
                   f'{val:.1f}', ha='center', va='bottom',
                   fontsize=9, fontweight='500', color='#263238')

    # Effect size referans çizgileri
    for thresh, label, col in [(0.8,'small','#B0BEC5'),
                                (2.0,'medium','#90A4AE'),
                                (5.0,'large','#78909C')]:
        ax_cd.axhline(thresh, color=col, lw=0.9, ls=':', zorder=2)
        ax_cd.text(-0.48, thresh + 0.2, label,
                   fontsize=7.5, color=col, va='bottom')

    # RB-r badge
    ax_cd.text(0.98, 0.97,
               'RB-r = 1.00  (all models)\np < 0.001, Bonferroni corrected',
               transform=ax_cd.transAxes, fontsize=8, ha='right', va='top',
               color='#1B5E20',
               bbox=dict(boxstyle='round,pad=0.35', facecolor='#E8F5E9',
                         edgecolor='#A5D6A7', linewidth=0.8))

    ax_cd.set_xticks(x)
    ax_cd.set_xticklabels(kisaltma, fontsize=10.5)
    ax_cd.set_ylabel("Cohen's d  (SHAP − LIME)", fontsize=10.5)
    ax_cd.set_ylim(0, max(cd_vals) * 1.20)
    ax_cd.set_xlim(-0.55, 6.55)
    ax_cd.set_title(f"{ds_name} — Effect Size (Cohen's d)",
                    fontsize=11.5, fontweight='bold', pad=8)
    ax_cd.spines['top'].set_visible(False)
    ax_cd.spines['right'].set_visible(False)
    ax_cd.yaxis.grid(True, alpha=0.22, zorder=0)
    ax_cd.set_axisbelow(True)
    ax_cd.tick_params(axis='both', labelsize=9.5)

    # ── Alt panel: SHAP vs LIME mean Spearman ────────────────
    # SHAP — tam çizgi
    ax_sp.plot(x, shap_m, color='#1565C0', marker='o', markersize=7,
               linewidth=2.2, zorder=4, label='SHAP',
               markerfacecolor='white', markeredgewidth=2.0,
               markeredgecolor='#1565C0')

    # LIME — DT'yi atlayarak çiz
    left_x  = x[:DT_IDX];    left_y  = [lime_m[i] for i in range(DT_IDX)]
    right_x = x[DT_IDX+1:];  right_y = [lime_m[i] for i in range(DT_IDX+1, len(modeller))]
    kw_lime = dict(color='#B71C1C', marker='s', markersize=7,
                   linewidth=2.0, linestyle='--', zorder=4,
                   markerfacecolor='white', markeredgewidth=2.0,
                   markeredgecolor='#B71C1C')
    if len(left_x):
        ax_sp.plot(left_x, left_y, **kw_lime, label='LIME')
    if len(right_x):
        ax_sp.plot(right_x, right_y, **kw_lime, label='_')

    # DT Jaccard noktası
    if lime_is_jac[DT_IDX]:
        jac_val = df.loc['DecisionTree']['LIME_mean'] \
                  if not np.isnan(df.loc['DecisionTree']['LIME_mean']) \
                  else 0.916
        ax_sp.scatter([DT_IDX], [jac_val], color='#B71C1C',
                      marker='*', s=160, zorder=6,
                      edgecolors='white', linewidths=0.5)
        ax_sp.text(DT_IDX, jac_val + 0.03, 'J@5*',
                   fontsize=7.5, ha='center', color='#B71C1C',
                   style='italic')

    # Shaded bands
    ax_sp.fill_between(x, shap_m, 1.0, color='#BBDEFB', alpha=0.12, zorder=1)
    valid_lime = [v for v in lime_m if not np.isnan(v)]
    if valid_lime:
        ax_sp.fill_between(
            [i for i, v in enumerate(lime_m) if not np.isnan(v)],
            [v for v in lime_m if not np.isnan(v)],
            0, color='#FFCDD2', alpha=0.15, zorder=1)

    # 0.9 eşiği
    ax_sp.axhline(0.9, color='#90A4AE', lw=1.0, ls=':', zorder=2)
    ax_sp.text(6.5, 0.912, '0.90', fontsize=8,
               color='#78909C', ha='right', va='bottom')

    ax_sp.set_xticks(x)
    ax_sp.set_xticklabels(kisaltma, fontsize=10.5)
    ax_sp.set_ylabel('Mean Spearman r  (@5% noise)', fontsize=10.5)
    ax_sp.set_ylim(-0.02, 1.10)
    ax_sp.set_xlim(-0.55, 6.55)
    ax_sp.set_title(f'{ds_name} — Mean Spearman r (SHAP vs. LIME)',
                    fontsize=11.5, fontweight='bold', pad=8)
    ax_sp.spines['top'].set_visible(False)
    ax_sp.spines['right'].set_visible(False)
    ax_sp.yaxis.grid(True, alpha=0.22, zorder=0)
    ax_sp.set_axisbelow(True)
    ax_sp.tick_params(axis='both', labelsize=9.5)
    ax_sp.legend(fontsize=9.5, loc='lower right',
                 frameon=True, framealpha=0.92, edgecolor='#CFD8DC')

    # DT notu alt panel
    ax_sp.text(0.02, 0.04,
               '★ DT*: Spearman NaN → Jaccard@5 (Edge only)',
               transform=ax_sp.transAxes, fontsize=7.5,
               color='#616161', style='italic',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='#F5F5F5',
                         edgecolor='#CFD8DC', linewidth=0.6))

# Ortak model renk legend
model_handles = [mpatches.Patch(color=renkler[m], alpha=0.88,
                                 label=k)
                 for m, k in zip(modeller, kisaltma)]
fig.legend(handles=model_handles, loc='lower center', ncol=7,
           fontsize=9.5, frameon=True, framealpha=0.93,
           edgecolor='#CFD8DC', bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    "Statistical Analysis: SHAP vs. LIME Explanation Stability\n"
    "Cohen's d effect size and mean Spearman correlation — "
    "Wilcoxon signed-rank test, Bonferroni corrected (n = 500 samples/model)",
    fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout(rect=[0, 0.04, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig5_statistical_analysis.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

EDGE_DIR  = str(EDGEIIOT_DIR)
CIC_DIR   = str(CICIDS2017_DIR)
DNN_DIR   = str(DNN_LSTM_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(DNN_DIR, "df_xai_score_lstm_edge.pkl"), 'rb') as f: df_e9 = pickle.load(f)
with open(os.path.join(DNN_DIR, "df_xai_score_lstm_cic.pkl"),  'rb') as f: df_c9 = pickle.load(f)

modeller9 = ['XGBoost','LightGBM','DecisionTree','CatBoost',
             'RandomForest','ExtraTrees','LogisticRegression','DNN','LSTM']
kisaltma9 = ['XGB','LGBM','DT','CB','RF','ET','LR','DNN','LSTM']
renkler   = {'XGBoost':'#F44336','LightGBM':'#9C27B0','DecisionTree':'#FF9800',
             'CatBoost':'#00BCD4','RandomForest':'#2196F3','ExtraTrees':'#4CAF50',
             'LogisticRegression':'#795548','DNN':'#E91E63','LSTM':'#FF6F00'}

def normalize(vals, inverse=False):
    mn, mx = vals.min(), vals.max()
    if mx == mn: return np.full_like(vals, 0.5, dtype=float)
    n = (vals - mn) / (mx - mn)
    return 1 - n if inverse else n

def compute_scores(df, wp, wv, wf, ws):
    d = df.copy()
    fcol = 'Fidelity_d5' if 'Fidelity_d5' in d.columns else 'Fidelity_5'
    d['nP'] = normalize(d['Perturb'].values)
    d['nV'] = normalize(d['Varyans'].values)
    d['nF'] = normalize(d[fcol].values)
    d['nS'] = normalize(d['Sure_ms'].values, inverse=True)
    d['Score'] = wp*d['nP'] + wv*d['nV'] + wf*d['nF'] + ws*d['nS']
    return d.set_index('Model')['Score']

senaryolar = {
    'S1: Equal (25/25/25/25)':     (0.25,0.25,0.25,0.25),
    'S2: No runtime (33/33/34/0)': (0.33,0.33,0.34,0.00),
    'S3: Stability (40/30/20/10)': (0.40,0.30,0.20,0.10),
    'S4: Fidelity (20/20/40/20)':  (0.20,0.20,0.40,0.20),
    'S5: Reliability (50/50/0/0)': (0.50,0.50,0.00,0.00),
}

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')
gs  = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

ax_bar_e  = fig.add_subplot(gs[0, 0])
ax_bar_c  = fig.add_subplot(gs[0, 1])
ax_sens_e = fig.add_subplot(gs[1, 0])
ax_sens_c = fig.add_subplot(gs[1, 1])

# ── Üst: XAI-Score bar ──────────────────────────────────────
for ax, df9, ds_name in [(ax_bar_e, df_e9, 'Edge-IIoT'),
                          (ax_bar_c, df_c9, 'CIC-IDS-2017')]:
    scores_s1 = compute_scores(df9, 0.25, 0.25, 0.25, 0.25)
    order = [m for m in modeller9 if m in scores_s1.index]
    vals  = [scores_s1[m] for m in order]
    kisa  = [kisaltma9[modeller9.index(m)] for m in order]
    cols  = [renkler[m] for m in order]

    # S1 skoruna göre sırala (yüksekten düşüğe)
    sidx   = np.argsort(vals)[::-1]
    order_s = [order[i] for i in sidx]
    vals_s  = [vals[i]  for i in sidx]
    kisa_s  = [kisa[i]  for i in sidx]
    cols_s  = [cols[i]  for i in sidx]

    x = np.arange(len(order_s))
    bars = ax.bar(x, vals_s, 0.58,
                  color=cols_s, alpha=0.88, zorder=3)

    for bar, val, model in zip(bars, vals_s, order_s):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.012,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=8.5, fontweight='500', color='#263238')
        # DNN/LSTM etiketlerini italik yap
        if model in ['DNN','LSTM']:
            ax.get_xticklabels()  # sonra düzelteceğiz

    # DL modellere ince dikey ayırıcı — geniş kutu yerine
    dl_positions = [i for i,m in enumerate(order_s) if m in ['DNN','LSTM']]
    if dl_positions:
        sep_x = min(dl_positions) - 0.5
        ax.axvline(sep_x, color='#CE93D8', lw=1.2,
                   ls='--', zorder=2, alpha=0.7)
        ax.text(sep_x + 0.05, max(vals_s)*1.10,
                '← Classical  |  DL →',
                fontsize=7.5, color='#7B1FA2', va='top')

    ax.axhline(0.5, color='#90A4AE', lw=1.0, ls=':', zorder=2)
    ax.text(len(order_s)-0.05, 0.515, '0.5 baseline',
            fontsize=7.5, color='#78909C', ha='right')

    ax.set_xticks(x)
    xlabels = ax.set_xticklabels(kisa_s, fontsize=10.5)
    # DNN/LSTM italik
    for lbl, model in zip(xlabels, order_s):
        if model in ['DNN','LSTM']:
            lbl.set_style('italic')
            lbl.set_color('#7B1FA2')

    ax.set_ylabel('XAI-Score  [0–1]', fontsize=11)
    ax.set_ylim(0, max(vals_s) * 1.20)
    ax.set_xlim(-0.55, len(order_s)-0.45)
    ax.set_title(f'{ds_name} — XAI-Score Ranking  (S1: Equal weights)',
                 fontsize=11.5, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', labelsize=9.5)

# ── Alt: Sensitivity heatmap ────────────────────────────────
for ax, df9, ds_name in [(ax_sens_e, df_e9, 'Edge-IIoT'),
                          (ax_sens_c, df_c9, 'CIC-IDS-2017')]:

    scores_s1 = compute_scores(df9, 0.25, 0.25, 0.25, 0.25)
    order = [m for m in modeller9 if m in scores_s1.index]
    # S1 sıralamasına göre model sırası
    s1_rank = scores_s1[order].rank(ascending=False).astype(int)
    col_order = np.argsort([s1_rank[m] for m in order])
    order_sorted = [order[i] for i in col_order]
    kisa_sorted  = [kisaltma9[modeller9.index(m)] for m in order_sorted]

    sira_matrix = []
    s_labels = list(senaryolar.keys())
    for s_name, (wp,wv,wf,ws) in senaryolar.items():
        sc = compute_scores(df9, wp, wv, wf, ws)
        ranked = sc[order_sorted].rank(ascending=False).astype(int)
        sira_matrix.append([ranked[m] for m in order_sorted])
    sira_matrix = np.array(sira_matrix)

    im = ax.imshow(sira_matrix, cmap='RdYlGn_r',
                   aspect='auto', vmin=1, vmax=len(order))

    for r in range(len(s_labels)):
        for c in range(len(kisa_sorted)):
            val = sira_matrix[r, c]
            txt_col = 'white' if val >= 7 else \
                      ('white' if val == 1 else '#263238')
            ax.text(c, r, str(val), ha='center', va='center',
                    fontsize=10.5, fontweight='600', color=txt_col)

    # DL ayırıcı çizgi
    dl_cols = [i for i,m in enumerate(order_sorted) if m in ['DNN','LSTM']]
    if dl_cols:
        ax.axvline(min(dl_cols)-0.5, color='#CE93D8',
                   lw=1.5, ls='--', alpha=0.8)

    ax.set_xticks(range(len(kisa_sorted)))
    xlbls = ax.set_xticklabels(kisa_sorted, fontsize=10.5)
    for lbl, m in zip(xlbls, order_sorted):
        if m in ['DNN','LSTM']:
            lbl.set_style('italic')
            lbl.set_color('#7B1FA2')

    ax.set_yticks(range(len(s_labels)))
    ax.set_yticklabels(s_labels, fontsize=9)
    ax.set_title(f'{ds_name} — Ranking Sensitivity  (1 = best)',
                 fontsize=11.5, fontweight='bold', pad=8)
    ax.tick_params(axis='both', labelsize=9.5)

    cbar = plt.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Rank', fontsize=8.5)
    cbar.ax.tick_params(labelsize=8)

fig.suptitle(
    'XAI-Score: Multi-Dimensional Explainability Reliability Framework\n'
    'Top: Equal-weight ranking  ·  Bottom: Sensitivity across 5 weighting scenarios',
    fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout(rect=[0, 0.01, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig6_xai_score_sensitivity.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

CIC_DIR   = str(CICIDS2017_DIR)
DNN_DIR   = str(DNN_LSTM_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(DNN_DIR, "df_fgsm_edge.pkl"), 'rb') as f: df_e = pickle.load(f)
with open(os.path.join(DNN_DIR, "df_fgsm_cic.pkl"),  'rb') as f: df_c = pickle.load(f)

print("Sütunlar:", df_e.columns.tolist())
print(df_e.head(4))

In [ ]:
import pickle, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

DNN_DIR   = str(DNN_LSTM_DIR)
CIKTI_DIR = str(FIGURES_DIR)

with open(os.path.join(DNN_DIR, "df_fgsm_edge.pkl"), 'rb') as f: df_e = pickle.load(f)
with open(os.path.join(DNN_DIR, "df_fgsm_cic.pkl"),  'rb') as f: df_c = pickle.load(f)

modeller9 = ['DecisionTree','CatBoost','XGBoost','RandomForest',
             'ExtraTrees','LightGBM','LogisticRegression','DNN','LSTM']
kisaltma9 = ['DT','CB','XGB','RF','ET','LGBM','LR','DNN','LSTM']
renkler   = {'DecisionTree':'#FF9800','CatBoost':'#00BCD4','XGBoost':'#F44336',
             'RandomForest':'#2196F3','ExtraTrees':'#4CAF50','LightGBM':'#9C27B0',
             'LogisticRegression':'#795548','DNN':'#E91E63','LSTM':'#FF6F00'}

epsilons    = [0.01, 0.05, 0.10]
eps_labels  = ['ε=0.01','ε=0.05','ε=0.10']
eps_markers = ['o','s','D']
eps_colors  = ['#1565C0','#0288D1','#4FC3F7']

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')
gs  = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

ax_atk_e  = fig.add_subplot(gs[0, 0])
ax_atk_c  = fig.add_subplot(gs[0, 1])
ax_shap_e = fig.add_subplot(gs[1, 0])
ax_shap_c = fig.add_subplot(gs[1, 1])

x = np.arange(len(modeller9))
bw = 0.26

for (ax_atk, ax_shap), (ds_name, df) in zip(
        [(ax_atk_e, ax_shap_e), (ax_atk_c, ax_shap_c)],
        [('Edge-IIoT', df_e), ('CIC-IDS-2017', df_c)]):

    # ── Üst: Saldırı başarı oranı ──────────────────────────
    for ei, (eps, elabel, ecol) in enumerate(
            zip(epsilons, eps_labels, eps_colors)):
        sub  = df[df['epsilon'] == eps]
        vals = []
        for m in modeller9:
            row = sub[sub['model'] == m]
            vals.append(row['atk_basari_pct'].values[0]
                        if len(row) else np.nan)

        offset = (ei - 1) * bw
        bars = ax_atk.bar(x + offset, vals, bw,
                          color=ecol, alpha=0.85,
                          label=elabel, zorder=3)

    # FGSM vs Grad-Approx ayırıcı
    ax_atk.axvline(6.5, color='#7B1FA2', lw=1.2,
                   ls='--', zorder=2, alpha=0.7)
    ax_atk.text(6.55, 95,
                '← Grad-Approx  |  True FGSM →',
                fontsize=7.5, color='#7B1FA2', va='top')

    ax_atk.set_xticks(x)
    xlbls = ax_atk.set_xticklabels(kisaltma9, fontsize=10.5)
    for lbl, m in zip(xlbls, modeller9):
        if m in ['DNN','LSTM']:
            lbl.set_style('italic')
            lbl.set_color('#7B1FA2')

    ax_atk.set_ylabel('Attack Success Rate (%)', fontsize=11)
    ax_atk.set_ylim(0, 105)
    ax_atk.set_xlim(-0.55, len(modeller9)-0.45)
    ax_atk.set_title(f'{ds_name} — Adversarial Attack Success Rate',
                     fontsize=11.5, fontweight='bold', pad=8)
    ax_atk.legend(fontsize=9, loc='upper left',
                  frameon=True, framealpha=0.92,
                  edgecolor='#CFD8DC')
    ax_atk.spines['top'].set_visible(False)
    ax_atk.spines['right'].set_visible(False)
    ax_atk.yaxis.grid(True, alpha=0.22, zorder=0)
    ax_atk.set_axisbelow(True)
    ax_atk.tick_params(axis='both', labelsize=9.5)

    # ── Alt: SHAP Spearman under attack ────────────────────
    for ei, (eps, elabel, ecol, emk) in enumerate(
            zip(epsilons, eps_labels, eps_colors, eps_markers)):
        sub  = df[df['epsilon'] == eps]
        vals = []
        for m in modeller9:
            row = sub[sub['model'] == m]
            vals.append(row['shap_spearman'].values[0]
                        if len(row) else np.nan)

        ax_shap.plot(x, vals, color=ecol, marker=emk,
                     markersize=7, linewidth=2.0,
                     label=elabel, zorder=4,
                     markerfacecolor='white',
                     markeredgewidth=2.0,
                     markeredgecolor=ecol)

    # 0.9 eşiği
    ax_shap.axhline(0.9, color='#90A4AE', lw=1.0,
                    ls=':', zorder=2)
    ax_shap.text(len(modeller9)-0.1, 0.912, '0.90',
                 fontsize=8, color='#78909C',
                 ha='right', va='bottom')

    # LSTM annotation — en savunmasız
    lstm_vals = []
    for eps in epsilons:
        sub = df[df['epsilon'] == eps]
        row = sub[sub['model'] == 'LSTM']
        if len(row):
            lstm_vals.append(row['shap_spearman'].values[0])
    if lstm_vals:
        ax_shap.annotate(
            f'LSTM: most vulnerable\n(min Spearman={min(lstm_vals):.3f})',
            xy=(x[-1], min(lstm_vals)),
            xytext=(x[-1]-2.5, min(lstm_vals)-0.12),
            fontsize=8, color='#FF6F00',
            arrowprops=dict(arrowstyle='->',
                           color='#FF6F00', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='#FFF3E0',
                      edgecolor='#FFCC80',
                      linewidth=0.8))

    # Grad-Approx ayırıcı
    ax_shap.axvline(6.5, color='#7B1FA2', lw=1.2,
                    ls='--', zorder=2, alpha=0.7)

    ax_shap.set_xticks(x)
    xlbls = ax_shap.set_xticklabels(kisaltma9, fontsize=10.5)
    for lbl, m in zip(xlbls, modeller9):
        if m in ['DNN','LSTM']:
            lbl.set_style('italic')
            lbl.set_color('#7B1FA2')

    ax_shap.set_ylabel('SHAP Spearman r  (under attack)', fontsize=11)
    ax_shap.set_ylim(0.35, 1.05)
    ax_shap.set_xlim(-0.55, len(modeller9)-0.45)
    ax_shap.set_title(
        f'{ds_name} — SHAP Explanation Stability Under Attack',
        fontsize=11.5, fontweight='bold', pad=8)
    ax_shap.legend(fontsize=9, loc='lower left',
                   frameon=True, framealpha=0.92,
                   edgecolor='#CFD8DC')
    ax_shap.spines['top'].set_visible(False)
    ax_shap.spines['right'].set_visible(False)
    ax_shap.yaxis.grid(True, alpha=0.22, zorder=0)
    ax_shap.set_axisbelow(True)
    ax_shap.tick_params(axis='both', labelsize=9.5)

    # Grad-Approx notu
    ax_shap.text(0.02, 0.04,
                 '← Gradient-approximated perturbation  |  True FGSM →',
                 transform=ax_shap.transAxes, fontsize=7.5,
                 color='#7B1FA2', style='italic',
                 bbox=dict(boxstyle='round,pad=0.3',
                           facecolor='#F3E5F5',
                           edgecolor='#CE93D8',
                           linewidth=0.6))

fig.suptitle(
    'Adversarial Robustness Analysis: FGSM Attack Impact on Explanation Stability\n'
    'Top: Attack success rate  ·  Bottom: SHAP Spearman r under adversarial perturbation',
    fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout(rect=[0, 0.01, 1, 1])
out = os.path.join(CIKTI_DIR, 'fig7_fgsm_adversarial.png')
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Kaydedildi: {out}")

In [ ]:
import sklearn, xgboost, lightgbm, catboost
import shap, lime, torch, numpy, pandas

print(f"Python      : import sys; sys.version")
import sys
print(f"Python      : {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"XGBoost     : {xgboost.__version__}")
print(f"LightGBM    : {lightgbm.__version__}")
print(f"CatBoost    : {catboost.__version__}")
print(f"SHAP        : {shap.__version__}")
print(f"LIME        : {lime.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"NumPy       : {numpy.__version__}")
print(f"Pandas      : {pandas.__version__}")

In [ ]:
import torch, numpy, pandas
import importlib.metadata

print(f"PyTorch     : {torch.__version__}")
print(f"NumPy       : {numpy.__version__}")
print(f"Pandas      : {pandas.__version__}")
print(f"LIME        : {importlib.metadata.version('lime')}")

In [ ]:
import os, pickle
import pandas as pd
from scipy.stats import pearsonr, spearmanr

CHECKPOINT_EDGE = str(EDGEIIOT_DIR)
CHECKPOINT_CIC  = str(CICIDS2017_DIR)

def yukle(klasor, dosya_adi):
    yol = os.path.join(klasor, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(klasor, dosya_adi):
    return os.path.exists(os.path.join(klasor, dosya_adi))

# ── Hangi dosyalar var? ────────────────────────────────────────────────────
print("📂 Checkpoint kontrolü:\n")

aday_dosyalar = ["df_rq3.pkl", "df_var.pkl"]
print("EDGE-IIoT:")
for d in aday_dosyalar:
    var_mi = kontrol_et(CHECKPOINT_EDGE, d)
    print(f"  {'✅' if var_mi else '❌'} {d}")

print("\nCIC-IDS2017:")
for d in aday_dosyalar:
    var_mi = kontrol_et(CHECKPOINT_CIC, d)
    print(f"  {'✅' if var_mi else '❌'} {d}")

# ── Hangisi varsa onu kullan ────────────────────────────────────────────────
def rq3_verisini_al(klasor):
    if kontrol_et(klasor, "df_rq3.pkl"):
        return yukle(klasor, "df_rq3.pkl")
    elif kontrol_et(klasor, "df_rq3_cicids.pkl"):
        return yukle(klasor, "df_rq3_cicids.pkl")
    elif kontrol_et(klasor, "df_var.pkl"):
        df = yukle(klasor, "df_var.pkl")
        return df[['model_adi', 'f1_macro', 'spearman']].copy()
    elif kontrol_et(klasor, "df_var_cicids.pkl"):
        df = yukle(klasor, "df_var_cicids.pkl")
        return df[['model_adi', 'f1_macro', 'spearman']].copy()
    return None

df_edge = rq3_verisini_al(CHECKPOINT_EDGE)
df_cic  = rq3_verisini_al(CHECKPOINT_CIC)

print(f"\n{'='*50}")
print("VERİ DURUMU")
print(f"{'='*50}")
print(f"Edge-IIoT veri bulundu : {df_edge is not None}" +
      (f" → {df_edge.shape}" if df_edge is not None else ""))
print(f"CIC-IDS2017 veri bulundu: {df_cic is not None}" +
      (f" → {df_cic.shape}" if df_cic is not None else ""))

# ── Combined w/o LR hesapla (ikisi de varsa) ────────────────────────────────
if df_edge is not None and df_cic is not None:
    print(f"\n{'='*50}")
    print("COMBINED w/o LR HESAPLAMA")
    print(f"{'='*50}")

    edge_wo_lr = df_edge[df_edge['model_adi'] != 'LogisticRegression']
    cic_wo_lr  = df_cic[df_cic['model_adi']  != 'LogisticRegression']

    combined = pd.concat([
        edge_wo_lr[['f1_macro', 'spearman']],
        cic_wo_lr[['f1_macro', 'spearman']]
    ], ignore_index=True)

    print(f"Birleşik veri boyutu: {combined.shape} (Edge: {len(edge_wo_lr)}, CIC: {len(cic_wo_lr)})")

    pearson_r, pearson_p   = pearsonr(combined['f1_macro'], combined['spearman'])
    spearman_r, spearman_p = spearmanr(combined['f1_macro'], combined['spearman'])

    print(f"\n📊 Combined w/o LR:")
    print(f"  Pearson  r = {pearson_r:+.3f}, p = {pearson_p:.4g}")
    print(f"  Spearman r = {spearman_r:+.3f}, p = {spearman_p:.4g}")

    print(f"\n📋 Makaledeki Pearson değeriyle karşılaştırma:")
    print(f"  Makalede : Pearson r = +0.743")
    print(f"  Hesaplanan: Pearson r = {pearson_r:+.3f}")
    fark = abs(pearson_r - 0.743)
    print(f"  Fark: {fark:.4f} → {'✅ Eşleşiyor' if fark < 0.02 else '⚠️ FARKLI — kontrol et!'}")

else:
    print(f"\n⚠️  Veri eksik — hangi dosyalar checkpoint'te yok kontrol et.")
    print(f"   Gereken dosyalar: df_rq3.pkl / df_var.pkl (Edge ve CIC için)")

In [ ]:
import os, pickle
import pandas as pd
from scipy.stats import pearsonr, spearmanr

CHECKPOINT_EDGE = str(EDGEIIOT_DIR)
CHECKPOINT_CIC  = str(CICIDS2017_DIR)

def yukle(klasor, dosya_adi):
    yol = os.path.join(klasor, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(klasor, dosya_adi):
    return os.path.exists(os.path.join(klasor, dosya_adi))

# ── Dosya varlık kontrolü ────────────────────────────────────────────────
print("📂 Dosya kontrolü:")
print(f"  Edge df_var.pkl       : {kontrol_et(CHECKPOINT_EDGE, 'df_var.pkl')}")
print(f"  CIC  df_var_cicids.pkl: {kontrol_et(CHECKPOINT_CIC, 'df_var_cicids.pkl')}")

# ── Yükle ────────────────────────────────────────────────────────────────
df_edge = yukle(CHECKPOINT_EDGE, "df_var.pkl")
df_cic  = yukle(CHECKPOINT_CIC, "df_var_cicids.pkl")

print(f"\n✅ Edge yüklendi: {df_edge.shape}")
print(f"✅ CIC yüklendi : {df_cic.shape}")

# ── LR hariç filtrele ────────────────────────────────────────────────────
edge_wo_lr = df_edge[df_edge['model_adi'] != 'LogisticRegression']
cic_wo_lr  = df_cic[df_cic['model_adi']  != 'LogisticRegression']

print(f"\nEdge w/o LR: {len(edge_wo_lr)} satır")
print(f"CIC w/o LR : {len(cic_wo_lr)} satır")

# ── Birleştir ────────────────────────────────────────────────────────────
combined = pd.concat([
    edge_wo_lr[['f1_macro', 'spearman']],
    cic_wo_lr[['f1_macro', 'spearman']]
], ignore_index=True)

print(f"Birleşik veri: {combined.shape}")

# ── Korelasyon hesapla ───────────────────────────────────────────────────
pearson_r, pearson_p   = pearsonr(combined['f1_macro'], combined['spearman'])
spearman_r, spearman_p = spearmanr(combined['f1_macro'], combined['spearman'])

print(f"\n{'='*50}")
print("COMBINED w/o LR SONUÇLARI")
print(f"{'='*50}")
print(f"Pearson  r = {pearson_r:+.4f}, p = {pearson_p:.4g}")
print(f"Spearman r = {spearman_r:+.4f}, p = {spearman_p:.4g}")

In [ ]:
import pickle, os, re, sys
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

KLASOR_DNN_LSTM = str(DNN_LSTM_DIR)
KLASOR_CIC      = str(CICIDS2017_DIR)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}")

# ── GERÇEK sınıf tanımları (forward() ile, stub değil) ────────────────────
class IDS_DNN(nn.Module):
    def __init__(self, *a, **kw):
        super().__init__()
    def forward(self, x):
        return self.network(x)

class IDS_LSTM(nn.Module):
    def __init__(self, *a, **kw):
        super().__init__()
    def forward(self, x):
        x = x.unsqueeze(-1)            # (batch, n_features) → (batch, n_features, 1)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)

def yukle(klasor, dosya_adi):
    with open(os.path.join(klasor, dosya_adi), 'rb') as f:
        return pickle.load(f)

# ── DNN seed=0 fold=0 modelini al ──────────────────────────────────────────
dnn_cic = yukle(KLASOR_DNN_LSTM, "dnn_cic_modeller.pkl")
ref_kayit = next(k for k in dnn_cic if k['seed']==0 and k['fold']==0)
model = ref_kayit['model_obj'].to(DEVICE)
model.eval()
print(f"✅ DNN model yüklendi, kayıtlı f1_macro={ref_kayit['f1_macro']}")

# ── Veri yükle ────────────────────────────────────────────────────────────
veri = yukle(KLASOR_CIC, "veri_cicids.pkl")
X_scaled = veri['X_scaled']
y = veri['y']
print(f"✅ Veri: {X_scaled.shape}")

def model_predict(model, X, batch_size=512):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch  = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            logits = model(batch)
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
    return np.concatenate(preds)

# ── Aday referans setleriyle f1_orig'i hedef 0.8868 ile eşleştir ──────────
print(f"\n🎯 Hedef f1_orig (fidelity_dnn_cic.pkl'den) = 0.8868\n")

for boyut in [200, 500]:
    np.random.seed(42)
    idx = np.random.choice(len(X_scaled), size=boyut, replace=False)
    X_aday, y_aday = X_scaled[idx], y[idx]
    y_pred = model_predict(model, X_aday)
    f1 = f1_score(y_aday, y_pred, average='macro')
    esleme = "✅ EŞLEŞTİ" if abs(f1 - 0.8868) < 0.001 else "❌ farklı"
    print(f"  Boyut={boyut}, seed=42 → f1={f1:.4f}  {esleme}")

# Tüm veri üzerinde de bakalım (belki hiç örnekleme yapılmadı)
y_pred_full = model_predict(model, X_scaled)
f1_full = f1_score(y, y_pred_full, average='macro')
print(f"  Tüm veri (n={len(X_scaled)}) → f1={f1_full:.4f}  "
      f"{'✅ EŞLEŞTİ' if abs(f1_full-0.8868)<0.001 else '❌ farklı'}")

print("\n✅ Doğrulama tamamlandı! Hangi satır eşleştiyse o referans setini kullanacağız.")

In [ ]:
import pickle, os, gc, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}\n")

KLASOR_DNN_LSTM = str(DNN_LSTM_DIR)
KLASOR_EDGE     = str(EDGEIIOT_DIR)
KLASOR_CIC      = str(CICIDS2017_DIR)

# ── Doğrulanmış gerçek sınıf tanımları ─────────────────────────────────────
class IDS_DNN(nn.Module):
    def __init__(self, *a, **kw):
        super().__init__()
    def forward(self, x):
        return self.network(x)

class IDS_LSTM(nn.Module):
    def __init__(self, *a, **kw):
        super().__init__()
    def forward(self, x):
        x = x.unsqueeze(-1)
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

def yukle(klasor, dosya_adi):
    with open(os.path.join(klasor, dosya_adi), 'rb') as f:
        return pickle.load(f)

def kaydet(nesne, klasor, dosya_adi):
    with open(os.path.join(klasor, dosya_adi), 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 Kaydedildi: {dosya_adi}")

def kontrol_et(klasor, dosya_adi):
    return os.path.exists(os.path.join(klasor, dosya_adi))

def veri_yukle_normalize(klasor, dosya_adi):
    """Edge-IIoT ve CIC-IDS2017 farklı key adları kullanıyor, normalize eder"""
    veri = yukle(klasor, dosya_adi)
    if 'X_scaled' in veri:
        X = veri['X_scaled']
        y = veri['y']
    elif 'X_multi_scaled' in veri:
        X = veri['X_multi_scaled']
        y = veri['y_multi']
    else:
        raise KeyError(f"Beklenmeyen key yapısı: {list(veri.keys())}")
    return np.asarray(X), np.asarray(y)

def model_predict(model, X, batch_size=512):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch  = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            logits = model(batch)
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
    return np.concatenate(preds)

# ============================================================
# BASELINE FIDELITY — Random + Least-important
# Mean-imputation maskeleme
# ============================================================
def baseline_fidelity(model, X_fid, y_fid, X_train_mean, shap_importance,
                       n_mask_list=range(1, 21), n_random_repeat=20, seed=42):
    rng = np.random.RandomState(seed)
    n_features = X_fid.shape[1]

    y_pred_orig = model_predict(model, X_fid)
    f1_orig = f1_score(y_fid, y_pred_orig, average='macro')

    en_onemsiz_sira = np.argsort(shap_importance)
    least_results = {}
    for n_mask in n_mask_list:
        mask_idx = en_onemsiz_sira[:n_mask]
        X_masked = X_fid.copy()
        X_masked[:, mask_idx] = X_train_mean[mask_idx]
        f1_mask = f1_score(y_fid, model_predict(model, X_masked), average='macro')
        least_results[n_mask] = f1_orig - f1_mask

    random_acc = {n: [] for n in n_mask_list}
    for _ in range(n_random_repeat):
        rastgele_sira = rng.permutation(n_features)
        for n_mask in n_mask_list:
            mask_idx = rastgele_sira[:n_mask]
            X_masked = X_fid.copy()
            X_masked[:, mask_idx] = X_train_mean[mask_idx]
            f1_mask = f1_score(y_fid, model_predict(model, X_masked), average='macro')
            random_acc[n_mask].append(f1_orig - f1_mask)
    random_avg = {n: np.mean(v) for n, v in random_acc.items()}

    return f1_orig, least_results, random_avg

# ============================================================
# KONFIG — doğrulanmış gerçek dosya adları
# ============================================================
KONFIG = [
    ('DNN',  'Edge', "dnn_edge_modeller.pkl",  KLASOR_EDGE, "veri.pkl",        "xai_dnn_edge.pkl",  "fidelity_dnn_edge.pkl"),
    ('LSTM', 'Edge', "lstm_edge_modeller.pkl", KLASOR_EDGE, "veri.pkl",        "xai_lstm_edge.pkl", "fidelity_lstm_edge.pkl"),
    ('DNN',  'CIC',  "dnn_cic_modeller.pkl",   KLASOR_CIC,  "veri_cicids.pkl", "xai_dnn_cic.pkl",   "fidelity_dnn_cic.pkl"),
    ('LSTM', 'CIC',  "lstm_cic_modeller.pkl",  KLASOR_CIC,  "veri_cicids.pkl", "xai_lstm_cic.pkl",  "fidelity_lstm_cic.pkl"),
]

SONUC_DOSYA = "df_fidelity_dnn_lstm_baseline.pkl"

# ============================================================
# ANA DÖNGÜ
# ============================================================
if kontrol_et(KLASOR_DNN_LSTM, SONUC_DOSYA):
    print("♻️  Sonuçlar checkpoint'ten yükleniyor...")
    df_baseline = yukle(KLASOR_DNN_LSTM, SONUC_DOSYA)
else:
    kayitlar = []

    for model_type, dataset_adi, model_dosya, veri_klasoru, veri_dosya, shap_dosya, eski_fid_dosya in KONFIG:
        print(f"\n{'='*55}")
        print(f"  🔷 {model_type} — {dataset_adi}")
        print(f"{'='*55}")

        try:
            modeller = yukle(KLASOR_DNN_LSTM, model_dosya)
            ref_kayit = next(k for k in modeller if k['seed']==0 and k['fold']==0)
            model = ref_kayit['model_obj'].to(DEVICE)
            model.eval()
            del modeller; gc.collect()

            X_scaled, y = veri_yukle_normalize(veri_klasoru, veri_dosya)
            X_train_mean = X_scaled.mean(axis=0)

            np.random.seed(42)
            idx = np.random.choice(len(X_scaled), size=500, replace=False)
            X_fid, y_fid = X_scaled[idx], y[idx]

            xai = yukle(KLASOR_DNN_LSTM, shap_dosya)
            shap_importance = np.mean(np.abs(xai['shap_values']), axis=0)
            del xai; gc.collect()

            y_pred_check = model_predict(model, X_fid)
            f1_check = f1_score(y_fid, y_pred_check, average='macro')
            eski_fid = yukle(KLASOR_DNN_LSTM, eski_fid_dosya)
            f1_eski = eski_fid[eski_fid['n_maskele']==1]['f1_orig'].values[0]
            esleme_ok = abs(f1_check - f1_eski) < 0.001
            print(f"  🔍 Doğrulama: hesaplanan f1={f1_check:.4f}, kayıtlı f1={f1_eski:.4f} "
                  f"{'✅' if esleme_ok else '⚠️ UYUŞMUYOR — devam ediliyor ama sonucu kontrol et'}")

            t0 = time.time()
            f1_orig, least_results, random_avg = baseline_fidelity(
                model, X_fid, y_fid, X_train_mean, shap_importance,
                n_mask_list=range(1, 21), n_random_repeat=20
            )
            print(f"  ✅ Tamamlandı ({time.time()-t0:.1f} sn)")
            print(f"     ΔF1@5 Random          = {random_avg[5]:.4f}")
            print(f"     ΔF1@5 Least-important = {least_results[5]:.4f}")

            for n_mask in range(1, 21):
                kayitlar.append({
                    'model': model_type, 'dataset': dataset_adi, 'n_maskele': n_mask,
                    'f1_orig': round(f1_orig, 4),
                    'delta_f1_random': round(random_avg[n_mask], 4),
                    'delta_f1_least_important': round(least_results[n_mask], 4),
                    'esleme_dogrulandi': esleme_ok,
                })

            del model; torch.cuda.empty_cache(); gc.collect()

        except Exception as e:
            print(f"  ❌ HATA: {model_type}-{dataset_adi} işlenemedi: {e}")
            print(f"     → Bu kombinasyon atlandı, diğerlerine geçiliyor.")
            continue

    if kayitlar:
        df_baseline = pd.DataFrame(kayitlar)
        kaydet(df_baseline, KLASOR_DNN_LSTM, SONUC_DOSYA)
    else:
        print("\n❌ Hiçbir kombinasyon başarılı olmadı — df_baseline oluşturulamadı.")
        df_baseline = None

# ── Table 10 formatında özet ─────────────────────────────────────────────
if df_baseline is not None and len(df_baseline) > 0:
    print(f"\n{'='*70}")
    print("Table 10 — DNN/LSTM Random + Least-important (@5) + SHAP/Random Oranı")
    print(f"{'='*70}")

    ozet = df_baseline[df_baseline['n_maskele'] == 5].copy()

    shap_degerleri = {
        ('DNN','Edge'): 0.6313, ('LSTM','Edge'): 0.6837,
        ('DNN','CIC'):  0.6304, ('LSTM','CIC'):  0.7073,
    }
    ozet['shap_delta'] = ozet.apply(
        lambda r: shap_degerleri.get((r['model'], r['dataset']), np.nan), axis=1)
    ozet['shap_random_orani'] = ozet['shap_delta'] / ozet['delta_f1_random']

    print(ozet[['model','dataset','shap_delta','delta_f1_random',
                'delta_f1_least_important','shap_random_orani',
                'esleme_dogrulandi']].to_string(index=False))

    print("\n✅ DNN/LSTM baseline fidelity tamamlandı!")
else:
    print("\n⚠️  Özet tablo oluşturulamadı — yukarıdaki hata mesajlarını kontrol et.")

In [ ]:
import pickle, os
import pandas as pd

KLASOR_DNN_LSTM = str(DNN_LSTM_DIR)

def yukle(klasor, dosya_adi):
    with open(os.path.join(klasor, dosya_adi), 'rb') as f:
        return pickle.load(f)

fgsm_cic  = yukle(KLASOR_DNN_LSTM, "fgsm_cic_sonuclar.pkl")
fgsm_edge = yukle(KLASOR_DNN_LSTM, "fgsm_edge_sonuclar.pkl")

df_cic  = pd.DataFrame(fgsm_cic)
df_edge = pd.DataFrame(fgsm_edge)

print("="*80)
print("CIC-IDS2017 — Tüm kayıtlar (model × epsilon)")
print("="*80)
print(df_cic.to_string(index=False))

print("\n" + "="*80)
print("EDGE-IIoT — Tüm kayıtlar (model × epsilon)")
print("="*80)
print(df_edge.to_string(index=False))

# ── Sadece ε=0.10 (Table 13'ün raporladığı değer) ───────────────────────
print("\n" + "="*80)
print("📊 ε=0.10 İÇİN — Table 13'e Eklenecek LIME Sütunu")
print("="*80)

eps_cic  = df_cic[df_cic['epsilon'] == 0.10].copy()
eps_edge = df_edge[df_edge['epsilon'] == 0.10].copy()

print("\nCIC-IDS-2017 (ε=0.10):")
print(eps_cic[['model','atk_basari_pct','shap_spearman','lime_spearman']].to_string(index=False))

print("\nEdge-IIoT (ε=0.10):")
print(eps_edge[['model','atk_basari_pct','shap_spearman','lime_spearman']].to_string(index=False))

# Eksik model var mı kontrol et (9 model bekleniyor: DT,CB,XGB,RF,ET,LGBM,LR,DNN,LSTM)
beklenen_modeller = {'DecisionTree','CatBoost','XGBoost','RandomForest',
                      'ExtraTrees','LightGBM','LogisticRegression','DNN','LSTM'}
print(f"\nCIC'de bulunan modeller : {set(eps_cic['model'])}")
print(f"Edge'de bulunan modeller: {set(eps_edge['model'])}")
print(f"Eksik (CIC) : {beklenen_modeller - set(eps_cic['model'])}")
print(f"Eksik (Edge): {beklenen_modeller - set(eps_edge['model'])}")